# M5.S1 Hybrid computing patterns

A hot edge heats a cold metal plate. We will calculate how temperature spreads through it, then run the same calculation using different combinations of CPU processes, CPU threads and a GPU.

You will create and compile the programs in your own cluster account. Start with a simple serial solver, check its temperatures and record its time. Then divide the work with MPI, add OpenMP threads and compare with GPU execution. The mathematical problem stays the same while the execution changes.

Read each explanation before using its commands. Run shell commands in your SSH terminal from the folder specified in the exercise. Stop at each checkpoint, inspect the saved files and explain the result before continuing.

By the end, you should be able to explain which process owns each part of the plate, why ranks exchange boundary data, which memory threads share, and where GPU data transfers happen. You should also be able to submit a Slurm job, identify the software it uses, validate its output and compare measured execution times.

Keep your predictions, job IDs, compiler settings, measurements and explanations. In the final student exercise, you will increase the workload and investigate whether a different execution layout becomes more useful.


## 1. What are we solving?

Imagine a thin metal plate. We hold its top edge at 100 °C and its other edges at 0 °C. Initially the interior is cold. We want to calculate how temperature spreads into the plate.

A solver is the program that approximates the mathematical equation. It stores a grid of temperatures and calculates the next grid from the current one. This is a simplified teaching model: constant diffusivity, no internal heat sources, no varying material properties.

For temperature $T(x,y,t)$, the two-dimensional heat equation is

$$\frac{\partial T}{\partial t}=\alpha\left(\frac{\partial^2T}{\partial x^2}+\frac{\partial^2T}{\partial y^2}\right).$$

Here $\alpha$ is thermal diffusivity. A forward time step and centred spatial differences give

$$T^{k+1}_{y,x}=T^k_{y,x}+r_x(T^k_{y,x-1}-2T^k_{y,x}+T^k_{y,x+1})+r_y(T^k_{y-1,x}-2T^k_{y,x}+T^k_{y+1,x}),$$

where $r_x=\alpha\Delta t/\Delta x^2$ and $r_y=\alpha\Delta t/\Delta y^2$. For this explicit scheme, use $r_x+r_y\leq 1/2$. We use dimensionless $\Delta x=\Delta y=1$, $\alpha=1$ and $\Delta t=1/4$, exactly at that bound. Thus

$$\boxed{T^{k+1}_{y,x}=\tfrac14(T^k_{y-1,x}+T^k_{y+1,x}+T^k_{y,x-1}+T^k_{y,x+1})}.$$

This is the four-neighbour update we will implement. Do not change the coefficient casually. A larger value can make this explicit method unstable. The number of steps is fixed: we are observing transient diffusion, not stopping on a steady-state residual.

Implementation choices

- `N` includes boundary cells. `old[y*N+x]` is a row-major, contiguous array element.
- We read `old` and write `next`. Reading partly updated neighbours would implement a different algorithm.
- We keep boundaries fixed. The top row, including its two corners, is 100 °C. The remaining side and bottom boundary cells are 0 °C. This corner convention is deliberate and identical in every version.
- Increasing `N` here increases the size of the dimensionless grid. We do not claim to refine a fixed physical plate. For that experiment you would also change spacing and the stable time step.

Write down your prediction. After the first update, what is the temperature of a non-corner cell immediately below the hot edge? It has one 100 °C neighbour and three 0 °C neighbours: 25 °C. The cell two rows below remains 0 °C at that first update.

### Watch one time step become many

Run this small Python illustration in Jupyter. It explains the stencil visually. The C programs below do the actual HPC work. Notice that the heated region expands gradually and that the fixed edge stays at 100 °C.

In [ ]:
from IPython.display import display, HTML
# Small explanatory calculation; cluster performance is measured later.
from html import escape
n = 32
old = [0.0] * (n*n)
old[:n] = [100.0]*n
snapshots = {0: old.copy()}
for step in range(1, 101):
    new = old.copy()
    for y in range(1, n-1):
        for x in range(1, n-1):
            k = y*n+x
            new[k] = 0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1])
    old = new
    if step in (1,20,100): snapshots[step] = old.copy()
assert snapshots[1][n+n//2] == 25.0
svg = ['<svg xmlns="http://www.w3.org/2000/svg" width="100%" viewBox="0 0 1060 300"><rect width="1060" height="300" fill="white"/>']
for panel, (step, field) in enumerate(snapshots.items()):
    left = 15 + panel*260
    svg.append(f'<text x="{left}" y="24" font-family="sans-serif" font-size="18">Step {step}</text>')
    for y in range(n):
        for x in range(n):
            v = field[y*n+x]/100
            colour = f'rgb({int(255*v)},{int(90*v)},{int(150*(1-v))})'
            svg.append(f'<rect x="{left+x*7}" y="{40+y*7}" width="7" height="7" fill="{colour}"/>')
svg.append('<text x="15" y="289" font-family="sans-serif" font-size="14">0 °C (blue) to 100 °C (red). The first interior row reaches 25 °C after one step.</text></svg>')
display(HTML(''.join(svg)))
print("PASS Python first-step calculation. No Matplotlib or NumPy installation is needed.")


## 2. Log in and find your way around

Use the account assigned to you by SciTech. Replace `YOUR_USERNAME`; do not copy it literally.

From your laptop, outside campus:

```bash
ssh -J YOUR_USERNAME@ssh.iesci.tech YOUR_USERNAME@10.205.20.10
```

From the campus network:

```bash
ssh YOUR_USERNAME@10.205.20.10
```

These are the connection routes documented in `cluster/README.md`. Once logged in:

```bash
hostname
whoami
pwd
sinfo
squeue -u "$USER"
mkdir -p ~/hpc-labs/m5s1-heat/{src,bin,scripts,results,notes,lib/v1,lib/v2,lib/fixed}
cd ~/hpc-labs/m5s1-heat
printf 'Prediction before running:\n' > notes/observations.txt
printf 'Current folder: %s\n' "$PWD"
```

Your files belong in your own account. We use `~/hpc-labs/m5s1-heat` throughout. The folder must be visible on the compute nodes. `src` holds source, `bin` executables, `scripts` launchers, `results` evidence and `notes` your explanations. `lib` will hold our small shared libraries.

The login node is the place to edit files, inspect the queue and submit work. Slurm assigns the compute resources. A job can be pending even when you have logged in successfully. Do not SSH directly to compute nodes.

For the first short run we will request a CPU shell. Subsequent experiments use batch jobs so that you can disconnect while they wait or run.

### Checkpoint 1 - Your cluster session

Run this in SSH from the lab folder. Check that the printed username is yours and that you are on the login node.

```bash
pwd
hostname
whoami
command -v sbatch
command -v srun
test -d src && test -d results && echo 'PASS lab folders exist'
```

Do not continue if Slurm commands or your folders are missing.

### Software modules and linked libraries

A module changes your shell environment; it does not install software into your folder. It can select a compiler, MPI stack or application and its dependencies. Use this in your SSH session before compiling MPI.

```bash
module list
module avail OpenMPI
module show OpenMPI/4.1.5-GCC-12.3.0
module load OpenMPI/4.1.5-GCC-12.3.0
module list
which gcc
which mpicc
mpicc --showme
```

Check which compiler and MPI libraries the wrapper selects. After compiling, inspect the runtime dependencies too.

```bash
ldd bin/heat_mpi
ldd bin/heat_omp
```

Keep the compiler and MPI stack consistent. Loading a different MPI module after compiling can leave an executable linked to an incompatible runtime. The batch jobs record the module list, wrapper command and linked libraries so you can trace the environment used for each result.

For GPU compilation, inspect `module avail nvhpc` and `module show nvhpc/25.7`. Load it in the GPU job rather than replacing the CPU stack halfway through a queued experiment.

### Create `src/heat_common.h`

This header contains the small tasks that every version needs. It checks arguments, creates the grid, reads the clock and saves the result. Keeping those tasks in one place helps us compare the solvers without accidentally changing their inputs or output format. Find `grid` and `save_grid`, and explain why the top row is set before the time loop begins.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > src/heat_common.h <<'EOF'
#ifndef HEAT_COMMON_H
#define HEAT_COMMON_H
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <errno.h>
#include <string.h>
/* Refuse invalid arguments before they can produce an invalid array index. */
static inline int integer(const char *s,int lo,int hi){
 char *end; errno=0; long v=strtol(s,&end,10);
 if(errno||*end||end==s||v<lo||v>hi){fprintf(stderr,"Invalid integer: %s\n",s);exit(2);}return (int)v;
}
/* A monotonic clock measures elapsed time without changes to the calendar clock. */
static inline double clock_seconds(void){
 struct timespec t;if(clock_gettime(CLOCK_MONOTONIC,&t)){perror("clock_gettime");exit(2);}
 return t.tv_sec+1e-9*t.tv_nsec;
}
/* calloc sets the cold plate to zero; only the hot top edge is changed. */
static inline double *grid(int n,double hot){
 double *a=(double*)calloc((size_t)n*n,sizeof(double));
 if(!a){perror("calloc");exit(2);}for(int x=0;x<n;x++)a[x]=hot;return a;
}
/* Save all cells in row-major order so another program can validate the field. */
static inline void save_grid(const char *path,const double *a,int n){
 FILE *f=fopen(path,"wb");if(!f){perror(path);exit(2);}
 if(fwrite(a,sizeof(double),(size_t)n*n,f)!=(size_t)n*n){fprintf(stderr,"Short grid write\n");exit(2);}
 if(fclose(f)){perror("fclose");exit(2);}
}
/* Keep timing scopes separate: update loop, output and the broader program interval. */
static inline void result(const char *mode,int n,int steps,int ranks,int threads,
 double setup,double loop,double halo,double update,double io,double total,const double *a){
 double sum=0;for(size_t k=0;k<(size_t)n*n;k++)sum+=a[k];
 printf("RESULT mode=%s n=%d steps=%d ranks=%d threads=%d setup_s=%.9f loop_s=%.9f halo_s=%.9f update_s=%.9f io_s=%.9f total_s=%.9f checksum=%.17g\n",
 mode,n,steps,ranks,threads,setup,loop,halo,update,io,total,sum);
}
#endif
EOF
```

### Create `src/heat_naive.c`

This is our starting point: one process runs on one CPU core. For every time step, the first pair of loops calculates temperatures from the old grid; the second pair copies the new values back. Follow one interior cell through those loops. The boundaries stay fixed because neither loop updates them. Later we will remove the copy without changing the calculation.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > src/heat_naive.c <<'EOF'
/* First program: one CPU, explicit copy after every time step. */
#include "heat_common.h"
int main(int argc,char **argv){
 if(argc!=4){fprintf(stderr,"usage: heat_naive N STEPS field.bin\n");return 2;}
 /* N includes boundary cells; STEPS is the number of time updates. */
 int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000);
 /* Two arrays keep all reads at the old time level while we write the next one. */
 double start=clock_seconds();double *old=grid(n,100),*next=grid(n,100);
 double setup=clock_seconds()-start, t=clock_seconds();
 /* Boundary rows and columns are excluded from the update loops. */
 for(int s=0;s<steps;s++){
  for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++){
   /* Flatten (row, column) to one index; neighbours are +/-n and +/-1. */
   size_t k=(size_t)y*n+x;
   next[k]=0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);
  }
  /* This extra copy is correct but moves the full interior on every step. */
  for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++)old[(size_t)y*n+x]=next[(size_t)y*n+x];
 }
 /* Stop the loop timer before writing the field to storage. */
 double loop=clock_seconds()-t;t=clock_seconds();save_grid(argv[3],old,n);
 double io=clock_seconds()-t;
 result("naive",n,steps,1,1,setup,loop,0,loop,io,clock_seconds()-start,old);
 free(old);free(next);return 0;
}
EOF
```

## 3. Compile and run your first program

From the login node, request a small allocation. The command waits if necessary. Run it once.

```bash
srun -p cpu --nodes=1 --ntasks=1 --cpus-per-task=1 --mem=1G --time=00:05:00 --pty bash -l
```

When the compute shell opens:

```bash
cd ~/hpc-labs/m5s1-heat
hostname
printf 'Job allocation: %s\n' "$SLURM_JOB_ID"
gcc --version
gcc -std=c11 -O0 -g -Wall -Wextra src/heat_naive.c -o bin/heat_naive
./bin/heat_naive 32 1 results/first-step.bin | tee results/first-step.log
./bin/heat_naive 256 350 results/naive.bin | tee results/naive.log
ls -lh results/*.bin
```

`gcc` compiles C into an executable. `-O0` disables compiler optimisation, `-g` includes debugging information and the warnings help catch mistakes. `./bin/heat_naive` runs the file you just built. The next two arguments are N and the number of updates. `tee` displays stdout and saves a copy.

The program prints `loop_s`, the time spent updating and copying, and `total_s`, the program interval including allocation and field output. Neither includes time waiting in Slurm's queue. A shell timer, introduced below, also includes launching the program.

Check the first update without trusting the C code:

```bash
python3 - <<'PY'
from array import array
with open('results/first-step.bin','rb') as f:
    a=array('d');a.fromfile(f,32*32)
assert a[32+16] == 25.0
assert a[2*32+16] == 0.0
assert all(a[x] == 100.0 for x in range(32))
print('PASS: first-step checks agree with the calculation on paper')
PY
```

Write the job ID, N, steps and observed `loop_s` into `notes/observations.txt`. Then leave the allocation, returning to the login node:

```bash
exit
```

Why must each update read the same old time level? Describe what could go wrong if you wrote into the array you were still reading.

### Checkpoint 2 - The first update

The first-step checks above must print PASS. For N=32, `results/first-step.bin` must contain exactly 8192 bytes (32 × 32 × 8). Check the file and retain its log.

```bash
wc -c results/first-step.bin
cat results/first-step.log
```

The first non-boundary row should contain 25 °C away from the sides. Fix any failed assertion before adding parallelism.

## 4. Remove the copy, then introduce threads

The mathematical update stays the same. Instead of copying every cell, swap the two pointer variables. Both arrays already have the correct fixed boundaries.

We compile this source three ways: unoptimised serial, optimised serial and OpenMP. That separates the effect of removing the copy from compiler optimisation and adding threads.

The OpenMP directive assigns different output rows to threads. They share the input array. Each cell has one writer. The end of the parallel loop includes a barrier, so the pointer swap occurs only after the update is complete. This introductory implementation creates a parallel region on each time step; that overhead is something to measure.

### Create `src/heat_cpu.c`

Here we replace a whole-grid copy with a swap of two pointers. The arrays stay where they are in memory; only the names that refer to them change. Compile without `-fopenmp` for serial execution and with it for OpenMP. The compiler then divides rows between threads. Each thread writes different cells, and all threads finish before the arrays swap roles.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > src/heat_cpu.c <<'EOF'
/* Same mathematics. Swap pointers; optionally compile OpenMP or a library boundary. */
#include "heat_common.h"
#ifdef _OPENMP
#include <omp.h>
#endif
#ifdef HOT_FROM_LIBRARY
extern double top_temperature(void);
#endif
int main(int argc,char **argv){
 if(argc!=4){fprintf(stderr,"usage: heat_cpu N STEPS field.bin\n");return 2;}
 /* Keep the same grid and number of steps for every execution layout. */
 int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000),threads=1;
 const char *mode="serial";double hot=100;
#ifdef HOT_FROM_LIBRARY
 /* The shared library must return Celsius, the units expected by this solver. */
 hot=top_temperature();mode="library";
#endif
#ifdef _OPENMP
 mode="openmp";
 /* Measure the actual team size, rather than assuming the requested count. */
 #pragma omp parallel
 {
  #pragma omp single
  threads=omp_get_num_threads();
 }
#endif
 double start=clock_seconds();double *old=grid(n,hot),*next=grid(n,hot);
 double setup=clock_seconds()-start,t=clock_seconds();
 for(int s=0;s<steps;s++){
#ifdef _OPENMP
  /* Threads read shared old data and write disjoint rows in next.
     The implicit end-of-loop barrier completes all writes before the swap. */
  #pragma omp parallel for schedule(static)
#endif
  for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++){
   size_t k=(size_t)y*n+x;
   next[k]=0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);
  }
  /* Swap names instead of copying every temperature. Boundaries exist in both arrays. */
  double *tmp=old;old=next;next=tmp;
 }
 /* I/O is timed separately so it cannot be mistaken for stencil computation. */
 double loop=clock_seconds()-t;t=clock_seconds();save_grid(argv[3],old,n);
 double io=clock_seconds()-t;
 result(mode,n,steps,1,threads,setup,loop,0,loop,io,clock_seconds()-start,old);
 free(old);free(next);return 0;
}
EOF
```

### A short manual comparison

Request four CPU cores from the login node:

```bash
srun -p cpu -N1 -n1 -c4 --mem=1G --time=00:05:00 --pty bash -l
```

Inside the allocation:

```bash
cd ~/hpc-labs/m5s1-heat
gcc -std=c11 -O0 -g src/heat_cpu.c -o bin/heat_serial_O0
gcc -std=c11 -O2 -g src/heat_cpu.c -o bin/heat_serial
gcc -std=c11 -O2 -g -fopenmp src/heat_cpu.c -o bin/heat_omp
export OMP_DYNAMIC=FALSE OMP_PLACES=cores OMP_PROC_BIND=close
./bin/heat_serial_O0 256 350 results/swap-O0.bin | tee results/swap-O0.log
./bin/heat_serial 256 350 results/serial-O2.bin | tee results/serial-O2.log
OMP_NUM_THREADS=1 ./bin/heat_omp 256 350 results/omp-1.bin | tee results/omp-1.log
OMP_NUM_THREADS=4 ./bin/heat_omp 256 350 results/omp-4.bin | tee results/omp-4.log
exit
```

Before comparing the timings, remember that these single measurements are a first look, not sufficient evidence of a reliable speedup. Next we repeat measurements and validate complete grids. Compare naive `-O0` against pointer-swap `-O0` to discuss the copy. Compare serial `-O0` against serial `-O2` for compiler optimisation. Compare OpenMP 1 thread against 4 threads to investigate threading. Do not attribute all differences between naive `-O0` and OpenMP `-O2` to parallelism.

Append your answer: *Which change helped on this input? Which comparison would you repeat before making a claim?*

### Checkpoint 3 - Serial and threaded fields

Run in SSH from the lab folder. This reads the complete fields from the manual runs; no compute allocation is needed for this small check.

```bash
python3 - <<'PY'
from array import array
from pathlib import Path
files=['naive.bin','swap-O0.bin','serial-O2.bin','omp-1.bin','omp-4.bin']
fields=[]
for name in files:
    a=array('d');a.frombytes((Path('results')/name).read_bytes())
    assert len(a)==256*256, name
    fields.append(a)
for name,a in zip(files,fields):
    error=max(abs(x-y) for x,y in zip(fields[0],a))
    assert error<=1e-10, (name,error)
    print('PASS', name, 'max error', error)
PY
```

All five fields should agree. If they do not, check N, steps, boundaries and compiler commands before comparing their times.

## 5. Save evidence and use batch jobs

A batch job is a shell script with resource requests. `sbatch` submits it and returns a job ID. Slurm runs the script later on assigned hardware. A request for two ranks with two CPU cores per rank uses four CPU cores in total.

The scripts below provide a reusable measurement harness. You still create, compile, submit and inspect the experiment yourself. Read them before running them.

- `site.sh` selects the documented compiler stack and stops on a failed module load. Batch scripts use a login shell because the course has observed different environments in notebook and batch sessions.
- `measure.sh` starts one measured execution and saves its raw output and full field.
- `validate.py` compares every temperature, including boundaries and finiteness. It accepts at most `1e-10` °C absolute difference for these double-precision examples.
- `summarize.py` rechecks saved fields before creating `measurements.csv`. It excludes repeat 0 from measurements.
- `submit.sh` freezes source/scripts in a new run folder before submission. Editing your working source later cannot change a queued job.

Create the helper files below. Use the serial stage first; it does not need MPI or a GPU. You can copy the long blocks, but explain what each file does in your notes.

### Create `scripts/site.sh`

A batch shell can have a different environment from your SSH terminal. This file makes the compiler and MPI or GPU tools visible in the job, then sets thread placement. A module changes search paths and library settings; it does not rewrite your source. Read the selected module names and compare them with `module list`. A failed load stops the job so that we do not silently benchmark a different stack.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/site.sh <<'EOF'
# Source this file in SSH and batch shells. The login shell is important on SciTech.
# These module names are documented in the course; check module avail on your login.
# Initialise the module command if this batch shell did not inherit it.
if ! type module >/dev/null 2>&1; then
 for init in /etc/profile.d/modules.sh /usr/share/Modules/init/bash; do
  if [[ -r "$init" ]]; then source "$init"; fi
  if type module >/dev/null 2>&1; then break; fi
 done
fi
# CPU and GPU jobs select their own supported software stack.
if [[ ${LAB_DEVICE:-cpu} == gpu ]]; then
 if type module >/dev/null 2>&1; then module load nvhpc/25.7; fi
 command -v nvcc >/dev/null || { echo 'nvcc unavailable: check the site GPU module' >&2; return 2; }
else
 if [[ ${LAB_NEEDS_MPI:-1} == 1 ]]; then
  if type module >/dev/null 2>&1; then module load OpenMPI/4.1.5-GCC-12.3.0; fi
  command -v mpicc >/dev/null || { echo 'mpicc unavailable: check the site MPI module' >&2; return 2; }
 fi
fi
# Keep thread counts fixed and bind threads to nearby allocated cores.
export OMP_DYNAMIC=FALSE OMP_PLACES=cores OMP_PROC_BIND=close
# No hidden fallback to another compiler stack after a failed module load.
EOF
```

### Create `scripts/measure.sh`

This script runs one version once and saves a separate log and temperature field. `srun` starts the process inside the allocated resources, while `OMP_NUM_THREADS` sets the thread count. The solver's timer covers its update loop. The outer wall timer also includes starting the program and saving the output. Those numbers answer different questions, so keep both.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/measure.sh <<'EOF'
#!/bin/bash
# Called inside an allocation, once per program execution. Separate raw logs per trial.
set -euo pipefail
# Each call receives a mode, worker layout, executable, inputs and repeat number.
mode=$1; ranks=$2; threads=$3; exe=$4; n=$5; steps=$6; out=$7; repeat=$8
mkdir -p "$out"
# Thread count does not allocate CPUs; the job/step resource requests do that.
export OMP_NUM_THREADS=$threads
log="$out/$mode-r$repeat.log"; field="$out/$mode-r$repeat.bin"
# MPI modes need a launcher that creates a shared MPI communicator.
if [[ "$mode" == mpi || "$mode" == hybrid ]]; then
 export LAB_EXPECT_RANKS=$ranks
 case ${MPI_LAUNCHER:-srun} in
  srun) launch=(srun --ntasks="$ranks" --cpus-per-task="$threads" --cpu-bind=cores)
        if [[ -n ${SRUN_MPI_TYPE:-} ]]; then launch+=("--mpi=$SRUN_MPI_TYPE"); fi;;
  mpirun) launch=(mpirun -np "$ranks" --map-by "slot:PE=$threads" --bind-to core);;
  *) echo 'MPI_LAUNCHER must be srun or mpirun' >&2; exit 2;;
 esac
else
 launch=(srun --ntasks=1 --cpus-per-task="$threads" --cpu-bind=cores)
fi
# /usr/bin/time includes launcher/startup/compute/output; RESULT loop_s is solver-only.
if [[ -x /usr/bin/time ]]; then
 /usr/bin/time -f 'WALL elapsed_s=%e user_s=%U sys_s=%S maxrss_kb=%M' \
  "${launch[@]}" "$exe" "$n" "$steps" "$field" >"$log" 2>&1
else
 # Bash's timer still measures real execution; peak RSS is explicitly unavailable.
 TIMEFORMAT='WALL elapsed_s=%R user_s=%U sys_s=%S maxrss_kb=UNAVAILABLE'
 { time "${launch[@]}" "$exe" "$n" "$steps" "$field"; } >"$log" 2>&1
fi
cat "$log"
EOF
```

### Create `scripts/validate.py`

A program that finishes may still contain an indexing, boundary or communication error. This checker reads every saved temperature, rejects missing or non-finite values, checks the boundaries and compares the full field with serial. The tolerance allows a small numerical difference. The Kelvin exercise explicitly expects a mismatch; all normal solvers must pass the ordinary check.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/validate.py <<'EOF'
"""Full-field numerical checks; no third-party packages required."""
import argparse, array, math
p=argparse.ArgumentParser();p.add_argument('reference');p.add_argument('candidate');p.add_argument('n',type=int)
p.add_argument('--expect-different',action='store_true');args=p.parse_args();n=args.n

# Refuse truncated fields and NaNs before doing a numerical comparison.
def read(path):
 a=array.array('d');a.frombytes(open(path,'rb').read())
 if len(a)!=n*n:raise SystemExit(f'FAIL {path}: expected {n*n} doubles, got {len(a)}')
 if not all(math.isfinite(x) for x in a):raise SystemExit('FAIL: non-finite temperature')
 return a
ref=read(args.reference);got=read(args.candidate)
# A checksum can hide errors; compare corresponding cells everywhere.
err=max(abs(a-b) for a,b in zip(ref,got))
# Check the physical constraints separately from agreement with reference.
boundaries=(all(abs(got[x]-100)<1e-10 for x in range(n)) and
 all(abs(got[(n-1)*n+x])<1e-10 for x in range(n)) and
 all(abs(got[y*n])<1e-10 and abs(got[y*n+n-1])<1e-10 for y in range(1,n-1)))
bounded=min(got)>=-1e-10 and max(got)<=100+1e-10
matches=err<=1e-10 and boundaries and bounded
if args.expect_different:
 # The v2 Kelvin example is an intentionally incorrect Celsius field.
 if err<=1e-10:raise SystemExit('FAIL: expected the incompatible library to change the field')
 print(f'EXPECTED MISMATCH (not a correct heat solution): max_abs_error={err:.6g}, boundaries_ok={boundaries}')
else:
 if not matches:raise SystemExit(f'FAIL max_abs_error={err:.6g} boundaries_ok={boundaries} bounded={bounded}')
 print(f'PASS full field; max_abs_error={err:.6g}; boundaries and range correct')
EOF
```

### Create `scripts/summarize.py`

Timing a single run can be misleading. This script collects repeated runs, leaves out the warm-up and reports the median and range. It checks the saved fields again before writing the CSV. Read the raw rows as well as the summary, and keep the grid size, number of steps and worker layout beside every timing.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/summarize.py <<'EOF'
"""Collect actual RESULT rows; repeat 0 is a warmup, not an observation."""
from pathlib import Path
import csv,re,statistics,sys,subprocess
folder=Path(sys.argv[1]);rows=[]
for log in sorted(folder.glob('*-r*.log')):
 match=re.search(r'-r(\d+)\.log$',log.name)
 if not match or int(match[1])==0:continue
 lines=[s for s in log.read_text().splitlines() if s.startswith('RESULT ')]
 if len(lines)!=1:raise SystemExit(f'FAIL: expected exactly one RESULT in {log.name}; inspect MPI launcher')
 values=dict(piece.split('=',1) for piece in lines[0].split()[1:]);values['mode']=log.name.rsplit('-r',1)[0]
 values['repeat']=int(match[1]);values['log']=log.name;values['job_dir']=str(folder.resolve())
 check=[sys.executable,str(Path(__file__).with_name('validate.py')),str(folder/'reference-r0.bin'),str(log.with_suffix('.bin')),values['n']]
 if values['mode']=='library_v2':check.append('--expect-different')
 subprocess.run(check,check=True,capture_output=True,text=True)
 values['scientific_status']='EXPECTED_MISMATCH' if values['mode']=='library_v2' else 'PASS'
 rows.append(values)
if not rows:raise SystemExit('NOT RUN: no measured repeats found')
with (folder/'measurements.csv').open('w',newline='') as f:
 w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
for mode in sorted({r['mode'] for r in rows}):
 times=[float(r['loop_s']) for r in rows if r['mode']==mode]
 print(f'{mode:15} repeats={len(times)} median_loop_s={statistics.median(times):.9f} range=[{min(times):.9f}, {max(times):.9f}]')
print('Saved',folder/'measurements.csv')
EOF
```

### Create `scripts/submit.sh`

Submitting a job and running a program are separate actions. This script first copies your source and launch scripts to a new folder, then asks Slurm to run them. The returned job ID lets you follow the job in the queue. The copy matters because you might edit your working files while the job is waiting; a queued experiment must keep the code it was submitted with.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/submit.sh <<'EOF'
#!/bin/bash
# Freeze all code and scripts into each submission. Queued jobs never read edited files.
set -euo pipefail
cd "$(dirname "$0")/.."
device=${1:-cpu};stage=${2:-all}
case "$device" in cpu|gpu|paraview|profile) ;; *) echo 'Use cpu, gpu, paraview or profile' >&2;exit 2;;esac
stamp=$(date -u +%Y%m%dT%H%M%S)-$$
run="$PWD/runs/$device-$stage-$stamp"
mkdir -p "$run";cp -r src scripts "$run/"
cd "$run"
# A notebook/interactive Slurm session can export incompatible memory variables.
unset SLURM_MEM_PER_CPU SLURM_MEM_PER_GPU SLURM_MEM_PER_NODE
export STAGE=$stage
export MPI_LAUNCHER=${MPI_LAUNCHER:-auto}
job=$(sbatch --parsable "scripts/$device.sbatch");job=${job%%;*}
[[ "$job" =~ ^[0-9]+$ ]] || { echo "Unexpected sbatch output: $job" >&2;exit 2; }
printf '%s\n' "$job" >job_id.txt
printf '%s\n' "$PWD" >../../last_run_path.txt
printf 'Job %s\nFrozen run: %s\nCheck: squeue -j %s\n' "$job" "$PWD" "$job"
EOF
```

### Create `scripts/visualize.py`

The field file contains numbers, not a picture. This helper turns those numbers into a heatmap and a centre-column profile, then plots the recorded timings. It uses the same temperature colour scale for each field. Display sampling keeps the picture small; numerical validation still uses every cell. These SVG previews are separate from the ParaView rendering exercise.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/visualize.py <<'EOF'
"""Plots from saved doubles and CSV; standard library only, no Matplotlib required."""
import array,csv,html,math,statistics,sys
from pathlib import Path

def heat_svg(values,n,title='Temperature field',width=430):
 if len(values)!=n*n or not all(math.isfinite(x) for x in values):raise ValueError('Invalid heat field')
 stride=max(1,math.ceil(n/64));size=320;cell=size/math.ceil(n/stride)
 parts=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="410" viewBox="0 0 {width} 410">',
 '<rect width="100%" height="100%" fill="white"/>',f'<text x="20" y="25" font-family="sans-serif" font-size="18">{html.escape(title)}</text>']
 for row,y in enumerate(range(0,n,stride)):
  for col,x in enumerate(range(0,n,stride)):
   v=max(0,min(1,values[y*n+x]/100));color=f'rgb({int(255*v)},{int(90*v)},{int(150*(1-v))})'
   parts.append(f'<rect x="{20+col*cell:.2f}" y="{45+row*cell:.2f}" width="{cell+.1:.2f}" height="{cell+.1:.2f}" fill="{color}"/>')
 parts += ['<text x="20" y="390" font-family="sans-serif" font-size="13">0 °C (blue) to 100 °C (red). Top edge is y=0.</text>','</svg>']
 return ''.join(parts)

def write_plots(folder):
 folder=Path(folder);records=list(csv.DictReader((folder/'measurements.csv').open()))
 correct=[r for r in records if r['scientific_status']=='PASS']
 if not correct:raise ValueError('No validated records')
 n=int(correct[0]['n']);a=array.array('d');a.frombytes((folder/'reference-r0.bin').read_bytes())
 (folder/'heat.svg').write_text(heat_svg(a,n,f'Final temperature, N={n}'))
 # Centre-column profile from the actual saved reference, sampled only for display.
 points=' '.join(f'{40+a[y*n+n//2]*3.2:.2f},{40+300*y/(n-1):.2f}' for y in range(n))
 svg=f'<svg xmlns="http://www.w3.org/2000/svg" width="440" height="400"><rect width="100%" height="100%" fill="white"/><text x="30" y="24" font-family="sans-serif" font-size="18">Centre-column temperature</text><path d="M40 40 V340 H360" fill="none" stroke="grey"/><polyline points="{points}" fill="none" stroke="#124b78" stroke-width="2"/><text x="40" y="370" font-family="sans-serif">0 °C</text><text x="300" y="370" font-family="sans-serif">100 °C</text><text x="5" y="43" font-family="sans-serif">0</text><text x="5" y="340" font-family="sans-serif">{n-1}</text></svg>'
 (folder/'profile.svg').write_text(svg)
 modes=list(dict.fromkeys(r['mode'] for r in correct));height=75+38*len(modes)
 chart=[f'<svg xmlns="http://www.w3.org/2000/svg" width="850" height="{height}"><rect width="100%" height="100%" fill="white"/><text x="20" y="24" font-family="sans-serif" font-size="18">Measured update-loop time: dots and median (seconds)</text>']
 all_times=[float(r['loop_s']) for r in correct];max_time=max(all_times) or 1
 for k,m in enumerate(modes):
  vals=[float(r['loop_s']) for r in correct if r['mode']==m];median=statistics.median(vals);y=55+k*38
  chart.append(f'<text x="20" y="{y+4}" font-family="sans-serif">{html.escape(m)}</text><path d="M165 {y} H{165+480*median/max_time:.2f}" stroke="#124b78" stroke-width="5"/>')
  for v in vals:chart.append(f'<circle cx="{165+480*v/max_time:.2f}" cy="{y}" r="3" fill="#7d8790"/>')
  chart.append(f'<text x="670" y="{y+4}" font-family="sans-serif" font-size="12">{median:.6g} [{min(vals):.3g}, {max(vals):.3g}]</text>')
 chart.append('</svg>');(folder/'timings.svg').write_text(''.join(chart))
 return [folder/'heat.svg',folder/'profile.svg',folder/'timings.svg']

if __name__=='__main__':
 for p in write_plots(sys.argv[1]):print(p)
EOF
```

### Create `scripts/plot.py`

This short command calls the plotting helper for a completed run folder. It reads existing files and does not rerun the solver. Check the folder path carefully so that the image and timing table belong to the same experiment. Open the generated SVG files in Jupyter to inspect them.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/plot.py <<'EOF'
"""Generate temperature, profile and timing SVGs without extra Python packages."""
from visualize import write_plots
import sys
for path in write_plots(sys.argv[1]):print(path)
EOF
```

### Create `scripts/build.sh`

The compiler translates source into executable programs. This script builds the variants with explicit flags: `-O0` keeps optimisation off, `-O2` enables it, and `-fopenmp` enables threading. MPI uses `mpicc` to supply its headers and libraries. Compare flags before attributing a speed difference to parallelism. The job records hashes so that you can identify the exact binaries used.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/build.sh <<'EOF'
#!/bin/bash
set -euo pipefail
cd "$(dirname "$0")/.."
mkdir -p bin lib/v1 lib/v2 lib/fixed
stage=${1:-cpu}
# CC can select the C compiler; keep it consistent with the MPI stack.
cc=${CC:-gcc}
# Build the same mathematics with different copy, optimisation and threading choices.
if [[ "$stage" == serial || "$stage" == cpu ]]; then
 "$cc" -std=c11 -O0 -g -Wall -Wextra src/heat_naive.c -o bin/heat_naive
 "$cc" -std=c11 -O0 -g -Wall -Wextra src/heat_cpu.c -o bin/heat_serial_O0
 "$cc" -std=c11 -O2 -g -Wall -Wextra src/heat_cpu.c -o bin/heat_serial
 "$cc" -std=c11 -O2 -g -Wall -Wextra -fopenmp src/heat_cpu.c -o bin/heat_omp
fi
# mpicc supplies MPI headers/libraries; -fopenmp adds local threads.
if [[ "$stage" == mpi || "$stage" == cpu ]]; then
 command -v mpicc
 mpicc -std=c11 -O2 -g -Wall -Wextra src/heat_mpi.c -o bin/heat_mpi
 mpicc -std=c11 -O2 -g -Wall -Wextra -fopenmp src/heat_mpi.c -o bin/heat_hybrid
fi
# Build interchangeable shared objects, then link one unchanged client executable.
if [[ "$stage" == libraries || "$stage" == cpu ]]; then
 for pair in 'v1 temperature_v1.c' 'v2 temperature_v2.c' 'fixed temperature_v2_fixed.c'; do
  read -r version source <<< "$pair"
  "$cc" -O2 -fPIC -shared "src/$source" -o "lib/$version/libtemperature.so"
 done
 "$cc" -std=c11 -O2 -g -DHOT_FROM_LIBRARY src/heat_cpu.c -Llib/v1 -ltemperature -o bin/heat_library
fi
# CUDA and OpenACC use different compilers but solve the same heat problem.
if [[ "$stage" == gpu ]]; then
 command -v nvcc
 nvcc -O2 -lineinfo src/heat_cuda.cu -o bin/heat_cuda
 if command -v nvc >/dev/null; then
  nvc -O2 -acc=gpu -Minfo=accel src/heat_acc.c -o bin/heat_acc 2>bin/openacc-build.txt
 else
  echo 'SKIPPED OpenACC: nvc is not available' >bin/openacc-build.txt
 fi
 "$cc" -std=c11 -O2 src/heat_cpu.c -o bin/heat_serial
fi
case "$stage" in serial|mpi|libraries|cpu|gpu) ;; *) echo "Unknown build stage: $stage" >&2; exit 2;; esac
EOF
```

### Create `scripts/cpu.sbatch`

The lines starting with `#SBATCH` ask Slurm for one node, two tasks and two CPU cores per task. This reserves four cores in total. The script compiles the programs, checks the MPI launcher with a small calculation, then runs and validates each variant in order. The combined job uses two OpenMP threads for the OpenMP-only run and two ranks with two threads each for hybrid. That keeps every step within the allocation's CPUs per task. The earlier manual four-thread example uses its own one-task, four-core allocation.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/cpu.sbatch <<'EOF'
#!/bin/bash -l
#SBATCH --job-name=m5s1-ssh
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=2
#SBATCH --cpus-per-task=2
#SBATCH --mem=2G
#SBATCH --time=00:06:00
#SBATCH --output=cpu-%j.out
set -euo pipefail
cd "$SLURM_SUBMIT_DIR"
stage=${STAGE:-all};export LAB_DEVICE=cpu LAB_NEEDS_MPI=0
if [[ "$stage" == mpi || "$stage" == hybrid || "$stage" == all ]]; then LAB_NEEDS_MPI=1; fi
# Match the compiler/runtime environment before building or running binaries.
source scripts/site.sh
n=${N:-512};steps=${STEPS:-500};repeats=${REPEATS:-3}
# One evidence directory per job keeps logs and measurements associated.
out="results/cpu-$SLURM_JOB_ID";mkdir -p "$out"
{
 date -u;hostname;pwd;echo "JOB=$SLURM_JOB_ID STAGE=$stage N=$n STEPS=$steps REPEATS=$repeats"
 echo "MPI_LAUNCHER=${MPI_LAUNCHER:-srun} SRUN_MPI_TYPE=${SRUN_MPI_TYPE:-site-default}"
 scontrol show job "$SLURM_JOB_ID";gcc --version
 if [[ "$LAB_NEEDS_MPI" == 1 ]]; then mpicc --showme 2>/dev/null || mpicc -show; fi
 if type module >/dev/null 2>&1; then module list; fi
 env | sort | sed -n '/^OMP_/p;/^SLURM_/p'
 sha256sum src/* scripts/*.sh scripts/*.py scripts/*.sbatch
} >"$out/environment.txt" 2>&1
bash scripts/build.sh serial
if [[ "$LAB_NEEDS_MPI" == 1 ]]; then bash scripts/build.sh mpi; fi
if [[ "$stage" == libraries || "$stage" == all ]]; then bash scripts/build.sh libraries; fi
ldd bin/heat_omp >"$out/openmp-libraries.txt"
if [[ "$LAB_NEEDS_MPI" == 1 ]];then ldd bin/heat_mpi >"$out/mpi-libraries.txt";fi
sha256sum bin/* >"$out/binary-hashes.txt"
if [[ "$stage" == libraries || "$stage" == all ]]; then
 sha256sum lib/*/*.so >"$out/library-hashes.txt"
fi
if [[ "$LAB_NEEDS_MPI" == 1 && ${MPI_LAUNCHER:-auto} == auto ]]; then
 echo 'STEP MPI launcher preflight'
 command -v timeout
 export LAB_EXPECT_RANKS=2
 bin/heat_serial 11 2 "$out/launcher-reference.bin" >"$out/launcher-reference.log"
 found=0
 for candidate in srun mpirun; do
  if [[ "$candidate" == srun ]]; then
   launch=(srun --ntasks=2 --cpus-per-task=1 --cpu-bind=cores)
   if [[ -n ${SRUN_MPI_TYPE:-} ]]; then launch+=("--mpi=$SRUN_MPI_TYPE"); fi
  else
   command -v mpirun >/dev/null || continue
   launch=(mpirun -np 2 --map-by slot:PE=1 --bind-to core)
  fi
  if timeout 25s "${launch[@]}" bin/heat_mpi 11 2 "$out/launcher-$candidate.bin" >"$out/launcher-$candidate.log" 2>&1 &&
    python3 scripts/validate.py "$out/launcher-reference.bin" "$out/launcher-$candidate.bin" 11 >>"$out/launcher-$candidate.log" 2>&1; then
   export MPI_LAUNCHER=$candidate;found=1;echo "CHECKPOINT PASS MPI launcher=$candidate";break
  fi
  echo "Launcher $candidate did not pass; log: $out/launcher-$candidate.log"
 done
 [[ "$found" == 1 ]] || { echo 'FAIL no validated MPI launcher';exit 2; }
 echo "VALIDATED_MPI_LAUNCHER=$MPI_LAUNCHER" >>"$out/environment.txt"
fi

# All stages make an independent serial reference. Repeat 0 warms up; do not plot it.
bash scripts/measure.sh reference 1 1 bin/heat_serial "$n" "$steps" "$out" 0
modes=()
case "$stage" in
 serial) modes=(naive serial_O0 serial);;
 openmp) modes=(serial openmp);;
 mpi) modes=(serial mpi);;
 hybrid) modes=(serial mpi openmp hybrid);;
 libraries) modes=(library_v1 library_v2 library_fixed);;
 all) modes=(naive serial_O0 serial openmp mpi hybrid library_v1 library_v2 library_fixed);;
 *) echo 'STAGE must be serial/openmp/mpi/hybrid/libraries/all' >&2;exit 2;;
esac
# Run configurations sequentially so they do not compete for the same cores.
for mode in "${modes[@]}"; do
 echo "STEP $mode: compile complete; execute and validate"
 ranks=1;threads=1
 case "$mode" in
 naive) exe=bin/heat_naive;; serial_O0) exe=bin/heat_serial_O0;; serial) exe=bin/heat_serial;;
 mpi) exe=bin/heat_mpi;ranks=2;; openmp) exe=bin/heat_omp;threads=2;; hybrid) exe=bin/heat_hybrid;ranks=2;threads=2;;
 library_*) exe=bin/heat_library;version=${mode#library_};
 export LD_LIBRARY_PATH="$PWD/lib/$version${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"
 ldd bin/heat_library >"$out/$mode-ldd.txt";;
 esac
 # Repeat zero warms up; measured repetitions start at one.
 for ((r=0;r<=repeats;r++)); do
  bash scripts/measure.sh "$mode" "$ranks" "$threads" "$exe" "$n" "$steps" "$out" "$r"
  # Only the deliberate Kelvin error is expected to differ from serial.
  if [[ "$mode" == library_v2 ]]; then
   python3 scripts/validate.py "$out/reference-r0.bin" "$out/$mode-r$r.bin" "$n" --expect-different
  else
   python3 scripts/validate.py "$out/reference-r0.bin" "$out/$mode-r$r.bin" "$n"
  fi
 done
 echo "CHECKPOINT PASS $mode"
done
python3 scripts/summarize.py "$out"
echo "CPU_GUIDED_LAB_DONE $out"
EOF
```

### Submit a repeatable serial experiment

Back on the login node, in `~/hpc-labs/m5s1-heat`:

```bash
N=512 STEPS=500 REPEATS=3 bash scripts/submit.sh cpu serial
cat last_run_path.txt
run_dir=$(cat last_run_path.txt)
job_id=$(cat "$run_dir/job_id.txt")
squeue -j "$job_id"
scontrol show job "$job_id"
```

Do not submit again just because a job is pending. Wait for it to finish, or inspect its reason in `squeue`. It continues if your SSH connection closes. To cancel a job you no longer need, use `scancel "$job_id"`.

After completion:

```bash
run_dir=$(cat last_run_path.txt)
job_id=$(cat "$run_dir/job_id.txt")
sacct -j "$job_id" --format=JobID,State,ExitCode,Elapsed,AllocCPUS,MaxRSS
less "$run_dir/cpu-$job_id.out"
less "$run_dir/results/cpu-$job_id/environment.txt"
cat "$run_dir/results/cpu-$job_id/measurements.csv"
printf 'Serial job: %s; saved in %s\n' "$job_id" "$run_dir" >> notes/observations.txt
```

Look for `CPU_GUIDED_LAB_DONE`, successful Slurm status and full-field checks. If accounting is delayed, keep stdout and check again. A missing completion marker is not a pass.

For the measurements, repeat 0 is excluded. Repeats 1-3 are separate process executions. This warms filesystem/code caches but does not preserve a CUDA context across GPU repeats. Increase to five repeats for a report. The median and range help show variability. Extremely short durations are dominated by noise and overhead.

`loop_s` excludes field output. The `WALL` record includes the launcher and program. GNU `/usr/bin/time` also records peak RSS; the Bash fallback marks RSS unavailable. Reporting precision differs between timers. Job accounting measures broader Slurm steps. Do not mix these into one speedup calculation.

Check your understanding. Which comparison isolates pointer swapping? Naive `-O0` versus serial pointer swap `-O0`. Which isolates compiler flags? Serial pointer swap `-O0` versus serial pointer swap `-O2`.

### Checkpoint 4 - Serial batch

Wait for the submitted job to finish, then run this in SSH from `~/hpc-labs/m5s1-heat`.

```bash
run_dir=$(cat last_run_path.txt)
job_id=$(cat "$run_dir/job_id.txt")
grep 'CPU_GUIDED_LAB_DONE' "$run_dir/cpu-$job_id.out"
python3 "$run_dir/scripts/summarize.py" "$run_dir/results/cpu-$job_id"
grep 'serial' "$run_dir/results/cpu-$job_id/measurements.csv"
```

The completion marker and the full-field checks must be present. A job that is still queued has not passed this checkpoint. Keep the stdout log and CSV before moving on.

## 6. Divide the plate with MPI

So far all data belongs to one process. MPI gives us processes with separate address spaces. A process cannot read a neighbour's pointer, even when both run on the same node.

We divide the interior rows between ranks. If there are $M=N-2$ interior rows and $P$ ranks, rank $r$ owns

$$q=\lfloor M/P\rfloor,\qquad n_r=q+[r<M\bmod P],\qquad first_r=1+rq+\min(r,M\bmod P).$$

For N=11 and P=2, the nine interior rows split 5 + 4. The program supports this remainder. It rejects more ranks than interior rows.

Each rank allocates its owned rows plus a halo above and below. At every time step:

1. Send the first owned row upwards and receive the lower neighbour's row into the bottom halo.
2. Send the last owned row downwards and receive the upper neighbour's row into the top halo.
3. Update owned interior cells from the old field, then swap pointers.

`MPI_Sendrecv` makes paired transfers without relying on unsafe send ordering. Physical top/bottom boundaries use `MPI_PROC_NULL` and retain their values. `MPI_Gatherv` collects unequal row counts at the end; it does not run on every time step.

Two ranks on one node still have separate memory. This core lab demonstrates that model within a four-core budget. For a multi-node experiment, first confirm the supported MPI stack and launcher with SciTech. We use one node for this exercise.

### Create `src/heat_mpi.c`

Each MPI rank owns a different set of rows, plus two halo rows that hold copies from its neighbours. Read the two `MPI_Sendrecv` calls and identify the first and last owned rows. Once the halo data arrives, the rank updates its own cells. `MPI_Gatherv` joins the pieces only at the end so that we can save and check the complete field. With OpenMP enabled, threads share each rank's local work; MPI calls still happen on the original thread.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > src/heat_mpi.c <<'EOF'
/* Interior row decomposition; same stencil, optional OpenMP, MPI outside threads. */
#include "heat_common.h"
#include <mpi.h>
#ifdef _OPENMP
#include <omp.h>
#endif
int main(int argc,char **argv){
 int provided,rank,p;double total_start=clock_seconds();
 /* FUNNELED means only the original thread makes MPI calls. */
 MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&provided);
 MPI_Comm_rank(MPI_COMM_WORLD,&rank);MPI_Comm_size(MPI_COMM_WORLD,&p);
 /* Catch a launcher that accidentally starts independent one-rank programs. */
 if(getenv("LAB_EXPECT_RANKS") && p!=atoi(getenv("LAB_EXPECT_RANKS"))){
  if(!rank)fprintf(stderr,"MPI launcher mismatch: expected %s ranks, got %d. Check site launcher.\n",getenv("LAB_EXPECT_RANKS"),p);
  MPI_Abort(MPI_COMM_WORLD,2);
 }
 if(provided<MPI_THREAD_FUNNELED)MPI_Abort(MPI_COMM_WORLD,2);
 if(argc!=4){if(!rank)fprintf(stderr,"usage: heat_mpi N STEPS field.bin\n");MPI_Abort(MPI_COMM_WORLD,2);}
 int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000),interior=n-2;
 if(p>interior){if(!rank)fprintf(stderr,"Use at most N-2 ranks\n");MPI_Abort(MPI_COMM_WORLD,2);}
 /* Divide N-2 interior rows; early ranks receive any leftover rows. */
 int rows=interior/p+(rank<interior%p),first=1+rank*(interior/p)+(rank<interior%p?rank:interior%p);
 /* Each process stores its owned rows and one halo row on either side. */
 size_t cells=(size_t)(rows+2)*n;
 double *old=(double*)calloc(cells,sizeof(double)),*next=(double*)calloc(cells,sizeof(double));
 if(!old||!next)MPI_Abort(MPI_COMM_WORLD,3);
 /* Rank 0 keeps the physical hot edge in its upper halo. */
 if(rank==0)for(int x=0;x<n;x++){old[x]=100;next[x]=100;}
 /* MPI_PROC_NULL marks a physical edge where there is no neighbour. */
 int up=rank?rank-1:MPI_PROC_NULL,down=rank==p-1?MPI_PROC_NULL:rank+1,threads=1;
 const char *mode="mpi";
#ifdef _OPENMP
 mode="hybrid";
 #pragma omp parallel
 {
  #pragma omp single
  threads=omp_get_num_threads();
 }
#endif
 char host[MPI_MAX_PROCESSOR_NAME];int host_len;MPI_Get_processor_name(host,&host_len);
 /* Print ownership so we can compare the predicted layout with the actual run. */
 printf("MAP rank=%d ranks=%d host=%s first_global_row=%d rows=%d threads=%d\n",rank,p,host,first,rows,threads);
 /* Start the measured loop after every rank has finished initialisation. */
 MPI_Barrier(MPI_COMM_WORLD);double setup=clock_seconds()-total_start;
 double t0=MPI_Wtime(),halo=0,update=0;
 for(int s=0;s<steps;s++){
  double h=MPI_Wtime();
  /* Send our top owned row up; receive the lower neighbour into our bottom halo. */
  MPI_Sendrecv(old+n,n,MPI_DOUBLE,up,10,old+(size_t)(rows+1)*n,n,MPI_DOUBLE,down,10,MPI_COMM_WORLD,MPI_STATUS_IGNORE);
  /* Send our bottom owned row down; receive the upper neighbour into our top halo. */
  MPI_Sendrecv(old+(size_t)rows*n,n,MPI_DOUBLE,down,11,old,n,MPI_DOUBLE,up,11,MPI_COMM_WORLD,MPI_STATUS_IGNORE);
  halo+=MPI_Wtime()-h;double u=MPI_Wtime();
#ifdef _OPENMP
  /* OpenMP shares only this process's local rows. MPI stays outside this region. */
  #pragma omp parallel for schedule(static)
#endif
  for(int y=1;y<=rows;y++)for(int x=1;x<n-1;x++){
   size_t k=(size_t)y*n+x;next[k]=0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);
  }
  /* Both halo exchanges and the local update have finished before roles swap. */
  double *tmp=old;old=next;next=tmp;update+=MPI_Wtime()-u;
 }
 /* Maximum rank duration excludes gather and output; phase maxima need not add up. */
 double local[3]={MPI_Wtime()-t0,halo,update},times[3];
 MPI_Reduce(local,times,3,MPI_DOUBLE,MPI_MAX,0,MPI_COMM_WORLD);
 double io_start=clock_seconds();double *global=NULL;int *counts=NULL,*offsets=NULL;
 if(rank==0){
  global=grid(n,100);counts=(int*)malloc(p*sizeof(int));offsets=(int*)malloc(p*sizeof(int));
  if(!counts||!offsets)MPI_Abort(MPI_COMM_WORLD,3);
  for(int r=0;r<p;r++){
   counts[r]=(interior/p+(r<interior%p))*n;
   offsets[r]=(1+r*(interior/p)+(r<interior%p?r:interior%p))*n;
  }
 }
 /* Counts and offsets allow ranks to own different numbers of rows.
    Only root allocates the full global field, and gathering happens once. */
 MPI_Gatherv(old+n,rows*n,MPI_DOUBLE,global,counts,offsets,MPI_DOUBLE,0,MPI_COMM_WORLD);
 if(rank==0){
  /* Validate this complete field against serial, not just its checksum. */
  save_grid(argv[3],global,n);
  result(mode,n,steps,p,threads,setup,times[0],times[1],times[2],clock_seconds()-io_start,clock_seconds()-total_start,global);
 }
 free(old);free(next);free(global);free(counts);free(offsets);MPI_Finalize();return 0;
}
EOF
```

### Compile, inspect and submit MPI

From SSH on the login node:

```bash
cd ~/hpc-labs/m5s1-heat
module avail OpenMPI
module load OpenMPI/4.1.5-GCC-12.3.0
which mpicc
mpicc --showme
mpicc -std=c11 -O2 -g -Wall -Wextra src/heat_mpi.c -o bin/heat_mpi
srun --mpi=list
N=512 STEPS=500 REPEATS=3 bash scripts/submit.sh cpu mpi
```

`mpicc` is a wrapper that selects MPI headers and link libraries. It does not launch ranks. The batch script compiles again within its frozen directory so the binary matches the submitted source.

The submission script checks the MPI launcher with a tiny field comparison before the main run. It tries `srun`, then `mpirun` if necessary. Look for two MAP lines with ranks 0 and 1. The code rejects communicator-size mismatch, so an incompatible launcher cannot quietly report independent singleton processes as an MPI pass.

If the site-supported route is Open MPI's launcher, explicitly select it before a new submission:

```bash
MPI_LAUNCHER=mpirun N=512 STEPS=500 REPEATS=3 bash scripts/submit.sh cpu mpi
```

If SciTech instead requires an `srun` MPI plugin, set `SRUN_MPI_TYPE` to the value confirmed by the administrator and listed by `srun --mpi=list`. Do not guess a plugin or launch MPI through SSH. The chosen launcher is recorded in `environment.txt`.

Use the queue/log/accounting commands from the batch-job exercise. Then test N=11, STEPS=31 in a new submission to exercise uneven row ownership. Its final field must still match serial.

Why does a halo contain a copy, rather than an additional part of the plate that this rank owns?

### Checkpoint 5 - MPI ownership

Wait for the submitted job to finish, then run this in SSH from `~/hpc-labs/m5s1-heat`.

```bash
run_dir=$(cat last_run_path.txt)
job_id=$(cat "$run_dir/job_id.txt")
grep 'CPU_GUIDED_LAB_DONE' "$run_dir/cpu-$job_id.out"
python3 "$run_dir/scripts/summarize.py" "$run_dir/results/cpu-$job_id"
grep 'mpi' "$run_dir/results/cpu-$job_id/measurements.csv"
```

The completion marker and the full-field checks must be present. A job that is still queued has not passed this checkpoint. Keep the stdout log and CSV before moving on.

For N=11, inspect the MAP lines for five owned rows on rank 0 and four on rank 1.

## 7. Combine MPI and OpenMP

Now each MPI rank owns a block of rows and its threads share the local arrays. The global grid and number of time steps remain identical.

```bash
cd ~/hpc-labs/m5s1-heat
mpicc -std=c11 -O2 -g -Wall -Wextra -fopenmp src/heat_mpi.c -o bin/heat_hybrid
N=512 STEPS=500 REPEATS=3 bash scripts/submit.sh cpu hybrid
```

This stage runs optimised serial, 2 ranks × 1 thread, 1 rank × 2 OpenMP threads, and 2 ranks × 2 threads. It reserves one four-core allocation. Serial and MPI-only intentionally use fewer of those reserved cores: this compares execution configurations, not equal resource consumption. For throughput or core-hour accounting, include the full allocation.

For only serial versus OpenMP:

```bash
N=512 STEPS=500 REPEATS=3 bash scripts/submit.sh cpu openmp
```

Try one controlled edit: in working `scripts/cpu.sbatch`, change the OpenMP mode's `threads=2` to `threads=1`, then submit again. You now have the same OpenMP binary with one versus two threads. Preserve the first run folder. This is more informative about thread overhead than comparing different compiler options.

For matching inputs and timing scope, speedup is $S=T_{serial}/T_{variant}$. Record ranks, threads and allocated cores beside it. If the variant is slower, keep that result and explain it. Small workloads, repeated parallel regions and halo transfers are plausible causes to investigate.

Questions

1. Can two MPI ranks share `old` by using the same variable name? No, each has its own address space.
2. Why is the pointer swap outside the parallel loop? All outputs must be ready before the next time level begins.
3. Does `OMP_NUM_THREADS=8` allocate eight cores? No. Slurm allocates resources; OpenMP controls threads within that allocation.

The separate manual OpenMP experiment can use four threads because it requests four CPUs for one task. This combined batch job reserves two tasks with two CPUs each. Keeping the step requests consistent avoids a Slurm warning and makes the worker layout easier to explain.

### Checkpoint 6 - Hybrid execution

Wait for the submitted job to finish, then run this in SSH from `~/hpc-labs/m5s1-heat`.

```bash
run_dir=$(cat last_run_path.txt)
job_id=$(cat "$run_dir/job_id.txt")
grep 'CPU_GUIDED_LAB_DONE' "$run_dir/cpu-$job_id.out"
python3 "$run_dir/scripts/summarize.py" "$run_dir/results/cpu-$job_id"
grep 'hybrid' "$run_dir/results/cpu-$job_id/measurements.csv"
```

The completion marker and the full-field checks must be present. A job that is still queued has not passed this checkpoint. Keep the stdout log and CSV before moving on.

Check two MPI ranks and two threads per rank. Both must use the same N and number of steps as the reference.

## 8. See the temperatures and measurements

Inspect the full field, centre-line profile and repeated timings. The notebook previews use standard-library SVG output, so missing plotting packages do not stop the experiment. The next exercise uses ParaView for the scientific visualisation pipeline.

After a manual CPU run, generate the previews with:

```bash
cd ~/hpc-labs/m5s1-heat
run_dir=$(cat last_run_path.txt)
job_id=$(cat "$run_dir/job_id.txt")
python3 "$run_dir/scripts/plot.py" "$run_dir/results/cpu-$job_id"
```

Open `heat.svg`, `profile.svg` and `timings.svg` in Jupyter. These are views of the saved results; full-field numerical validation remains a separate check. Predict how the profile changes with more time steps, then test it while keeping N fixed.

### Visualise the field with ParaView

We will now use a scientific visualisation application rather than drawing only a notebook heatmap. The solver's binary file contains temperatures without geometry or units metadata. We export it to VTK with dimensions, spacing and a named Temperature array, then let ParaView read it.

The ParaView job uses one allocated CPU core. It discovers an available ParaView module, records the selected version, opens the VTK field with `pvpython`, exports a centre-line CSV and VTI file, and saves a screenshot and a `.pvsm` state file. Headless rendering depends on the site's ParaView build; a rendering failure is reported, not treated as a successful image.

Inspect the available software in SSH.

```bash
module avail ParaView
module avail paraview
```

Once you have the exact module name, inspect it with `module show NAME`. In the batch job, `module list`, `which pvpython` and `pvpython --version` record what actually runs. `pvpython` is ParaView's Python interpreter; importing its modules into the ordinary notebook kernel is not equivalent.

Create the following files in the lab folder.

### Create `scripts/export_vtk.py`

The raw solver output stores only temperatures. VTK also describes the grid dimensions, coordinates and the name of the temperature array. This converter adds that information while preserving the values. Check that N by N points are exported. The coordinates use our dimensionless unit grid, and the array values are Celsius.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/export_vtk.py <<'EOF'
"""Export the native-double field to an ASCII VTK structured-points file."""
from pathlib import Path
import array,math,sys

def export(source,n,target):
 # Native doubles are in row-major order: x changes fastest, as VTK expects.
 a=array.array('d');a.frombytes(Path(source).read_bytes())
 if len(a)!=n*n or not all(math.isfinite(v) for v in a):raise ValueError('Invalid field')
 # Add geometry and a scalar name while preserving each temperature value.
 with Path(target).open('w') as f:
  f.write(f'# vtk DataFile Version 3.0\nHeat diffusion, Celsius\nASCII\nDATASET STRUCTURED_POINTS\nDIMENSIONS {n} {n} 1\nORIGIN 0 0 0\nSPACING 1 1 1\nPOINT_DATA {n*n}\nSCALARS Temperature double 1\nLOOKUP_TABLE default\n')
  for v in a:f.write(f'{v:.17g}\n')
 return {'n':n,'points':len(a),'minimum':min(a),'maximum':max(a)}
if __name__=='__main__':print(export(sys.argv[1],int(sys.argv[2]),sys.argv[3]))
EOF
```

### Create `scripts/paraview_heat.py`

This script runs with ParaView's own `pvpython`, which supplies the `paraview.simple` library. It opens the VTK field and checks the point count and temperature range. Then it samples a centre line, saves a VTI field, renders an image and records the pipeline as a state file. The state describes how to reproduce the view; keep the data files too. The notebook displays the actual saved PNG only after the ParaView check passes.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/paraview_heat.py <<'EOF'
"""Run with pvpython, not the notebook's Python interpreter."""
from pathlib import Path
import json,sys
from paraview.simple import (OpenDataFile,GetActiveViewOrCreate,Show,ColorBy,
 GetColorTransferFunction,Render,SaveScreenshot,SaveState,SaveData,PlotOverLine)
source=Path(sys.argv[1]).resolve();n=int(sys.argv[2]);out=Path(sys.argv[3]).resolve();out.mkdir(parents=True,exist_ok=True)
# Read the grid into ParaView and confirm the dimensions and scalar range.
field=OpenDataFile(str(source));field.UpdatePipeline()
info=field.GetDataInformation();points=info.GetNumberOfPoints()
# PointData exposes ParaView's Python array-information wrapper, not the raw VTK API.
array_info=field.PointData['Temperature']
if array_info is None:raise RuntimeError('Temperature array missing')
low,high=array_info.GetRange(0)
assert points==n*n,(points,n*n)
assert abs(low)<1e-10 and abs(high-100)<1e-10,(low,high)
# VTI stores the structured field; the CSV samples the centre column.
SaveData(str(out/'temperature.vti'),proxy=field)
line=PlotOverLine(Input=field)
geometry=line if hasattr(line,'Point1') else line.Source
geometry.Point1=[n//2,0,0];geometry.Point2=[n//2,n-1,0];geometry.Resolution=n-1
line.UpdatePipeline();SaveData(str(out/'centre-line.csv'),proxy=line)
# Build a view with a fixed 0-100 Celsius colour scale for honest comparisons.
view=GetActiveViewOrCreate('RenderView');display=Show(field,view)
ColorBy(display,('POINTS','Temperature'));GetColorTransferFunction('Temperature').RescaleTransferFunction(0,100)
display.SetScalarBarVisibility(view,True)
view.ViewSize=[1000,800];view.CameraParallelProjection=1
view.CameraPosition=[(n-1)/2,(n-1)/2,2*n];view.CameraFocalPoint=[(n-1)/2,(n-1)/2,0]
view.CameraViewUp=[0,-1,0];view.CameraParallelScale=n*.6
# The batch launcher requests offscreen rendering; save the actual rendered image.
Render(view);SaveScreenshot(str(out/'paraview-temperature.png'),view,ImageResolution=[1000,800])
# A state file saves the pipeline/view settings, not the input field itself.
SaveState(str(out/'heat.pvsm'))
assert (out/'paraview-temperature.png').is_file()
(out/'validation.json').write_text(json.dumps({'points':points,'n':n,'minimum':low,'maximum':high,'status':'PASS'},indent=2))
print('PARAVIEW_VALIDATION_PASS',points,'points, range',low,high)
EOF
```

### Create `scripts/paraview.sbatch`

Visualisation also needs software and compute resources. This job requests one CPU core, finds a ParaView module and records the version it loads. `pvpython` processes the validated CPU field without opening an interactive window. The cluster's ParaView build must support headless rendering. Missing software is reported as skipped, while a failed rendering is reported as a failure; neither is replaced with a pretend ParaView image.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/paraview.sbatch <<'EOF'
#!/bin/bash -l
#SBATCH --job-name=m5s1-paraview
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G
#SBATCH --time=00:06:00
#SBATCH --output=paraview-%j.out
set -euo pipefail
cd "$SLURM_SUBMIT_DIR"
out="results/paraview-$SLURM_JOB_ID";mkdir -p "$out"
# Make Environment Modules visible in this non-interactive shell.
if ! type module >/dev/null 2>&1; then
 for init in /etc/profile.d/modules.sh /usr/share/Modules/init/bash;do
  if [[ -r "$init" ]];then source "$init";fi
  if type module >/dev/null 2>&1;then break;fi
 done
fi
{
 date -u;hostname;scontrol show job "$SLURM_JOB_ID"
 if type module >/dev/null 2>&1;then module list;fi
} >"$out/environment.txt" 2>&1
if type module >/dev/null 2>&1; then
 # Inspect the site catalogue instead of inventing a ParaView module version.
 module -t avail ParaView paraview >"$out/available-modules.txt" 2>&1 || true
 selected=${PARAVIEW_MODULE:-}
 if [[ -z "$selected" ]];then
  selected=$(python3 - "$out/available-modules.txt" <<'PY'
import re,sys
text=open(sys.argv[1]).read()
items=re.findall(r'(?<!\S)([^\s:]*[Pp][Aa][Rr][Aa][Vv][Ii][Ee][Ww][^\s:()]*)',text)
items=[s for s in items if '/' in s and not s.startswith('/')]
print(sorted(set(items))[-1] if items else '')
PY
)
 fi
 if [[ -n "$selected" ]];then
  echo "STEP module load $selected"
  module load "$selected"
  echo "PARAVIEW_MODULE=$selected" >>"$out/environment.txt"
 fi
 module list >>"$out/environment.txt" 2>&1
fi
# A missing application is SKIPPED; an attempted but failed render is FAIL.
if ! command -v pvpython >/dev/null;then
 echo 'SKIPPED_PARAVIEW: pvpython unavailable; see available-modules.txt and environment.txt'
 exit 0
fi
command -v pvpython | tee -a "$out/environment.txt"
pvpython --version >>"$out/environment.txt" 2>&1
# Runs only after a validated CPU field exists. Both paths are on shared user storage.
python3 scripts/export_vtk.py "$HEAT_INPUT" "$N" "$out/temperature.vtk"
pvpython --force-offscreen-rendering scripts/paraview_heat.py "$out/temperature.vtk" "$N" "$out"
echo "PARAVIEW_DONE $out"
EOF
```

### Submit the ParaView job

Use the validated CPU run as input. The steps below read N from its recorded measurements rather than assuming a grid size.

```bash
cd ~/hpc-labs/m5s1-heat
cpu_run=$(cat last_run_path.txt)
cpu_job=$(cat "$cpu_run/job_id.txt")
field="$cpu_run/results/cpu-$cpu_job/reference-r0.bin"
field_n=$(python3 -c 'import csv,sys;print(next(csv.DictReader(open(sys.argv[1])))["n"])' "$cpu_run/results/cpu-$cpu_job/measurements.csv")
HEAT_INPUT="$field" N="$field_n" bash scripts/submit.sh paraview all
```

After completion, inspect `paraview-JOBID.out` and the result folder. ParaView must confirm N × N points and a temperature range from 0 to 100 °C. Open the PNG in Jupyter, or open `temperature.vti` in desktop ParaView. Use `heat.pvsm` to reload the pipeline while keeping its input files available.

If no ParaView module or `pvpython` is available, the job reports SKIPPED_PARAVIEW. The VTK conversion and lightweight notebook previews remain useful, but they do not count as a ParaView execution.

### Checkpoint - ParaView and file formats

Check for `PARAVIEW_DONE`, `validation.json`, `temperature.vti`, `centre-line.csv`, `paraview-temperature.png` and `heat.pvsm`. Compare the centre-line values with the original binary field. Explain why changing the file format should preserve the temperatures, and why the `.pvsm` file alone is not a copy of the simulation data.

See the [ParaView Python guide](https://docs.paraview.org/en/latest/Tutorials/SelfDirectedTutorial/batchPythonScripting.html) and [offscreen rendering documentation](https://www.paraview.org/paraview-docs/latest/cxx/Offscreen.html).

## 9. Run the same heat solver on a GPU

Complete the CPU work first. GPU access may queue for longer. This extension uses the same full double-precision heat field, not vector addition. Start with N=512 and 500 steps, keeping the boundaries unchanged.

CUDA makes allocation, copies and kernel launches explicit. OpenACC uses loop directives and a data region. In both, we copy arrays to device once, perform many time steps with resident data, and copy only the final field back. Two arrays still separate time levels.

This uses one GPU on one node. It does not add GPU-aware MPI or require all hardware levels to update the same cell simultaneously. A distributed GPU solver would also exchange halos between rank-owned device subdomains.

### Create `src/heat_cuda.cu`

A GPU cannot simply follow a pointer to the CPU's array. We allocate device arrays and copy the initial temperatures to them. Each CUDA thread updates one cell, and successive kernels calculate successive time steps. The data stays on the GPU between steps; only the final field comes back. Compare the device-loop time with the whole operation to see the cost of setup and movement.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > src/heat_cuda.cu <<'EOF'
/* One GPU heat solver. Both arrays stay on the device between time steps. */
#include "heat_common.h"
#include <cuda_runtime.h>
#define CUDA(call) do{cudaError_t e=(call);if(e!=cudaSuccess){fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e));exit(3);}}while(0)
__global__ void stencil(const double *old,double *next,int n){
 /* The block and thread indices give each GPU thread one grid coordinate. */
 int x=blockIdx.x*blockDim.x+threadIdx.x,y=blockIdx.y*blockDim.y+threadIdx.y;
 /* Skip fixed boundaries; each interior output cell has exactly one writer. */
 if(x>0&&x<n-1&&y>0&&y<n-1){size_t k=(size_t)y*n+x;next[k]=0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);}
}
int main(int argc,char **argv){
 if(argc!=4){fprintf(stderr,"usage: heat_cuda N STEPS field.bin\n");return 2;}
 int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000);
 /* Host arrays hold the initial grid and receive the final result. */
 double start=clock_seconds();double *a=grid(n,100),*b=grid(n,100),*da,*db;
 /* Allocate separate device memory; CPU pointers do not identify GPU storage. */
 size_t bytes=(size_t)n*n*sizeof(double);CUDA(cudaMalloc((void**)&da,bytes));CUDA(cudaMalloc((void**)&db,bytes));
 /* Transfer once, then reuse device-resident data for all time steps. */
 CUDA(cudaMemcpy(da,a,bytes,cudaMemcpyHostToDevice));CUDA(cudaMemcpy(db,b,bytes,cudaMemcpyHostToDevice));
 /* CUDA events measure the asynchronous device loop in its own stream. */
 cudaEvent_t e0,e1;CUDA(cudaEventCreate(&e0));CUDA(cudaEventCreate(&e1));
 double setup=clock_seconds()-start;dim3 block(16,16),blocks((n+15)/16,(n+15)/16);
 CUDA(cudaEventRecord(e0));
 /* Default-stream ordering keeps each step behind the previous kernel. */
 for(int s=0;s<steps;s++){
  /* Launch enough 16x16 blocks to cover the plate, including partial edge blocks. */
  stencil<<<blocks,block>>>(da,db,n);CUDA(cudaGetLastError());double *tmp=da;da=db;db=tmp;
 }
 /* Synchronise before reading the duration or copying results back. */
 CUDA(cudaEventRecord(e1));CUDA(cudaEventSynchronize(e1));float ms;CUDA(cudaEventElapsedTime(&ms,e0,e1));
 /* Copy the completed field back once and save it for CPU/GPU comparison. */
 double t=clock_seconds();CUDA(cudaMemcpy(a,da,bytes,cudaMemcpyDeviceToHost));save_grid(argv[3],a,n);
 double io=clock_seconds()-t;
 CUDA(cudaEventDestroy(e0));CUDA(cudaEventDestroy(e1));CUDA(cudaFree(da));CUDA(cudaFree(db));
 result("cuda",n,steps,1,1,setup,ms/1000.0,0,ms/1000.0,io,clock_seconds()-start,a);
 free(a);free(b);return 0;
}
EOF
```

### Create `src/heat_acc.c`

OpenACC describes which loops can run on the accelerator using directives around familiar C code. The data region keeps both arrays on the device for the whole time loop. Odd and even steps alternate which array supplies the input. Identify where the final array returns to the CPU. This solver should produce the same temperatures as CUDA and serial, even though its programming style is different.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > src/heat_acc.c <<'EOF'
/* OpenACC: fixed array identities, alternating read/write role; no pointer remapping. */
#include "heat_common.h"
#include <openacc.h>
int main(int argc,char **argv){
 if(argc!=4){fprintf(stderr,"usage: heat_acc N STEPS field.bin\n");return 2;}
 int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000);
 /* Refuse CPU fallback: this exercise must actually use an NVIDIA accelerator. */
 if(acc_get_num_devices(acc_device_nvidia)<1){fprintf(stderr,"No NVIDIA accelerator\n");return 3;}
 acc_set_device_type(acc_device_nvidia);
 double start=clock_seconds();double *a=grid(n,100),*b=grid(n,100);size_t count=(size_t)n*n;
 double setup=0,loop=0,copy=0;
 /* Keep both arrays on the GPU while the entire time loop runs. */
 #pragma acc data copyin(a[0:count],b[0:count])
 {
  setup=clock_seconds()-start;double t=clock_seconds();
  for(int s=0;s<steps;s++){
   /* Alternating array roles avoids changing pointer-to-device mappings. */
   if((s%2)==0){
    /* collapse(2) distributes both grid loops; present reuses mapped device data. */
    #pragma acc parallel loop collapse(2) present(a[0:count],b[0:count])
    for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++){
     size_t k=(size_t)y*n+x;b[k]=0.25*(a[k-n]+a[k+n]+a[k-1]+a[k+1]);
    }
   }else{
    #pragma acc parallel loop collapse(2) present(a[0:count],b[0:count])
    for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++){
     size_t k=(size_t)y*n+x;a[k]=0.25*(b[k-n]+b[k+n]+b[k-1]+b[k+1]);
    }
   }
  }
  /* Finish device work before recording the host-clock duration. */
  #pragma acc wait
  loop=clock_seconds()-t;t=clock_seconds();
  /* Odd steps end in b, even steps in a. Transfer only the final array. */
  if(steps%2){
   #pragma acc update self(b[0:count])
  }else{
   #pragma acc update self(a[0:count])
  }
  copy=clock_seconds()-t;
 }
 /* The same binary output format lets serial validate the whole GPU field. */
 double *final=(steps%2)?b:a;double t=clock_seconds();save_grid(argv[3],final,n);
 result("openacc",n,steps,1,1,setup,loop,0,loop,copy+clock_seconds()-t,clock_seconds()-start,final);
 free(a);free(b);return 0;
}
EOF
```

### Create `scripts/gpu.sbatch`

This job asks Slurm for a GPU before running accelerator code. It records the GPU and compiler information, builds CUDA and OpenACC, and runs a serial reference on that same node. Every GPU field is compared with that reference. If the device is busy, the job waits in the queue. A queued job has not yet demonstrated a working kernel.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/gpu.sbatch <<'EOF'
#!/bin/bash -l
#SBATCH --job-name=m5s1-heat-gpu
#SBATCH --partition=gpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --gpus=1
#SBATCH --mem=2G
#SBATCH --time=00:06:00
#SBATCH --output=gpu-%j.out
set -euo pipefail
cd "$SLURM_SUBMIT_DIR";export LAB_DEVICE=gpu
# The GPU compiler stack is selected inside the GPU allocation.
source scripts/site.sh
n=${N:-512};steps=${STEPS:-500};repeats=${REPEATS:-3}
out="results/gpu-$SLURM_JOB_ID";mkdir -p "$out"
{
 date -u;hostname;scontrol show job "$SLURM_JOB_ID";nvidia-smi;nvcc --version
 if type module >/dev/null 2>&1; then module list; fi
 sha256sum src/* scripts/*.sh scripts/*.py scripts/*.sbatch
} >"$out/environment.txt" 2>&1
# Compile on the assigned node and keep compiler diagnostics with this run.
bash scripts/build.sh gpu
sha256sum bin/* >"$out/binary-hashes.txt"
bash scripts/measure.sh reference 1 1 bin/heat_serial "$n" "$steps" "$out" 0
# The serial reference uses this same node, avoiding a different CPU model.
for mode in serial cuda openacc; do
 echo "STEP $mode: execute and validate"
 case "$mode" in serial) exe=bin/heat_serial;;cuda) exe=bin/heat_cuda;;openacc)
 if [[ ! -x bin/heat_acc ]]; then echo 'SKIPPED OpenACC: compiler unavailable';continue;fi
 exe=bin/heat_acc;;esac
 for ((r=0;r<=repeats;r++));do
  bash scripts/measure.sh "$mode" 1 1 "$exe" "$n" "$steps" "$out" "$r"
  python3 scripts/validate.py "$out/reference-r0.bin" "$out/$mode-r$r.bin" "$n"
 done
 echo "CHECKPOINT PASS $mode"
done
python3 scripts/summarize.py "$out"
echo "GPU_GUIDED_LAB_DONE $out"
EOF
```

### Submit the GPU job

From the login node:

```bash
cd ~/hpc-labs/m5s1-heat
module avail nvhpc
module load nvhpc/25.7
which nvcc
which nvc
N=512 STEPS=500 REPEATS=3 bash scripts/submit.sh gpu all
```

The module name is documented in the course validation notes. If changed, select the current supported module and update your `site.sh` before submission. Do not execute GPU work on the login node.

```bash
run_dir=$(cat last_run_path.txt)
job_id=$(cat "$run_dir/job_id.txt")
squeue -j "$job_id"
# After completion:
less "$run_dir/gpu-$job_id.out"
cat "$run_dir/bin/openacc-build.txt"
cat "$run_dir/results/gpu-$job_id/measurements.csv"
```

The serial reference runs in the same GPU-node allocation, avoiding a comparison against a different CPU type. Every candidate field must match serial. `nvc -Minfo=accel` records generated GPU code; without nvc, OpenACC is explicitly skipped. Missing CUDA stops the GPU stage.

When comparing timings, CUDA `loop_s` uses device events, OpenACC uses synchronized host wall time, and CPU uses host wall time. They cover the update loop but use different instrumentation. Keep these separate from `WALL`, the broader launcher-to-completion interval. Use your own SciTech measurements for the comparison.

Why can a fast kernel give a slow complete execution? What happens if we copy arrays every step? Why might a double-precision stencil show limited benefit on a GPU designed mainly for other arithmetic workloads?

### Checkpoint 7 - GPU heat solver

Wait for the submitted job to finish, then run this in SSH from `~/hpc-labs/m5s1-heat`.

```bash
run_dir=$(cat last_run_path.txt)
job_id=$(cat "$run_dir/job_id.txt")
grep 'GPU_GUIDED_LAB_DONE' "$run_dir/gpu-$job_id.out"
python3 "$run_dir/scripts/summarize.py" "$run_dir/results/gpu-$job_id"
grep 'cuda' "$run_dir/results/gpu-$job_id/measurements.csv"
```

The completion marker and the full-field checks must be present. A job that is still queued has not passed this checkpoint. Keep the stdout log and CSV before moving on.

CUDA must match the reference. OpenACC must either pass or explicitly report a missing compiler; SKIPPED is not a GPU pass.

## 10. Extension: software libraries and units

This exercise connects the solver to the software-environment topics from the course. Its purpose is to show why we record library versions and check results after changing an environment. It is not a technique for making the heat calculation faster.

We compile one solver executable that gets the hot-edge temperature from a shared library. We then change which library the runtime loader finds, without recompiling the solver. This lets us separate the executable from the software it depends on.

Version 1 returns 100 Celsius, as the solver expects. Version 2 deliberately returns 373.15 Kelvin under the same function name. It is an intentionally incompatible interface, not an example of a normal library upgrade. The program still runs, but the boundary and temperature field are wrong. A small adapter restores Celsius and the reference result.

The lesson is that successful loading does not prove scientific correctness. Record the resolved library, version, units and output checks when you change software stacks. This is a short extension to the hybrid-computing lesson; the main solver progression is serial, MPI, OpenMP and GPU execution.

### Create `src/temperature_v1.c`

This tiny shared library supplies the hot-edge temperature in Celsius. The solver calls its function instead of using a value compiled directly into the executable. We will keep the executable unchanged and switch which library the runtime loader finds.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > src/temperature_v1.c <<'EOF'
/* Contract: Celsius. */
/* The caller expects Celsius, so this returns the agreed physical value. */
double top_temperature(void){return 100.0;}
EOF
```

### Create `src/temperature_v2.c`

This deliberately incompatible version returns Kelvin under the same function name. It loads successfully, but the solver expects Celsius and interprets the number incorrectly. Predict the new top-edge value before running it. The expected mismatch demonstrates why successful compilation and execution do not guarantee a correct scientific result.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > src/temperature_v2.c <<'EOF'
/* Deliberate incompatible contract: Kelvin. Same symbol, different units. */
/* This is intentionally wrong for the caller: a loadable symbol can still use wrong units. */
double top_temperature(void){return 373.15;}
EOF
```

### Create `src/temperature_v2_fixed.c`

The adapter converts Kelvin to the Celsius value required by the solver. It restores the contract while keeping the executable unchanged. The corrected field should match the serial reference again. Explain what the function name alone failed to tell us about the units.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > src/temperature_v2_fixed.c <<'EOF'
/* Adapter restores the Celsius contract expected by this solver. */
/* Convert units at the library boundary so the existing solver remains correct. */
double top_temperature(void){return 373.15-273.15;}
EOF
```

### Compile and choose the library

Read the commands, then submit the library stage:

```bash
cd ~/hpc-labs/m5s1-heat
gcc -O2 -fPIC -shared src/temperature_v1.c -o lib/v1/libtemperature.so
gcc -O2 -fPIC -shared src/temperature_v2.c -o lib/v2/libtemperature.so
gcc -O2 -fPIC -shared src/temperature_v2_fixed.c -o lib/fixed/libtemperature.so
gcc -std=c11 -O2 -DHOT_FROM_LIBRARY src/heat_cpu.c -Llib/v1 -ltemperature -o bin/heat_library
LD_LIBRARY_PATH="$PWD/lib/v1" ldd bin/heat_library
LD_LIBRARY_PATH="$PWD/lib/v2" ldd bin/heat_library
N=256 STEPS=350 REPEATS=3 bash scripts/submit.sh cpu libraries
```

`-L` selects a link-time search directory. `-ltemperature` resolves `libtemperature.so`. `LD_LIBRARY_PATH` influences the runtime loader and can change the library without rebuilding the executable. Each library mode saves `ldd` output.

The batch expects v2 to mismatch, labels it `EXPECTED_MISMATCH`, and excludes it from correct-solution timing plots. Version 1 and the adapter must pass ordinary full-field validation.

Record the executable hash, resolved library path, library hash, units and numerical difference. Use `module list`, `which gcc`, `which mpicc`, `mpicc --showme` and `ldd` to explain the selected software. Never run `ldd` on an untrusted executable.

Does a program completing without an error establish scientific correctness? Explain using the Kelvin example.

### Checkpoint 8 - Library contract

Wait for the submitted job to finish, then run this in SSH from `~/hpc-labs/m5s1-heat`.

```bash
run_dir=$(cat last_run_path.txt)
job_id=$(cat "$run_dir/job_id.txt")
grep 'CPU_GUIDED_LAB_DONE' "$run_dir/cpu-$job_id.out"
python3 "$run_dir/scripts/summarize.py" "$run_dir/results/cpu-$job_id"
grep 'library_fixed' "$run_dir/results/cpu-$job_id/measurements.csv"
```

The completion marker and the full-field checks must be present. A job that is still queued has not passed this checkpoint. Keep the stdout log and CSV before moving on.

The unadapted Kelvin library must report EXPECTED_MISMATCH. Version 1 and the adapter must report PASS.

## 11. Find where the time goes

Start with the timers already present. MPI `halo_s` includes messaging and waiting. `update_s` measures stencil work and OpenMP loop overhead. Each is a maximum across ranks. Those maxima may belong to different ranks, so do not add them and call the sum the critical path. These fine-grained teaching timers add some overhead.

Rebuild a separate binary for profiling. `gprof` samples CPU execution: a short run may have too few samples, and inlining changes function attribution. It does not explain all MPI waiting or GPU execution. Profiled durations are not benchmark timings.

### Create `scripts/profile.sbatch`

The timing table tells us how long a version took; profiling helps us locate the work. We rebuild a separate binary with `-pg` and let `gprof` sample its execution. Look for time in the stencil and copy loops. Instrumentation changes runtime, so these durations must stay out of the speedup table. Too few samples make the profile inconclusive.

Create this file from `~/hpc-labs/m5s1-heat` in your SSH session. The final `EOF` finishes writing the file.

```bash
cat > scripts/profile.sbatch <<'EOF'
#!/bin/bash -l
#SBATCH --job-name=m5s1-profile
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G
#SBATCH --time=00:06:00
#SBATCH --output=profile-%j.out
set -euo pipefail
cd "$SLURM_SUBMIT_DIR";export LAB_DEVICE=cpu LAB_NEEDS_MPI=0
source scripts/site.sh
out="results/profile-$SLURM_JOB_ID";mkdir -p "$out"
command -v gprof >/dev/null || { echo 'SKIPPED_PROFILE: gprof unavailable'; exit 0; }
# Use the naive version so its explicit copying is visible in a sampling profile.
gcc -O0 -g -pg src/heat_naive.c -o "$out/heat_profile"
cd "$out"
if [[ -x /usr/bin/time ]]; then
 srun -n1 -c1 --cpu-bind=cores /usr/bin/time -v ./heat_profile 512 1000 field.bin >run.log 2>&1
else
 TIMEFORMAT='WALL elapsed_s=%R user_s=%U sys_s=%S maxrss_kb=UNAVAILABLE'
 { time srun -n1 -c1 --cpu-bind=cores ./heat_profile 512 1000 field.bin; } >run.log 2>&1
fi
gprof ./heat_profile gmon.out >gprof.txt
# Keep the binary/code/flags together. Profiling overhead disqualifies these timings from speedup claims.
echo "PROFILE_DONE $PWD"
EOF
```

### Submit a CPU profile

From login, snapshot this experiment too:

```bash
cd ~/hpc-labs/m5s1-heat
profile_dir="$PWD/runs/profile-$(date -u +%Y%m%dT%H%M%S)-$$"
mkdir -p "$profile_dir"
cp -r src scripts "$profile_dir/"
cd "$profile_dir"
unset SLURM_MEM_PER_CPU SLURM_MEM_PER_GPU SLURM_MEM_PER_NODE
sbatch scripts/profile.sbatch
```

After completion read `results/profile-JOBID/gprof.txt` and `run.log` in that folder. Examine user/system time, peak memory and hot functions. With loops inside `main`, the flat profile may attribute most compute there. `gprof -A ./heat_profile gmon.out`, from that result folder, requests source annotation where available. Too few samples means an inconclusive profile: extend the run within permitted limits.

Optional tools, inside the appropriate allocation:

```bash
# CPU allocation, if performance counters are permitted:
perf stat -e cycles,instructions,cache-misses ./bin/heat_serial 512 1000 results/perf.bin
# One-GPU allocation, if Nsight Systems is installed:
nsys profile --trace=cuda --output=results/heat-cuda ./bin/heat_cuda 512 500 results/profile-cuda.bin
```

These execute the program. If `perf` denies access or `nsys` is absent, record `UNAVAILABLE`; do not change kernel settings or install system software. `nsys` can show device transfers and kernel launches.

If most time lies in halo exchange, will adding local threads necessarily help? If output dominates wall time but is excluded from `loop_s`, which number describes the user's wait?

### Check your understanding

1. A four-core allocation runs 2 MPI ranks × 2 threads. What does each rank share?  
A. All arrays with every rank. B. Its local arrays with its own threads. C. Only the halo with its threads. D. No memory at all.

2. Which observation supports a correct hybrid calculation?  
A. Lower runtime. B. Two MAP lines. C. Agreement of the entire final field, boundaries and input parameters. D. High CPU usage.

3. CUDA reports a short device loop but a much longer shell wall time. What should you examine?  
A. Transfers, setup, launch and output. B. Only the grid colours. C. The number of notebook cells. D. Whether Slurm queued the job yesterday.

4. Replacing a shared library changes a result without changing the executable. What is the best response?  
A. Accept it because loading succeeded. B. Delete the old result. C. Check the resolved library, version, units and API contract. D. Increase the thread count.

<details><summary>Discuss before revealing the answers</summary>

1. B. OpenMP threads share their process's address space. MPI rank ownership remains separate.
2. C. Resource mapping and performance do not establish numerical correctness.
3. A. The full execution includes operations outside the timed kernel/update loop. Queue time is separate again.
4. C. A loadable binary interface can still conceal incompatible scientific semantics.

</details>

## 12. What to hand in

Save `notes/observations.txt`, frozen run folders, `measurements.csv` and the saved SVG plots. Keep raw logs and fields so somebody else can check your claims.

For each stage write a short paragraph with prediction, command, job ID, numerical check, measured median/range and explanation. A screenshot of a fast time is not enough. Explain one surprising result using saved evidence.

| Comparison | Keep fixed | What it investigates |
|---|---|---|
| naive O0 / swap O0 | N, steps, one core | Avoiding the full-grid copy |
| swap O0 / swap O2 | source, N, steps, one core | Compiler optimisation |
| OpenMP 1 / OpenMP 4 | binary, N, steps | Threading overhead and benefit |
| serial / MPI / hybrid | equation, N, steps, validation | Execution configurations |
| CPU / CUDA / OpenACC | field, N, steps, tolerance | Acceleration and movement cost |
| library v1 / v2 / adapter | executable and input | Library semantics and units |

For a final challenge, N=11 has nine interior rows. Explain their split across two ranks and which rank stores the physical hot boundary. Prove your answer using MAP lines and a complete-field check.

Further reading includes the [course access guide](https://github.com/OscarDiez/hpc_course/blob/main/cluster/README.md), [course validation notes](https://github.com/OscarDiez/hpc_course/blob/main/cluster/COURSE_VALIDATION_NOTES.md), [Open MPI thread support](https://docs.open-mpi.org/en/v5.0.0/man-openmpi/man3/MPI_Init_thread.3.html), [Slurm sbatch](https://slurm.schedmd.com/sbatch.html), [Slurm resource binding](https://slurm.schedmd.com/resource_binding.html), [NVIDIA OpenACC guide](https://docs.nvidia.com/hpc-sdk/compilers/openacc-gs/index.html). Module names are site-specific. The derivation above supplies the numerical definition used by the heat solvers.

## 13. Your larger experiment

Once your CPU checks pass, increase the work and find out whether the balance between computation and overhead changes. Write your prediction before each change. Use the same source files, compiler settings and worker layouts.

First save a small baseline experiment. Run this in SSH from your lab folder:

```bash
cd ~/hpc-labs/m5s1-heat
N=128 STEPS=150 REPEATS=3 bash scripts/submit.sh cpu all
run_dir=$(cat last_run_path.txt)
printf 'N=128 steps=150 folder=%s\n' "$run_dir" >> notes/experiment-folders.txt
```

Wait for the completion marker and check the saved fields and measurements using the batch-job checkpoint. Do not submit the next experiment until you have checked this one.

Now increase each grid dimension from 128 to 512 while keeping 150 time steps:

```bash
N=512 STEPS=150 REPEATS=3 bash scripts/submit.sh cpu all
run_dir=$(cat last_run_path.txt)
printf 'N=512 steps=150 folder=%s\n' "$run_dir" >> notes/experiment-folders.txt
```

Each dimension is four times larger, so the number of cells, approximate stencil work and storage per grid grow by about 16 times. The fixed launch cost does not grow in the same way. Predict which version will benefit most from the larger amount of computation.

After checking that run, keep the same grid and increase the number of time steps:

```bash
N=512 STEPS=600 REPEATS=3 bash scripts/submit.sh cpu all
run_dir=$(cat last_run_path.txt)
printf 'N=512 steps=600 folder=%s\n' "$run_dir" >> notes/experiment-folders.txt
```

This performs about four times as many updates as the 512-by-512, 150-step run, without increasing the size of the field. Heat also has more time to spread. Keep all three run folders so you can compare the results.

| Run | N | Steps | What changes |
|---|---|---|---|
| Baseline | 128 | 150 | Small correctness and workflow check |
| Larger grid | 512 | 150 | More cells and a larger field |
| Longer calculation | 512 | 600 | More updates of the same grid |

Keep the allocated CPUs, ranks, threads and GPU count unchanged. The six-minute job limit still applies. If a job times out, record that result and reduce the workload; a partial run is not a completed measurement.

If GPU resources are available, repeat the larger heat calculation on the GPU:

```bash
N=512 STEPS=600 REPEATS=3 bash scripts/submit.sh gpu all
```

Check CUDA and OpenACC against the serial reference from that GPU-node job. Use the ParaView exercise to inspect a validated field. Preserve its N, steps and run-folder path with the image.

For every run, record the job ID, numerical check, median and range of loop time, broader wall time and worker layout. Compare implementations using the same N and steps within that run. Do not compare the small-grid serial time with the large-grid GPU time and call it a speedup.

Answer these questions using your measurements:

1. Does the best-performing version change as the workload grows? Explain with timings, not just the number of workers.
2. How much time lies outside the update loop? Does that fraction change for the larger grid?
3. Why might reusing GPU-resident arrays matter more for a longer sequence of updates?
4. Do all correct implementations still agree on every cell? Why must this check come before the timing comparison?
5. How does the centre-line temperature profile change with 600 steps? Separate the temperature change from the timing change.

N describes the size of a dimensionless unit-spaced grid. Increasing it is not a fixed-physical-plate resolution study; that would also require changing spacing and the stable time step. These are workload-size experiments at fixed resources, not strong-scaling experiments.

---
# Teaching demonstration and automatic replay

The student exercise above uses SSH commands and checkpoints. This section provides a complete replay for preparing the class, demonstrating the results and checking the cluster environment.

Choose a connected Python 3 kernel in SciTech JupyterHub. Run the settings cell below and the following cells in order, or use Restart Kernel and Run All Cells for the whole notebook. The replay creates the source files, submits the CPU, ParaView, GPU and profiling jobs, waits for them and displays validated results.

Progress and saved job IDs appear as it runs. Keep the kernel connected while it waits. Repeating the replay reuses job IDs rather than duplicating submissions. The wait limit is 30 minutes per job; pending jobs remain in Slurm if that limit is reached.

The additional guided examples below are available with `RUN_GUIDED_EXAMPLE=True`. They are disabled by default to avoid submitting them as well as the complete heat experiment. Notebook previews use the Python standard library, while ParaView runs with its own interpreter and software module.

For a larger automatic demonstration, change `EXPERIMENT_N` to 512 in the settings below, then run the replay again. Keep 150 steps for the first comparison, then use 600 steps and three repeats. Changed inputs create a new experiment folder and preserve the small run.

## Check the demonstration kernel

The settings cell below first prints `M5S1 started: the Python kernel is executing this notebook.` It then shows the host, Python interpreter and location of `sbatch` before submitting any job.

If it stays at `In [ ]` with no output, select that cell and press Shift+Enter, then check the Python 3 kernel connection. An HTML export is only a saved view and cannot execute code. A running cell shows `In [*]`; an executed cell has a number.

Once submission starts, look for `Submitted cpu job ...` or `Reusing cpu job ...`, followed by queue status. A queued job has an ID and a reason such as Priority or Resources. It is different from a kernel that has not started executing cells.


In [ ]:
print("M5S1 started: the Python kernel is executing this notebook.", flush=True)
from pathlib import Path
import os, sys, json, re, time, uuid, hashlib, shutil, subprocess, statistics
from IPython.display import display, HTML, Markdown

# Run All executes the complete heat experiment at the end.
# The additional rank/thread and vector-add examples are kept for separate use.
RUN_GUIDED_EXAMPLE = False
RUN_COMPLETE_EXPERIMENT = True
RUN_GPU_HEAT = True
RUN_PARAVIEW = True
PARAVIEW_MODULE = ""  # empty: discover an advertised module; use an exact name to pin a version
RUN_CPU_PROFILE = True
EXPERIMENT_N = 128
EXPERIMENT_STEPS = 150
EXPERIMENT_REPEATS = 2
MAX_WAIT_SECONDS = 1800  # per job; queued jobs continue if this limit is reached
POLL_SECONDS = 5
MPI_LAUNCHER = "auto"   # tiny preflight validates srun, then mpirun if needed
NEW_EXPERIMENT = False  # reuses saved job IDs unless you deliberately change this
print("Notebook ready on", os.uname().nodename, flush=True)
print("Python interpreter:", sys.executable, flush=True)
print("Slurm sbatch:", shutil.which('sbatch') or 'NOT AVAILABLE', flush=True)
if not shutil.which('sbatch'):
    print("Cluster experiments cannot submit from this kernel. Open the notebook in SciTech JupyterHub.", flush=True)
print("Run All will create files, run CPU and GPU heat experiments, validate and plot results.")
print("Saved job IDs prevent duplicate submissions. No benchmark runs on the notebook host.")
import html, math
def heat_svg(values,n,title='Temperature field',width=430):
 if len(values)!=n*n or not all(math.isfinite(x) for x in values):raise ValueError('Invalid heat field')
 stride=max(1,math.ceil(n/64));size=320;cell=size/math.ceil(n/stride)
 parts=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="410" viewBox="0 0 {width} 410">',
 '<rect width="100%" height="100%" fill="white"/>',f'<text x="20" y="25" font-family="sans-serif" font-size="18">{html.escape(title)}</text>']
 for row,y in enumerate(range(0,n,stride)):
  for col,x in enumerate(range(0,n,stride)):
   v=max(0,min(1,values[y*n+x]/100));color=f'rgb({int(255*v)},{int(90*v)},{int(150*(1-v))})'
   parts.append(f'<rect x="{20+col*cell:.2f}" y="{45+row*cell:.2f}" width="{cell+.1:.2f}" height="{cell+.1:.2f}" fill="{color}"/>')
 parts += ['<text x="20" y="390" font-family="sans-serif" font-size="13">0 °C (blue) to 100 °C (red). Top edge is y=0.</text>','</svg>']
 return ''.join(parts)


## Run the complete heat experiment

The following cells create the files, calculate a serial reference, compare compiler settings, run OpenMP and MPI, check the hybrid layout and library versions, and collect GPU, ParaView and profiling results. The settings above control the problem size and optional stages.

The two CPU/GPU switches are enabled by default. Every result is displayed after its checks; a skipped or unfinished stage remains labelled as such.

In [ ]:
# Keep one recorded set of inputs for every implementation.
# This section also works independently if run from here in order.
from pathlib import Path
import os, sys, json, hashlib, subprocess, importlib.util
from IPython.display import display, HTML
GUIDE_SETTINGS = dict(n=globals().get("EXPERIMENT_N",128),
    steps=globals().get("EXPERIMENT_STEPS",150), repeats=globals().get("EXPERIMENT_REPEATS",2),
    wait_seconds=globals().get("MAX_WAIT_SECONDS",1800), poll_seconds=globals().get("POLL_SECONDS",5),
    launcher=globals().get("MPI_LAUNCHER","auto"),
    paraview_module=globals().get("PARAVIEW_MODULE",""))
assert 8 <= GUIDE_SETTINGS['n'] <= 2048
assert 1 <= GUIDE_SETTINGS['steps'] <= 10000
assert 1 <= GUIDE_SETTINGS['repeats'] <= 10
print("Experiment settings", GUIDE_SETTINGS)

### Create the source and scripts

Inspect the printed folder if you want to compare these files with your SSH work. The source and settings determine the run folder, so earlier results stay associated with the code that produced them.

In [ ]:
GUIDE_FILES = {'scripts/automate.py': '"""Submit frozen Slurm jobs once, wait, validate and keep a persistent run report."""\nfrom pathlib import Path\nimport os,json,subprocess,time,shutil,csv,statistics,html\n\nTERMINAL_BAD=(\'FAILED\',\'CANCELLED\',\'TIMEOUT\',\'OUT_OF_MEMORY\',\'NODE_FAIL\',\'BOOT_FAIL\',\'PREEMPTED\',\'REVOKED\',\'DEADLINE\')\nACTIVE={\'PENDING\',\'CONFIGURING\',\'RUNNING\',\'COMPLETING\',\'SUSPENDED\',\'REQUEUED\',\'RESIZING\',\'STAGE_OUT\'}\n\ndef command(args,cwd=None,env=None,timeout=25):\n return subprocess.run(args,cwd=cwd,env=env,capture_output=True,text=True,timeout=timeout)\ndef save(path,data):\n temp=path.with_suffix(\'.tmp\');temp.write_text(json.dumps(data,indent=2));temp.replace(path)\ndef state(job):\n if shutil.which(\'squeue\'):\n  q=command([\'squeue\',\'-h\',\'-j\',job,\'-o\',\'%T|%R\'])\n  if q.returncode==0 and q.stdout.strip():\n   a,_,reason=q.stdout.strip().splitlines()[0].partition(\'|\');return a.strip(),reason.strip(),None\n if shutil.which(\'sacct\'):\n  q=command([\'sacct\',\'-n\',\'-P\',\'-X\',\'-j\',job,\'-o\',\'State,ExitCode\'])\n  if q.returncode==0 and q.stdout.strip():\n   parts=q.stdout.strip().splitlines()[0].split(\'|\');return parts[0].strip().split()[0].rstrip(\'+\'),\'accounting\',parts[1].strip() if len(parts)>1 else None\n return \'UNKNOWN\',\'Queue/accounting unavailable\',None\n\ndef execute(root,device,settings,emit=print):\n root=Path(root);marker=root/f\'automation-{device}.json\'\n if not shutil.which(\'sbatch\'):return {\'device\':device,\'status\':\'NOT RUN\',\'reason\':\'sbatch unavailable; open on SciTech JupyterHub\'}\n # Reuse a saved submission rather than creating another queued job.\n if marker.exists():\n  record=json.loads(marker.read_text())\n  if not record.get(\'job_id\'):\n   return dict(record,status=\'UNKNOWN\',reason=\'Submission response uncertain. Inspect the saved folder and Slurm before retrying; no duplicate was sent.\')\n  emit(f"Reusing {device} job {record[\'job_id\']}")\n else:\n  # A separate immutable directory for each device/settings root, saved before submission.\n  directory=root/\'runs\'/device;directory.mkdir(parents=True,exist_ok=True)\n  for name in (\'src\',\'scripts\'):shutil.copytree(root/name,directory/name,dirs_exist_ok=False,ignore=shutil.ignore_patterns(\'__pycache__\',\'*.pyc\'))\n  record={\'device\':device,\'run_dir\':str(directory),\'status\':\'SUBMITTING\',\'job_id\':None}\n  save(marker,record)\n  # Slurm may reject conflicting inherited memory-request variables.\n  env=os.environ.copy()\n  for key in (\'SLURM_MEM_PER_CPU\',\'SLURM_MEM_PER_GPU\',\'SLURM_MEM_PER_NODE\'):env.pop(key,None)\n  env.update(HEAT_INPUT=settings.get(\'heat_input\',\'\'),PARAVIEW_MODULE=settings.get(\'paraview_module\',\'\'),N=str(settings[\'n\']),STEPS=str(settings[\'steps\']),REPEATS=str(settings[\'repeats\']),STAGE=\'all\',MPI_LAUNCHER=settings.get(\'launcher\',\'auto\'))\n  try:\n   q=command([\'sbatch\',\'--parsable\',f\'scripts/{device}.sbatch\'],cwd=directory,env=env,timeout=35)\n  except (OSError,subprocess.TimeoutExpired) as exc:\n   record.update(status=\'UNKNOWN\',reason=f\'Submission response uncertain: {exc}\');save(marker,record);return record\n  if q.returncode:\n   record.update(status=\'FAIL\',reason=q.stderr.strip() or q.stdout.strip());save(marker,record);return record\n  job=q.stdout.strip().split(\';\')[0]\n  if not job.isdigit():\n   record.update(status=\'UNKNOWN\',reason=\'Unexpected sbatch response: \'+q.stdout);save(marker,record);return record\n  record.update(job_id=job,status=\'SUBMITTED\');save(marker,record)\n  (directory/\'job_id.txt\').write_text(job+\'\\n\')\n  emit(f\'Submitted {device} job {job}; folder {directory}\')\n directory=Path(record[\'run_dir\']);job=record[\'job_id\'];log_path=directory/f\'{device}-{job}.out\'\n deadline=time.monotonic()+settings.get(\'wait_seconds\',1800);offset=0;last=None;last_notice=0;completed_since=None\n final_marker={\'cpu\':\'CPU_GUIDED_LAB_DONE\',\'gpu\':\'GPU_GUIDED_LAB_DONE\',\'profile\':\'PROFILE_DONE\',\'paraview\':\'PARAVIEW_DONE\'}[device]\n while True:\n  try:\n   st,reason,exit_code=state(job)\n  except (OSError,subprocess.TimeoutExpired) as exc:st,reason,exit_code=\'UNKNOWN\',str(exc),None\n  log=log_path.read_text(errors=\'replace\') if log_path.exists() else \'\'\n  # Stream only newly written log text while the job runs.\n  if len(log)>offset:\n   emit(log[offset:].rstrip());offset=len(log)\n  if st!=last or time.monotonic()-last_notice>=30:\n   emit(f\'{device} job {job}: {st} {reason}\');last=st;last_notice=time.monotonic()\n  # A completion-looking log must never overrule failed scheduler accounting.\n  if st.startswith(TERMINAL_BAD) or (exit_code and exit_code!=\'0:0\'):\n   record.update(status=\'FAIL\',reason=f\'Slurm {st}, exit={exit_code}\');break\n  if device==\'paraview\' and \'SKIPPED_PARAVIEW:\' in log and st not in ACTIVE:\n   record.update(status=\'SKIPPED\',reason=next(line for line in log.splitlines() if \'SKIPPED_PARAVIEW:\' in line));break\n  if device==\'profile\' and \'SKIPPED_PROFILE:\' in log and st not in ACTIVE:\n   record.update(status=\'SKIPPED\',reason=next(line for line in log.splitlines() if \'SKIPPED_PROFILE:\' in line));break\n  if final_marker in log and st not in ACTIVE:\n   if device==\'paraview\':\n    evidence=directory/\'results\'/f\'paraview-{job}\'\n    check=evidence/\'validation.json\'\n    if not check.exists():record.update(status=\'FAIL\',reason=\'Missing ParaView validation\')\n    else:\n     data=json.loads(check.read_text());png=evidence/\'paraview-temperature.png\'\n     valid=data.get(\'status\')==\'PASS\' and data.get(\'points\')==settings[\'n\']**2 and png.exists() and png.read_bytes()[:8]==b\'\\x89PNG\\r\\n\\x1a\\n\'\n     if valid:record.update(status=\'PASS\',paraview_dir=str(evidence),screenshot=str(png),accounting=st)\n     else:record.update(status=\'FAIL\',reason=\'ParaView field or screenshot check failed\')\n   elif device==\'profile\':\n    profiles=list((directory/\'results\').glob(\'profile-*/gprof.txt\'))\n    if not profiles:record.update(status=\'FAIL\',reason=\'Profile marker present but gprof.txt missing\')\n    else:record.update(status=\'PASS\',profile=str(profiles[0]),accounting=st)\n   else:\n    evidence=directory/\'results\'/f\'{device}-{job}\'\n    q=command([\'python3\',\'scripts/summarize.py\',str(evidence)],cwd=directory,timeout=60)\n    if q.returncode:record.update(status=\'FAIL\',reason=q.stderr or q.stdout)\n    else:\n     rows=list(csv.DictReader((evidence/\'measurements.csv\').open()))\n     # Require every expected mode, input and repetition before calling a run PASS.\n     expected={\'naive\',\'serial_O0\',\'serial\',\'openmp\',\'mpi\',\'hybrid\',\'library_v1\',\'library_v2\',\'library_fixed\'} if device==\'cpu\' else {\'serial\',\'cuda\'}\n     present={r[\'mode\'] for r in rows}\n     count_ok=all(len([r for r in rows if r[\'mode\']==m])==settings[\'repeats\'] for m in present)\n     inputs_ok=all(int(r[\'n\'])==settings[\'n\'] and int(r[\'steps\'])==settings[\'steps\'] for r in rows)\n     # Incomplete experiments never become PASS merely because some fields exist.\n     if not expected.issubset(present) or not count_ok or not inputs_ok:\n      record.update(status=\'FAIL\',reason=\'Missing modes, repeats or matching inputs\')\n     else:\n      from visualize import write_plots\n      plots=write_plots(evidence)\n      from export_vtk import export\n      export(evidence/\'reference-r0.bin\',settings[\'n\'],evidence/\'temperature.vtk\')\n      record.update(status=\'PASS\',evidence=str(evidence),plots=[str(p) for p in plots],accounting=st,\n                    skipped=[\'OpenACC compiler unavailable\'] if device==\'gpu\' and \'openacc\' not in present else [])\n      emit(q.stdout.rstrip())\n   if st==\'UNKNOWN\':emit(\'Numerical evidence checked; scheduler accounting unavailable. See saved log.\')\n   break\n  if st==\'COMPLETED\':\n   # Allow a brief shared-filesystem visibility delay before declaring the missing marker a failure.\n   if completed_since is None:completed_since=time.monotonic()\n   if time.monotonic()-completed_since>5:record.update(status=\'FAIL\',reason=\'Job completed without its completion marker\');break\n  if time.monotonic()>=deadline:\n   record.update(status=\'WAIT LIMIT\',reason=f\'{st}: notebook stopped waiting; job {job} was not resubmitted or cancelled\');break\n  time.sleep(settings.get(\'poll_seconds\',5))\n save(marker,record);return record\n\ndef show_result(record,display,HTML):\n display(HTML(f\'<h3>{html.escape(record["device"])} - {html.escape(record["status"])}</h3>\'))\n if record.get(\'reason\'):print(record[\'reason\'])\n if record.get(\'job_id\') and record.get(\'run_dir\'):\n  environment=Path(record[\'run_dir\'])/\'results\'/f"{record[\'device\']}-{record[\'job_id\']}"/\'environment.txt\'\n  if environment.exists():\n   display(HTML(\'<details><summary>Modules, compiler and allocation</summary><pre>\'+html.escape(environment.read_text(errors=\'replace\')[:6000])+\'</pre></details>\'))\n if record.get(\'paraview_dir\'):\n  from IPython.display import Image\n  display(Image(filename=record[\'screenshot\']))\n  print(\'ParaView files:\',record[\'paraview_dir\'])\n if record.get(\'profile\'):\n  print(Path(record[\'profile\']).read_text()[:9000])\n if record.get(\'evidence\'):\n  print(\'VTK field:\',Path(record[\'evidence\'])/\'temperature.vtk\')\n  rows=list(csv.DictReader((Path(record[\'evidence\'])/\'measurements.csv\').open()))\n  cells=[\'<table><tr><th>Mode</th><th>Layout</th><th>Repeats</th><th>Median loop (s)</th><th>Range (s)</th><th>Check</th></tr>\']\n  for mode in dict.fromkeys(r[\'mode\'] for r in rows):\n   subset=[r for r in rows if r[\'mode\']==mode];r=subset[0];values=[float(v[\'loop_s\']) for v in subset]\n   cells.append(\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(x))+\'</td>\' for x in [mode,f\'{r["ranks"]} × {r["threads"]}\',len(values),f\'{statistics.median(values):.6g}\',f\'{min(values):.6g}-{max(values):.6g}\',r[\'scientific_status\']])+\'</tr>\')\n  display(HTML(\'\'.join(cells)+\'</table>\'))\n for path in record.get(\'plots\',[]):display(HTML(Path(path).read_text()))\n for item in record.get(\'skipped\',[]):print(\'SKIPPED:\',item)\n\n\ndef show_final(report,display,HTML):\n """Display the actual ParaView image last, with an explicit GPU completion status."""\n records={r[\'device\']:r for r in report}\n gpu=records.get(\'gpu\',{})\n if gpu.get(\'status\')==\'PASS\':\n  rows=list(csv.DictReader((Path(gpu[\'evidence\'])/\'measurements.csv\').open()))\n  modes={r[\'mode\'] for r in rows}\n  print(\'CUDA heat solver: PASS; the GPU field matched its serial reference.\')\n  print(\'OpenACC heat solver: \'+(\'PASS; field validated.\' if \'openacc\' in modes else \'SKIPPED; compiler unavailable.\'))\n else:\n  print(\'GPU experiment:\',gpu.get(\'status\',\'NOT RUN\'))\n  print(gpu.get(\'reason\',\'No completed GPU validation is available.\'))\n pv=records.get(\'paraview\',{})\n display(HTML(\'<h3>ParaView result</h3>\'))\n image_path=Path(pv[\'screenshot\']) if pv.get(\'screenshot\') else None\n if pv.get(\'status\')==\'PASS\' and image_path and image_path.is_file():\n  from IPython.display import Image\n  print(\'Rendered by ParaView job\',pv.get(\'job_id\',\'\'),\'; input field and point count validated.\')\n  display(Image(filename=str(image_path)))\n  print(\'Image:\',image_path)\n  print(\'VTI, centre-line CSV and saved state:\',pv.get(\'paraview_dir\',\'\'))\n else:\n  print(\'ParaView:\',pv.get(\'status\',\'NOT RUN\'))\n  print(pv.get(\'reason\',\'No verified ParaView screenshot is available for this run.\'))\n  cpu=records.get(\'cpu\',{})\n  if cpu.get(\'status\')==\'PASS\' and cpu.get(\'evidence\'):\n   print(\'You can open this saved field in desktop ParaView:\',Path(cpu[\'evidence\'])/\'temperature.vtk\')\n', 'scripts/build.sh': '#!/bin/bash\nset -euo pipefail\ncd "$(dirname "$0")/.."\nmkdir -p bin lib/v1 lib/v2 lib/fixed\nstage=${1:-cpu}\n# CC can select the C compiler; keep it consistent with the MPI stack.\ncc=${CC:-gcc}\n# Build the same mathematics with different copy, optimisation and threading choices.\nif [[ "$stage" == serial || "$stage" == cpu ]]; then\n "$cc" -std=c11 -O0 -g -Wall -Wextra src/heat_naive.c -o bin/heat_naive\n "$cc" -std=c11 -O0 -g -Wall -Wextra src/heat_cpu.c -o bin/heat_serial_O0\n "$cc" -std=c11 -O2 -g -Wall -Wextra src/heat_cpu.c -o bin/heat_serial\n "$cc" -std=c11 -O2 -g -Wall -Wextra -fopenmp src/heat_cpu.c -o bin/heat_omp\nfi\n# mpicc supplies MPI headers/libraries; -fopenmp adds local threads.\nif [[ "$stage" == mpi || "$stage" == cpu ]]; then\n command -v mpicc\n mpicc -std=c11 -O2 -g -Wall -Wextra src/heat_mpi.c -o bin/heat_mpi\n mpicc -std=c11 -O2 -g -Wall -Wextra -fopenmp src/heat_mpi.c -o bin/heat_hybrid\nfi\n# Build interchangeable shared objects, then link one unchanged client executable.\nif [[ "$stage" == libraries || "$stage" == cpu ]]; then\n for pair in \'v1 temperature_v1.c\' \'v2 temperature_v2.c\' \'fixed temperature_v2_fixed.c\'; do\n  read -r version source <<< "$pair"\n  "$cc" -O2 -fPIC -shared "src/$source" -o "lib/$version/libtemperature.so"\n done\n "$cc" -std=c11 -O2 -g -DHOT_FROM_LIBRARY src/heat_cpu.c -Llib/v1 -ltemperature -o bin/heat_library\nfi\n# CUDA and OpenACC use different compilers but solve the same heat problem.\nif [[ "$stage" == gpu ]]; then\n command -v nvcc\n nvcc -O2 -lineinfo src/heat_cuda.cu -o bin/heat_cuda\n if command -v nvc >/dev/null; then\n  nvc -O2 -acc=gpu -Minfo=accel src/heat_acc.c -o bin/heat_acc 2>bin/openacc-build.txt\n else\n  echo \'SKIPPED OpenACC: nvc is not available\' >bin/openacc-build.txt\n fi\n "$cc" -std=c11 -O2 src/heat_cpu.c -o bin/heat_serial\nfi\ncase "$stage" in serial|mpi|libraries|cpu|gpu) ;; *) echo "Unknown build stage: $stage" >&2; exit 2;; esac\n', 'scripts/cpu.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s1-ssh\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=2\n#SBATCH --cpus-per-task=2\n#SBATCH --mem=2G\n#SBATCH --time=00:06:00\n#SBATCH --output=cpu-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR"\nstage=${STAGE:-all};export LAB_DEVICE=cpu LAB_NEEDS_MPI=0\nif [[ "$stage" == mpi || "$stage" == hybrid || "$stage" == all ]]; then LAB_NEEDS_MPI=1; fi\n# Match the compiler/runtime environment before building or running binaries.\nsource scripts/site.sh\nn=${N:-512};steps=${STEPS:-500};repeats=${REPEATS:-3}\n# One evidence directory per job keeps logs and measurements associated.\nout="results/cpu-$SLURM_JOB_ID";mkdir -p "$out"\n{\n date -u;hostname;pwd;echo "JOB=$SLURM_JOB_ID STAGE=$stage N=$n STEPS=$steps REPEATS=$repeats"\n echo "MPI_LAUNCHER=${MPI_LAUNCHER:-srun} SRUN_MPI_TYPE=${SRUN_MPI_TYPE:-site-default}"\n scontrol show job "$SLURM_JOB_ID";gcc --version\n if [[ "$LAB_NEEDS_MPI" == 1 ]]; then mpicc --showme 2>/dev/null || mpicc -show; fi\n if type module >/dev/null 2>&1; then module list; fi\n env | sort | sed -n \'/^OMP_/p;/^SLURM_/p\'\n sha256sum src/* scripts/*.sh scripts/*.py scripts/*.sbatch\n} >"$out/environment.txt" 2>&1\nbash scripts/build.sh serial\nif [[ "$LAB_NEEDS_MPI" == 1 ]]; then bash scripts/build.sh mpi; fi\nif [[ "$stage" == libraries || "$stage" == all ]]; then bash scripts/build.sh libraries; fi\nldd bin/heat_omp >"$out/openmp-libraries.txt"\nif [[ "$LAB_NEEDS_MPI" == 1 ]];then ldd bin/heat_mpi >"$out/mpi-libraries.txt";fi\nsha256sum bin/* >"$out/binary-hashes.txt"\nif [[ "$stage" == libraries || "$stage" == all ]]; then\n sha256sum lib/*/*.so >"$out/library-hashes.txt"\nfi\nif [[ "$LAB_NEEDS_MPI" == 1 && ${MPI_LAUNCHER:-auto} == auto ]]; then\n echo \'STEP MPI launcher preflight\'\n command -v timeout\n export LAB_EXPECT_RANKS=2\n bin/heat_serial 11 2 "$out/launcher-reference.bin" >"$out/launcher-reference.log"\n found=0\n for candidate in srun mpirun; do\n  if [[ "$candidate" == srun ]]; then\n   launch=(srun --ntasks=2 --cpus-per-task=1 --cpu-bind=cores)\n   if [[ -n ${SRUN_MPI_TYPE:-} ]]; then launch+=("--mpi=$SRUN_MPI_TYPE"); fi\n  else\n   command -v mpirun >/dev/null || continue\n   launch=(mpirun -np 2 --map-by slot:PE=1 --bind-to core)\n  fi\n  if timeout 25s "${launch[@]}" bin/heat_mpi 11 2 "$out/launcher-$candidate.bin" >"$out/launcher-$candidate.log" 2>&1 &&\n    python3 scripts/validate.py "$out/launcher-reference.bin" "$out/launcher-$candidate.bin" 11 >>"$out/launcher-$candidate.log" 2>&1; then\n   export MPI_LAUNCHER=$candidate;found=1;echo "CHECKPOINT PASS MPI launcher=$candidate";break\n  fi\n  echo "Launcher $candidate did not pass; log: $out/launcher-$candidate.log"\n done\n [[ "$found" == 1 ]] || { echo \'FAIL no validated MPI launcher\';exit 2; }\n echo "VALIDATED_MPI_LAUNCHER=$MPI_LAUNCHER" >>"$out/environment.txt"\nfi\n\n# All stages make an independent serial reference. Repeat 0 warms up; do not plot it.\nbash scripts/measure.sh reference 1 1 bin/heat_serial "$n" "$steps" "$out" 0\nmodes=()\ncase "$stage" in\n serial) modes=(naive serial_O0 serial);;\n openmp) modes=(serial openmp);;\n mpi) modes=(serial mpi);;\n hybrid) modes=(serial mpi openmp hybrid);;\n libraries) modes=(library_v1 library_v2 library_fixed);;\n all) modes=(naive serial_O0 serial openmp mpi hybrid library_v1 library_v2 library_fixed);;\n *) echo \'STAGE must be serial/openmp/mpi/hybrid/libraries/all\' >&2;exit 2;;\nesac\n# Run configurations sequentially so they do not compete for the same cores.\nfor mode in "${modes[@]}"; do\n echo "STEP $mode: compile complete; execute and validate"\n ranks=1;threads=1\n case "$mode" in\n naive) exe=bin/heat_naive;; serial_O0) exe=bin/heat_serial_O0;; serial) exe=bin/heat_serial;;\n mpi) exe=bin/heat_mpi;ranks=2;; openmp) exe=bin/heat_omp;threads=2;; hybrid) exe=bin/heat_hybrid;ranks=2;threads=2;;\n library_*) exe=bin/heat_library;version=${mode#library_};\n export LD_LIBRARY_PATH="$PWD/lib/$version${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"\n ldd bin/heat_library >"$out/$mode-ldd.txt";;\n esac\n # Repeat zero warms up; measured repetitions start at one.\n for ((r=0;r<=repeats;r++)); do\n  bash scripts/measure.sh "$mode" "$ranks" "$threads" "$exe" "$n" "$steps" "$out" "$r"\n  # Only the deliberate Kelvin error is expected to differ from serial.\n  if [[ "$mode" == library_v2 ]]; then\n   python3 scripts/validate.py "$out/reference-r0.bin" "$out/$mode-r$r.bin" "$n" --expect-different\n  else\n   python3 scripts/validate.py "$out/reference-r0.bin" "$out/$mode-r$r.bin" "$n"\n  fi\n done\n echo "CHECKPOINT PASS $mode"\ndone\npython3 scripts/summarize.py "$out"\necho "CPU_GUIDED_LAB_DONE $out"\n', 'scripts/export_vtk.py': '"""Export the native-double field to an ASCII VTK structured-points file."""\nfrom pathlib import Path\nimport array,math,sys\n\ndef export(source,n,target):\n # Native doubles are in row-major order: x changes fastest, as VTK expects.\n a=array.array(\'d\');a.frombytes(Path(source).read_bytes())\n if len(a)!=n*n or not all(math.isfinite(v) for v in a):raise ValueError(\'Invalid field\')\n # Add geometry and a scalar name while preserving each temperature value.\n with Path(target).open(\'w\') as f:\n  f.write(f\'# vtk DataFile Version 3.0\\nHeat diffusion, Celsius\\nASCII\\nDATASET STRUCTURED_POINTS\\nDIMENSIONS {n} {n} 1\\nORIGIN 0 0 0\\nSPACING 1 1 1\\nPOINT_DATA {n*n}\\nSCALARS Temperature double 1\\nLOOKUP_TABLE default\\n\')\n  for v in a:f.write(f\'{v:.17g}\\n\')\n return {\'n\':n,\'points\':len(a),\'minimum\':min(a),\'maximum\':max(a)}\nif __name__==\'__main__\':print(export(sys.argv[1],int(sys.argv[2]),sys.argv[3]))\n', 'scripts/gpu.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s1-heat-gpu\n#SBATCH --partition=gpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=1\n#SBATCH --gpus=1\n#SBATCH --mem=2G\n#SBATCH --time=00:06:00\n#SBATCH --output=gpu-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR";export LAB_DEVICE=gpu\n# The GPU compiler stack is selected inside the GPU allocation.\nsource scripts/site.sh\nn=${N:-512};steps=${STEPS:-500};repeats=${REPEATS:-3}\nout="results/gpu-$SLURM_JOB_ID";mkdir -p "$out"\n{\n date -u;hostname;scontrol show job "$SLURM_JOB_ID";nvidia-smi;nvcc --version\n if type module >/dev/null 2>&1; then module list; fi\n sha256sum src/* scripts/*.sh scripts/*.py scripts/*.sbatch\n} >"$out/environment.txt" 2>&1\n# Compile on the assigned node and keep compiler diagnostics with this run.\nbash scripts/build.sh gpu\nsha256sum bin/* >"$out/binary-hashes.txt"\nbash scripts/measure.sh reference 1 1 bin/heat_serial "$n" "$steps" "$out" 0\n# The serial reference uses this same node, avoiding a different CPU model.\nfor mode in serial cuda openacc; do\n echo "STEP $mode: execute and validate"\n case "$mode" in serial) exe=bin/heat_serial;;cuda) exe=bin/heat_cuda;;openacc)\n if [[ ! -x bin/heat_acc ]]; then echo \'SKIPPED OpenACC: compiler unavailable\';continue;fi\n exe=bin/heat_acc;;esac\n for ((r=0;r<=repeats;r++));do\n  bash scripts/measure.sh "$mode" 1 1 "$exe" "$n" "$steps" "$out" "$r"\n  python3 scripts/validate.py "$out/reference-r0.bin" "$out/$mode-r$r.bin" "$n"\n done\n echo "CHECKPOINT PASS $mode"\ndone\npython3 scripts/summarize.py "$out"\necho "GPU_GUIDED_LAB_DONE $out"\n', 'scripts/measure.sh': '#!/bin/bash\n# Called inside an allocation, once per program execution. Separate raw logs per trial.\nset -euo pipefail\n# Each call receives a mode, worker layout, executable, inputs and repeat number.\nmode=$1; ranks=$2; threads=$3; exe=$4; n=$5; steps=$6; out=$7; repeat=$8\nmkdir -p "$out"\n# Thread count does not allocate CPUs; the job/step resource requests do that.\nexport OMP_NUM_THREADS=$threads\nlog="$out/$mode-r$repeat.log"; field="$out/$mode-r$repeat.bin"\n# MPI modes need a launcher that creates a shared MPI communicator.\nif [[ "$mode" == mpi || "$mode" == hybrid ]]; then\n export LAB_EXPECT_RANKS=$ranks\n case ${MPI_LAUNCHER:-srun} in\n  srun) launch=(srun --ntasks="$ranks" --cpus-per-task="$threads" --cpu-bind=cores)\n        if [[ -n ${SRUN_MPI_TYPE:-} ]]; then launch+=("--mpi=$SRUN_MPI_TYPE"); fi;;\n  mpirun) launch=(mpirun -np "$ranks" --map-by "slot:PE=$threads" --bind-to core);;\n  *) echo \'MPI_LAUNCHER must be srun or mpirun\' >&2; exit 2;;\n esac\nelse\n launch=(srun --ntasks=1 --cpus-per-task="$threads" --cpu-bind=cores)\nfi\n# /usr/bin/time includes launcher/startup/compute/output; RESULT loop_s is solver-only.\nif [[ -x /usr/bin/time ]]; then\n /usr/bin/time -f \'WALL elapsed_s=%e user_s=%U sys_s=%S maxrss_kb=%M\' \\\n  "${launch[@]}" "$exe" "$n" "$steps" "$field" >"$log" 2>&1\nelse\n # Bash\'s timer still measures real execution; peak RSS is explicitly unavailable.\n TIMEFORMAT=\'WALL elapsed_s=%R user_s=%U sys_s=%S maxrss_kb=UNAVAILABLE\'\n { time "${launch[@]}" "$exe" "$n" "$steps" "$field"; } >"$log" 2>&1\nfi\ncat "$log"\n', 'scripts/paraview.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s1-paraview\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=1\n#SBATCH --mem=2G\n#SBATCH --time=00:06:00\n#SBATCH --output=paraview-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR"\nout="results/paraview-$SLURM_JOB_ID";mkdir -p "$out"\n# Make Environment Modules visible in this non-interactive shell.\nif ! type module >/dev/null 2>&1; then\n for init in /etc/profile.d/modules.sh /usr/share/Modules/init/bash;do\n  if [[ -r "$init" ]];then source "$init";fi\n  if type module >/dev/null 2>&1;then break;fi\n done\nfi\n{\n date -u;hostname;scontrol show job "$SLURM_JOB_ID"\n if type module >/dev/null 2>&1;then module list;fi\n} >"$out/environment.txt" 2>&1\nif type module >/dev/null 2>&1; then\n # Inspect the site catalogue instead of inventing a ParaView module version.\n module -t avail ParaView paraview >"$out/available-modules.txt" 2>&1 || true\n selected=${PARAVIEW_MODULE:-}\n if [[ -z "$selected" ]];then\n  selected=$(python3 - "$out/available-modules.txt" <<\'PY\'\nimport re,sys\ntext=open(sys.argv[1]).read()\nitems=re.findall(r\'(?<!\\S)([^\\s:]*[Pp][Aa][Rr][Aa][Vv][Ii][Ee][Ww][^\\s:()]*)\',text)\nitems=[s for s in items if \'/\' in s and not s.startswith(\'/\')]\nprint(sorted(set(items))[-1] if items else \'\')\nPY\n)\n fi\n if [[ -n "$selected" ]];then\n  echo "STEP module load $selected"\n  module load "$selected"\n  echo "PARAVIEW_MODULE=$selected" >>"$out/environment.txt"\n fi\n module list >>"$out/environment.txt" 2>&1\nfi\n# A missing application is SKIPPED; an attempted but failed render is FAIL.\nif ! command -v pvpython >/dev/null;then\n echo \'SKIPPED_PARAVIEW: pvpython unavailable; see available-modules.txt and environment.txt\'\n exit 0\nfi\ncommand -v pvpython | tee -a "$out/environment.txt"\npvpython --version >>"$out/environment.txt" 2>&1\n# Runs only after a validated CPU field exists. Both paths are on shared user storage.\npython3 scripts/export_vtk.py "$HEAT_INPUT" "$N" "$out/temperature.vtk"\npvpython --force-offscreen-rendering scripts/paraview_heat.py "$out/temperature.vtk" "$N" "$out"\necho "PARAVIEW_DONE $out"\n', 'scripts/paraview_heat.py': '"""Run with pvpython, not the notebook\'s Python interpreter."""\nfrom pathlib import Path\nimport json,sys\nfrom paraview.simple import (OpenDataFile,GetActiveViewOrCreate,Show,ColorBy,\n GetColorTransferFunction,Render,SaveScreenshot,SaveState,SaveData,PlotOverLine)\nsource=Path(sys.argv[1]).resolve();n=int(sys.argv[2]);out=Path(sys.argv[3]).resolve();out.mkdir(parents=True,exist_ok=True)\n# Read the grid into ParaView and confirm the dimensions and scalar range.\nfield=OpenDataFile(str(source));field.UpdatePipeline()\ninfo=field.GetDataInformation();points=info.GetNumberOfPoints()\n# PointData exposes ParaView\'s Python array-information wrapper, not the raw VTK API.\narray_info=field.PointData[\'Temperature\']\nif array_info is None:raise RuntimeError(\'Temperature array missing\')\nlow,high=array_info.GetRange(0)\nassert points==n*n,(points,n*n)\nassert abs(low)<1e-10 and abs(high-100)<1e-10,(low,high)\n# VTI stores the structured field; the CSV samples the centre column.\nSaveData(str(out/\'temperature.vti\'),proxy=field)\nline=PlotOverLine(Input=field)\ngeometry=line if hasattr(line,\'Point1\') else line.Source\ngeometry.Point1=[n//2,0,0];geometry.Point2=[n//2,n-1,0];geometry.Resolution=n-1\nline.UpdatePipeline();SaveData(str(out/\'centre-line.csv\'),proxy=line)\n# Build a view with a fixed 0-100 Celsius colour scale for honest comparisons.\nview=GetActiveViewOrCreate(\'RenderView\');display=Show(field,view)\nColorBy(display,(\'POINTS\',\'Temperature\'));GetColorTransferFunction(\'Temperature\').RescaleTransferFunction(0,100)\ndisplay.SetScalarBarVisibility(view,True)\nview.ViewSize=[1000,800];view.CameraParallelProjection=1\nview.CameraPosition=[(n-1)/2,(n-1)/2,2*n];view.CameraFocalPoint=[(n-1)/2,(n-1)/2,0]\nview.CameraViewUp=[0,-1,0];view.CameraParallelScale=n*.6\n# The batch launcher requests offscreen rendering; save the actual rendered image.\nRender(view);SaveScreenshot(str(out/\'paraview-temperature.png\'),view,ImageResolution=[1000,800])\n# A state file saves the pipeline/view settings, not the input field itself.\nSaveState(str(out/\'heat.pvsm\'))\nassert (out/\'paraview-temperature.png\').is_file()\n(out/\'validation.json\').write_text(json.dumps({\'points\':points,\'n\':n,\'minimum\':low,\'maximum\':high,\'status\':\'PASS\'},indent=2))\nprint(\'PARAVIEW_VALIDATION_PASS\',points,\'points, range\',low,high)\n', 'scripts/plot.py': '"""Generate temperature, profile and timing SVGs without extra Python packages."""\nfrom visualize import write_plots\nimport sys\nfor path in write_plots(sys.argv[1]):print(path)\n', 'scripts/profile.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s1-profile\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=1\n#SBATCH --mem=2G\n#SBATCH --time=00:06:00\n#SBATCH --output=profile-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR";export LAB_DEVICE=cpu LAB_NEEDS_MPI=0\nsource scripts/site.sh\nout="results/profile-$SLURM_JOB_ID";mkdir -p "$out"\ncommand -v gprof >/dev/null || { echo \'SKIPPED_PROFILE: gprof unavailable\'; exit 0; }\n# Use the naive version so its explicit copying is visible in a sampling profile.\ngcc -O0 -g -pg src/heat_naive.c -o "$out/heat_profile"\ncd "$out"\nif [[ -x /usr/bin/time ]]; then\n srun -n1 -c1 --cpu-bind=cores /usr/bin/time -v ./heat_profile 512 1000 field.bin >run.log 2>&1\nelse\n TIMEFORMAT=\'WALL elapsed_s=%R user_s=%U sys_s=%S maxrss_kb=UNAVAILABLE\'\n { time srun -n1 -c1 --cpu-bind=cores ./heat_profile 512 1000 field.bin; } >run.log 2>&1\nfi\ngprof ./heat_profile gmon.out >gprof.txt\n# Keep the binary/code/flags together. Profiling overhead disqualifies these timings from speedup claims.\necho "PROFILE_DONE $PWD"\n', 'scripts/site.sh': '# Source this file in SSH and batch shells. The login shell is important on SciTech.\n# These module names are documented in the course; check module avail on your login.\n# Initialise the module command if this batch shell did not inherit it.\nif ! type module >/dev/null 2>&1; then\n for init in /etc/profile.d/modules.sh /usr/share/Modules/init/bash; do\n  if [[ -r "$init" ]]; then source "$init"; fi\n  if type module >/dev/null 2>&1; then break; fi\n done\nfi\n# CPU and GPU jobs select their own supported software stack.\nif [[ ${LAB_DEVICE:-cpu} == gpu ]]; then\n if type module >/dev/null 2>&1; then module load nvhpc/25.7; fi\n command -v nvcc >/dev/null || { echo \'nvcc unavailable: check the site GPU module\' >&2; return 2; }\nelse\n if [[ ${LAB_NEEDS_MPI:-1} == 1 ]]; then\n  if type module >/dev/null 2>&1; then module load OpenMPI/4.1.5-GCC-12.3.0; fi\n  command -v mpicc >/dev/null || { echo \'mpicc unavailable: check the site MPI module\' >&2; return 2; }\n fi\nfi\n# Keep thread counts fixed and bind threads to nearby allocated cores.\nexport OMP_DYNAMIC=FALSE OMP_PLACES=cores OMP_PROC_BIND=close\n# No hidden fallback to another compiler stack after a failed module load.\n', 'scripts/submit.sh': '#!/bin/bash\n# Freeze all code and scripts into each submission. Queued jobs never read edited files.\nset -euo pipefail\ncd "$(dirname "$0")/.."\ndevice=${1:-cpu};stage=${2:-all}\ncase "$device" in cpu|gpu|paraview|profile) ;; *) echo \'Use cpu, gpu, paraview or profile\' >&2;exit 2;;esac\nstamp=$(date -u +%Y%m%dT%H%M%S)-$$\nrun="$PWD/runs/$device-$stage-$stamp"\nmkdir -p "$run";cp -r src scripts "$run/"\ncd "$run"\n# A notebook/interactive Slurm session can export incompatible memory variables.\nunset SLURM_MEM_PER_CPU SLURM_MEM_PER_GPU SLURM_MEM_PER_NODE\nexport STAGE=$stage\nexport MPI_LAUNCHER=${MPI_LAUNCHER:-auto}\njob=$(sbatch --parsable "scripts/$device.sbatch");job=${job%%;*}\n[[ "$job" =~ ^[0-9]+$ ]] || { echo "Unexpected sbatch output: $job" >&2;exit 2; }\nprintf \'%s\\n\' "$job" >job_id.txt\nprintf \'%s\\n\' "$PWD" >../../last_run_path.txt\nprintf \'Job %s\\nFrozen run: %s\\nCheck: squeue -j %s\\n\' "$job" "$PWD" "$job"\n', 'scripts/summarize.py': '"""Collect actual RESULT rows; repeat 0 is a warmup, not an observation."""\nfrom pathlib import Path\nimport csv,re,statistics,sys,subprocess\nfolder=Path(sys.argv[1]);rows=[]\nfor log in sorted(folder.glob(\'*-r*.log\')):\n match=re.search(r\'-r(\\d+)\\.log$\',log.name)\n if not match or int(match[1])==0:continue\n lines=[s for s in log.read_text().splitlines() if s.startswith(\'RESULT \')]\n if len(lines)!=1:raise SystemExit(f\'FAIL: expected exactly one RESULT in {log.name}; inspect MPI launcher\')\n values=dict(piece.split(\'=\',1) for piece in lines[0].split()[1:]);values[\'mode\']=log.name.rsplit(\'-r\',1)[0]\n values[\'repeat\']=int(match[1]);values[\'log\']=log.name;values[\'job_dir\']=str(folder.resolve())\n check=[sys.executable,str(Path(__file__).with_name(\'validate.py\')),str(folder/\'reference-r0.bin\'),str(log.with_suffix(\'.bin\')),values[\'n\']]\n if values[\'mode\']==\'library_v2\':check.append(\'--expect-different\')\n subprocess.run(check,check=True,capture_output=True,text=True)\n values[\'scientific_status\']=\'EXPECTED_MISMATCH\' if values[\'mode\']==\'library_v2\' else \'PASS\'\n rows.append(values)\nif not rows:raise SystemExit(\'NOT RUN: no measured repeats found\')\nwith (folder/\'measurements.csv\').open(\'w\',newline=\'\') as f:\n w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\nfor mode in sorted({r[\'mode\'] for r in rows}):\n times=[float(r[\'loop_s\']) for r in rows if r[\'mode\']==mode]\n print(f\'{mode:15} repeats={len(times)} median_loop_s={statistics.median(times):.9f} range=[{min(times):.9f}, {max(times):.9f}]\')\nprint(\'Saved\',folder/\'measurements.csv\')\n', 'scripts/validate.py': '"""Full-field numerical checks; no third-party packages required."""\nimport argparse, array, math\np=argparse.ArgumentParser();p.add_argument(\'reference\');p.add_argument(\'candidate\');p.add_argument(\'n\',type=int)\np.add_argument(\'--expect-different\',action=\'store_true\');args=p.parse_args();n=args.n\n\n# Refuse truncated fields and NaNs before doing a numerical comparison.\ndef read(path):\n a=array.array(\'d\');a.frombytes(open(path,\'rb\').read())\n if len(a)!=n*n:raise SystemExit(f\'FAIL {path}: expected {n*n} doubles, got {len(a)}\')\n if not all(math.isfinite(x) for x in a):raise SystemExit(\'FAIL: non-finite temperature\')\n return a\nref=read(args.reference);got=read(args.candidate)\n# A checksum can hide errors; compare corresponding cells everywhere.\nerr=max(abs(a-b) for a,b in zip(ref,got))\n# Check the physical constraints separately from agreement with reference.\nboundaries=(all(abs(got[x]-100)<1e-10 for x in range(n)) and\n all(abs(got[(n-1)*n+x])<1e-10 for x in range(n)) and\n all(abs(got[y*n])<1e-10 and abs(got[y*n+n-1])<1e-10 for y in range(1,n-1)))\nbounded=min(got)>=-1e-10 and max(got)<=100+1e-10\nmatches=err<=1e-10 and boundaries and bounded\nif args.expect_different:\n # The v2 Kelvin example is an intentionally incorrect Celsius field.\n if err<=1e-10:raise SystemExit(\'FAIL: expected the incompatible library to change the field\')\n print(f\'EXPECTED MISMATCH (not a correct heat solution): max_abs_error={err:.6g}, boundaries_ok={boundaries}\')\nelse:\n if not matches:raise SystemExit(f\'FAIL max_abs_error={err:.6g} boundaries_ok={boundaries} bounded={bounded}\')\n print(f\'PASS full field; max_abs_error={err:.6g}; boundaries and range correct\')\n', 'scripts/visualize.py': '"""Plots from saved doubles and CSV; standard library only, no Matplotlib required."""\nimport array,csv,html,math,statistics,sys\nfrom pathlib import Path\n\ndef heat_svg(values,n,title=\'Temperature field\',width=430):\n if len(values)!=n*n or not all(math.isfinite(x) for x in values):raise ValueError(\'Invalid heat field\')\n stride=max(1,math.ceil(n/64));size=320;cell=size/math.ceil(n/stride)\n parts=[f\'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="410" viewBox="0 0 {width} 410">\',\n \'<rect width="100%" height="100%" fill="white"/>\',f\'<text x="20" y="25" font-family="sans-serif" font-size="18">{html.escape(title)}</text>\']\n for row,y in enumerate(range(0,n,stride)):\n  for col,x in enumerate(range(0,n,stride)):\n   v=max(0,min(1,values[y*n+x]/100));color=f\'rgb({int(255*v)},{int(90*v)},{int(150*(1-v))})\'\n   parts.append(f\'<rect x="{20+col*cell:.2f}" y="{45+row*cell:.2f}" width="{cell+.1:.2f}" height="{cell+.1:.2f}" fill="{color}"/>\')\n parts += [\'<text x="20" y="390" font-family="sans-serif" font-size="13">0 °C (blue) to 100 °C (red). Top edge is y=0.</text>\',\'</svg>\']\n return \'\'.join(parts)\n\ndef write_plots(folder):\n folder=Path(folder);records=list(csv.DictReader((folder/\'measurements.csv\').open()))\n correct=[r for r in records if r[\'scientific_status\']==\'PASS\']\n if not correct:raise ValueError(\'No validated records\')\n n=int(correct[0][\'n\']);a=array.array(\'d\');a.frombytes((folder/\'reference-r0.bin\').read_bytes())\n (folder/\'heat.svg\').write_text(heat_svg(a,n,f\'Final temperature, N={n}\'))\n # Centre-column profile from the actual saved reference, sampled only for display.\n points=\' \'.join(f\'{40+a[y*n+n//2]*3.2:.2f},{40+300*y/(n-1):.2f}\' for y in range(n))\n svg=f\'<svg xmlns="http://www.w3.org/2000/svg" width="440" height="400"><rect width="100%" height="100%" fill="white"/><text x="30" y="24" font-family="sans-serif" font-size="18">Centre-column temperature</text><path d="M40 40 V340 H360" fill="none" stroke="grey"/><polyline points="{points}" fill="none" stroke="#124b78" stroke-width="2"/><text x="40" y="370" font-family="sans-serif">0 °C</text><text x="300" y="370" font-family="sans-serif">100 °C</text><text x="5" y="43" font-family="sans-serif">0</text><text x="5" y="340" font-family="sans-serif">{n-1}</text></svg>\'\n (folder/\'profile.svg\').write_text(svg)\n modes=list(dict.fromkeys(r[\'mode\'] for r in correct));height=75+38*len(modes)\n chart=[f\'<svg xmlns="http://www.w3.org/2000/svg" width="850" height="{height}"><rect width="100%" height="100%" fill="white"/><text x="20" y="24" font-family="sans-serif" font-size="18">Measured update-loop time: dots and median (seconds)</text>\']\n all_times=[float(r[\'loop_s\']) for r in correct];max_time=max(all_times) or 1\n for k,m in enumerate(modes):\n  vals=[float(r[\'loop_s\']) for r in correct if r[\'mode\']==m];median=statistics.median(vals);y=55+k*38\n  chart.append(f\'<text x="20" y="{y+4}" font-family="sans-serif">{html.escape(m)}</text><path d="M165 {y} H{165+480*median/max_time:.2f}" stroke="#124b78" stroke-width="5"/>\')\n  for v in vals:chart.append(f\'<circle cx="{165+480*v/max_time:.2f}" cy="{y}" r="3" fill="#7d8790"/>\')\n  chart.append(f\'<text x="670" y="{y+4}" font-family="sans-serif" font-size="12">{median:.6g} [{min(vals):.3g}, {max(vals):.3g}]</text>\')\n chart.append(\'</svg>\');(folder/\'timings.svg\').write_text(\'\'.join(chart))\n return [folder/\'heat.svg\',folder/\'profile.svg\',folder/\'timings.svg\']\n\nif __name__==\'__main__\':\n for p in write_plots(sys.argv[1]):print(p)\n', 'src/heat_acc.c': '/* OpenACC: fixed array identities, alternating read/write role; no pointer remapping. */\n#include "heat_common.h"\n#include <openacc.h>\nint main(int argc,char **argv){\n if(argc!=4){fprintf(stderr,"usage: heat_acc N STEPS field.bin\\n");return 2;}\n int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000);\n /* Refuse CPU fallback: this exercise must actually use an NVIDIA accelerator. */\n if(acc_get_num_devices(acc_device_nvidia)<1){fprintf(stderr,"No NVIDIA accelerator\\n");return 3;}\n acc_set_device_type(acc_device_nvidia);\n double start=clock_seconds();double *a=grid(n,100),*b=grid(n,100);size_t count=(size_t)n*n;\n double setup=0,loop=0,copy=0;\n /* Keep both arrays on the GPU while the entire time loop runs. */\n #pragma acc data copyin(a[0:count],b[0:count])\n {\n  setup=clock_seconds()-start;double t=clock_seconds();\n  for(int s=0;s<steps;s++){\n   /* Alternating array roles avoids changing pointer-to-device mappings. */\n   if((s%2)==0){\n    /* collapse(2) distributes both grid loops; present reuses mapped device data. */\n    #pragma acc parallel loop collapse(2) present(a[0:count],b[0:count])\n    for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++){\n     size_t k=(size_t)y*n+x;b[k]=0.25*(a[k-n]+a[k+n]+a[k-1]+a[k+1]);\n    }\n   }else{\n    #pragma acc parallel loop collapse(2) present(a[0:count],b[0:count])\n    for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++){\n     size_t k=(size_t)y*n+x;a[k]=0.25*(b[k-n]+b[k+n]+b[k-1]+b[k+1]);\n    }\n   }\n  }\n  /* Finish device work before recording the host-clock duration. */\n  #pragma acc wait\n  loop=clock_seconds()-t;t=clock_seconds();\n  /* Odd steps end in b, even steps in a. Transfer only the final array. */\n  if(steps%2){\n   #pragma acc update self(b[0:count])\n  }else{\n   #pragma acc update self(a[0:count])\n  }\n  copy=clock_seconds()-t;\n }\n /* The same binary output format lets serial validate the whole GPU field. */\n double *final=(steps%2)?b:a;double t=clock_seconds();save_grid(argv[3],final,n);\n result("openacc",n,steps,1,1,setup,loop,0,loop,copy+clock_seconds()-t,clock_seconds()-start,final);\n free(a);free(b);return 0;\n}\n', 'src/heat_common.h': '#ifndef HEAT_COMMON_H\n#define HEAT_COMMON_H\n#define _POSIX_C_SOURCE 200809L\n#include <stdio.h>\n#include <stdlib.h>\n#include <math.h>\n#include <time.h>\n#include <errno.h>\n#include <string.h>\n/* Refuse invalid arguments before they can produce an invalid array index. */\nstatic inline int integer(const char *s,int lo,int hi){\n char *end; errno=0; long v=strtol(s,&end,10);\n if(errno||*end||end==s||v<lo||v>hi){fprintf(stderr,"Invalid integer: %s\\n",s);exit(2);}return (int)v;\n}\n/* A monotonic clock measures elapsed time without changes to the calendar clock. */\nstatic inline double clock_seconds(void){\n struct timespec t;if(clock_gettime(CLOCK_MONOTONIC,&t)){perror("clock_gettime");exit(2);}\n return t.tv_sec+1e-9*t.tv_nsec;\n}\n/* calloc sets the cold plate to zero; only the hot top edge is changed. */\nstatic inline double *grid(int n,double hot){\n double *a=(double*)calloc((size_t)n*n,sizeof(double));\n if(!a){perror("calloc");exit(2);}for(int x=0;x<n;x++)a[x]=hot;return a;\n}\n/* Save all cells in row-major order so another program can validate the field. */\nstatic inline void save_grid(const char *path,const double *a,int n){\n FILE *f=fopen(path,"wb");if(!f){perror(path);exit(2);}\n if(fwrite(a,sizeof(double),(size_t)n*n,f)!=(size_t)n*n){fprintf(stderr,"Short grid write\\n");exit(2);}\n if(fclose(f)){perror("fclose");exit(2);}\n}\n/* Keep timing scopes separate: update loop, output and the broader program interval. */\nstatic inline void result(const char *mode,int n,int steps,int ranks,int threads,\n double setup,double loop,double halo,double update,double io,double total,const double *a){\n double sum=0;for(size_t k=0;k<(size_t)n*n;k++)sum+=a[k];\n printf("RESULT mode=%s n=%d steps=%d ranks=%d threads=%d setup_s=%.9f loop_s=%.9f halo_s=%.9f update_s=%.9f io_s=%.9f total_s=%.9f checksum=%.17g\\n",\n mode,n,steps,ranks,threads,setup,loop,halo,update,io,total,sum);\n}\n#endif\n', 'src/heat_cpu.c': '/* Same mathematics. Swap pointers; optionally compile OpenMP or a library boundary. */\n#include "heat_common.h"\n#ifdef _OPENMP\n#include <omp.h>\n#endif\n#ifdef HOT_FROM_LIBRARY\nextern double top_temperature(void);\n#endif\nint main(int argc,char **argv){\n if(argc!=4){fprintf(stderr,"usage: heat_cpu N STEPS field.bin\\n");return 2;}\n /* Keep the same grid and number of steps for every execution layout. */\n int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000),threads=1;\n const char *mode="serial";double hot=100;\n#ifdef HOT_FROM_LIBRARY\n /* The shared library must return Celsius, the units expected by this solver. */\n hot=top_temperature();mode="library";\n#endif\n#ifdef _OPENMP\n mode="openmp";\n /* Measure the actual team size, rather than assuming the requested count. */\n #pragma omp parallel\n {\n  #pragma omp single\n  threads=omp_get_num_threads();\n }\n#endif\n double start=clock_seconds();double *old=grid(n,hot),*next=grid(n,hot);\n double setup=clock_seconds()-start,t=clock_seconds();\n for(int s=0;s<steps;s++){\n#ifdef _OPENMP\n  /* Threads read shared old data and write disjoint rows in next.\n     The implicit end-of-loop barrier completes all writes before the swap. */\n  #pragma omp parallel for schedule(static)\n#endif\n  for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++){\n   size_t k=(size_t)y*n+x;\n   next[k]=0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);\n  }\n  /* Swap names instead of copying every temperature. Boundaries exist in both arrays. */\n  double *tmp=old;old=next;next=tmp;\n }\n /* I/O is timed separately so it cannot be mistaken for stencil computation. */\n double loop=clock_seconds()-t;t=clock_seconds();save_grid(argv[3],old,n);\n double io=clock_seconds()-t;\n result(mode,n,steps,1,threads,setup,loop,0,loop,io,clock_seconds()-start,old);\n free(old);free(next);return 0;\n}\n', 'src/heat_cuda.cu': '/* One GPU heat solver. Both arrays stay on the device between time steps. */\n#include "heat_common.h"\n#include <cuda_runtime.h>\n#define CUDA(call) do{cudaError_t e=(call);if(e!=cudaSuccess){fprintf(stderr,"CUDA: %s\\n",cudaGetErrorString(e));exit(3);}}while(0)\n__global__ void stencil(const double *old,double *next,int n){\n /* The block and thread indices give each GPU thread one grid coordinate. */\n int x=blockIdx.x*blockDim.x+threadIdx.x,y=blockIdx.y*blockDim.y+threadIdx.y;\n /* Skip fixed boundaries; each interior output cell has exactly one writer. */\n if(x>0&&x<n-1&&y>0&&y<n-1){size_t k=(size_t)y*n+x;next[k]=0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);}\n}\nint main(int argc,char **argv){\n if(argc!=4){fprintf(stderr,"usage: heat_cuda N STEPS field.bin\\n");return 2;}\n int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000);\n /* Host arrays hold the initial grid and receive the final result. */\n double start=clock_seconds();double *a=grid(n,100),*b=grid(n,100),*da,*db;\n /* Allocate separate device memory; CPU pointers do not identify GPU storage. */\n size_t bytes=(size_t)n*n*sizeof(double);CUDA(cudaMalloc((void**)&da,bytes));CUDA(cudaMalloc((void**)&db,bytes));\n /* Transfer once, then reuse device-resident data for all time steps. */\n CUDA(cudaMemcpy(da,a,bytes,cudaMemcpyHostToDevice));CUDA(cudaMemcpy(db,b,bytes,cudaMemcpyHostToDevice));\n /* CUDA events measure the asynchronous device loop in its own stream. */\n cudaEvent_t e0,e1;CUDA(cudaEventCreate(&e0));CUDA(cudaEventCreate(&e1));\n double setup=clock_seconds()-start;dim3 block(16,16),blocks((n+15)/16,(n+15)/16);\n CUDA(cudaEventRecord(e0));\n /* Default-stream ordering keeps each step behind the previous kernel. */\n for(int s=0;s<steps;s++){\n  /* Launch enough 16x16 blocks to cover the plate, including partial edge blocks. */\n  stencil<<<blocks,block>>>(da,db,n);CUDA(cudaGetLastError());double *tmp=da;da=db;db=tmp;\n }\n /* Synchronise before reading the duration or copying results back. */\n CUDA(cudaEventRecord(e1));CUDA(cudaEventSynchronize(e1));float ms;CUDA(cudaEventElapsedTime(&ms,e0,e1));\n /* Copy the completed field back once and save it for CPU/GPU comparison. */\n double t=clock_seconds();CUDA(cudaMemcpy(a,da,bytes,cudaMemcpyDeviceToHost));save_grid(argv[3],a,n);\n double io=clock_seconds()-t;\n CUDA(cudaEventDestroy(e0));CUDA(cudaEventDestroy(e1));CUDA(cudaFree(da));CUDA(cudaFree(db));\n result("cuda",n,steps,1,1,setup,ms/1000.0,0,ms/1000.0,io,clock_seconds()-start,a);\n free(a);free(b);return 0;\n}\n', 'src/heat_mpi.c': '/* Interior row decomposition; same stencil, optional OpenMP, MPI outside threads. */\n#include "heat_common.h"\n#include <mpi.h>\n#ifdef _OPENMP\n#include <omp.h>\n#endif\nint main(int argc,char **argv){\n int provided,rank,p;double total_start=clock_seconds();\n /* FUNNELED means only the original thread makes MPI calls. */\n MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&provided);\n MPI_Comm_rank(MPI_COMM_WORLD,&rank);MPI_Comm_size(MPI_COMM_WORLD,&p);\n /* Catch a launcher that accidentally starts independent one-rank programs. */\n if(getenv("LAB_EXPECT_RANKS") && p!=atoi(getenv("LAB_EXPECT_RANKS"))){\n  if(!rank)fprintf(stderr,"MPI launcher mismatch: expected %s ranks, got %d. Check site launcher.\\n",getenv("LAB_EXPECT_RANKS"),p);\n  MPI_Abort(MPI_COMM_WORLD,2);\n }\n if(provided<MPI_THREAD_FUNNELED)MPI_Abort(MPI_COMM_WORLD,2);\n if(argc!=4){if(!rank)fprintf(stderr,"usage: heat_mpi N STEPS field.bin\\n");MPI_Abort(MPI_COMM_WORLD,2);}\n int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000),interior=n-2;\n if(p>interior){if(!rank)fprintf(stderr,"Use at most N-2 ranks\\n");MPI_Abort(MPI_COMM_WORLD,2);}\n /* Divide N-2 interior rows; early ranks receive any leftover rows. */\n int rows=interior/p+(rank<interior%p),first=1+rank*(interior/p)+(rank<interior%p?rank:interior%p);\n /* Each process stores its owned rows and one halo row on either side. */\n size_t cells=(size_t)(rows+2)*n;\n double *old=(double*)calloc(cells,sizeof(double)),*next=(double*)calloc(cells,sizeof(double));\n if(!old||!next)MPI_Abort(MPI_COMM_WORLD,3);\n /* Rank 0 keeps the physical hot edge in its upper halo. */\n if(rank==0)for(int x=0;x<n;x++){old[x]=100;next[x]=100;}\n /* MPI_PROC_NULL marks a physical edge where there is no neighbour. */\n int up=rank?rank-1:MPI_PROC_NULL,down=rank==p-1?MPI_PROC_NULL:rank+1,threads=1;\n const char *mode="mpi";\n#ifdef _OPENMP\n mode="hybrid";\n #pragma omp parallel\n {\n  #pragma omp single\n  threads=omp_get_num_threads();\n }\n#endif\n char host[MPI_MAX_PROCESSOR_NAME];int host_len;MPI_Get_processor_name(host,&host_len);\n /* Print ownership so we can compare the predicted layout with the actual run. */\n printf("MAP rank=%d ranks=%d host=%s first_global_row=%d rows=%d threads=%d\\n",rank,p,host,first,rows,threads);\n /* Start the measured loop after every rank has finished initialisation. */\n MPI_Barrier(MPI_COMM_WORLD);double setup=clock_seconds()-total_start;\n double t0=MPI_Wtime(),halo=0,update=0;\n for(int s=0;s<steps;s++){\n  double h=MPI_Wtime();\n  /* Send our top owned row up; receive the lower neighbour into our bottom halo. */\n  MPI_Sendrecv(old+n,n,MPI_DOUBLE,up,10,old+(size_t)(rows+1)*n,n,MPI_DOUBLE,down,10,MPI_COMM_WORLD,MPI_STATUS_IGNORE);\n  /* Send our bottom owned row down; receive the upper neighbour into our top halo. */\n  MPI_Sendrecv(old+(size_t)rows*n,n,MPI_DOUBLE,down,11,old,n,MPI_DOUBLE,up,11,MPI_COMM_WORLD,MPI_STATUS_IGNORE);\n  halo+=MPI_Wtime()-h;double u=MPI_Wtime();\n#ifdef _OPENMP\n  /* OpenMP shares only this process\'s local rows. MPI stays outside this region. */\n  #pragma omp parallel for schedule(static)\n#endif\n  for(int y=1;y<=rows;y++)for(int x=1;x<n-1;x++){\n   size_t k=(size_t)y*n+x;next[k]=0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);\n  }\n  /* Both halo exchanges and the local update have finished before roles swap. */\n  double *tmp=old;old=next;next=tmp;update+=MPI_Wtime()-u;\n }\n /* Maximum rank duration excludes gather and output; phase maxima need not add up. */\n double local[3]={MPI_Wtime()-t0,halo,update},times[3];\n MPI_Reduce(local,times,3,MPI_DOUBLE,MPI_MAX,0,MPI_COMM_WORLD);\n double io_start=clock_seconds();double *global=NULL;int *counts=NULL,*offsets=NULL;\n if(rank==0){\n  global=grid(n,100);counts=(int*)malloc(p*sizeof(int));offsets=(int*)malloc(p*sizeof(int));\n  if(!counts||!offsets)MPI_Abort(MPI_COMM_WORLD,3);\n  for(int r=0;r<p;r++){\n   counts[r]=(interior/p+(r<interior%p))*n;\n   offsets[r]=(1+r*(interior/p)+(r<interior%p?r:interior%p))*n;\n  }\n }\n /* Counts and offsets allow ranks to own different numbers of rows.\n    Only root allocates the full global field, and gathering happens once. */\n MPI_Gatherv(old+n,rows*n,MPI_DOUBLE,global,counts,offsets,MPI_DOUBLE,0,MPI_COMM_WORLD);\n if(rank==0){\n  /* Validate this complete field against serial, not just its checksum. */\n  save_grid(argv[3],global,n);\n  result(mode,n,steps,p,threads,setup,times[0],times[1],times[2],clock_seconds()-io_start,clock_seconds()-total_start,global);\n }\n free(old);free(next);free(global);free(counts);free(offsets);MPI_Finalize();return 0;\n}\n', 'src/heat_naive.c': '/* First program: one CPU, explicit copy after every time step. */\n#include "heat_common.h"\nint main(int argc,char **argv){\n if(argc!=4){fprintf(stderr,"usage: heat_naive N STEPS field.bin\\n");return 2;}\n /* N includes boundary cells; STEPS is the number of time updates. */\n int n=integer(argv[1],8,2048),steps=integer(argv[2],1,10000);\n /* Two arrays keep all reads at the old time level while we write the next one. */\n double start=clock_seconds();double *old=grid(n,100),*next=grid(n,100);\n double setup=clock_seconds()-start, t=clock_seconds();\n /* Boundary rows and columns are excluded from the update loops. */\n for(int s=0;s<steps;s++){\n  for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++){\n   /* Flatten (row, column) to one index; neighbours are +/-n and +/-1. */\n   size_t k=(size_t)y*n+x;\n   next[k]=0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);\n  }\n  /* This extra copy is correct but moves the full interior on every step. */\n  for(int y=1;y<n-1;y++)for(int x=1;x<n-1;x++)old[(size_t)y*n+x]=next[(size_t)y*n+x];\n }\n /* Stop the loop timer before writing the field to storage. */\n double loop=clock_seconds()-t;t=clock_seconds();save_grid(argv[3],old,n);\n double io=clock_seconds()-t;\n result("naive",n,steps,1,1,setup,loop,0,loop,io,clock_seconds()-start,old);\n free(old);free(next);return 0;\n}\n', 'src/temperature_v1.c': '/* Contract: Celsius. */\n/* The caller expects Celsius, so this returns the agreed physical value. */\ndouble top_temperature(void){return 100.0;}\n', 'src/temperature_v2.c': '/* Deliberate incompatible contract: Kelvin. Same symbol, different units. */\n/* This is intentionally wrong for the caller: a loadable symbol can still use wrong units. */\ndouble top_temperature(void){return 373.15;}\n', 'src/temperature_v2_fixed.c': '/* Adapter restores the Celsius contract expected by this solver. */\n/* Convert units at the library boundary so the existing solver remains correct. */\ndouble top_temperature(void){return 373.15-273.15;}\n'}
GUIDE_KEY = hashlib.sha256(json.dumps({'files':GUIDE_FILES,'settings':GUIDE_SETTINGS},sort_keys=True).encode()).hexdigest()[:16]
if globals().get('NEW_EXPERIMENT',False):
    import uuid
    GUIDE_KEY += '-' + uuid.uuid4().hex[:8]
GUIDE_ROOT = Path.home() / 'hpc-labs' / 'm5s1-automatic' / GUIDE_KEY
for name,text in GUIDE_FILES.items():
    path=GUIDE_ROOT/name
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.read_text()!=text:raise RuntimeError('Saved source changed: '+str(path))
    if not path.exists():path.write_text(text)
sys.path.insert(0,str(GUIDE_ROOT/'scripts'))
# Reload helpers from this exact source bundle when the notebook is rerun.
for name in ('visualize','automate'):sys.modules.pop(name,None)
import automate
print('CHECKPOINT PASS source files saved in',GUIDE_ROOT)
GUIDE_REPORT = []

### CPU solvers and libraries

We first calculate a serial reference field. The naive and pointer-swap versions show the cost of copying data, and the compiler settings show what optimisation changes. OpenMP then shares the local work between two threads, MPI splits rows between two processes, and hybrid combines two ranks with two threads each.

After every run, we check the complete temperature field. Only then do we compare elapsed times. The library exercise deliberately changes units and then fixes them, showing why the software environment belongs in the experiment record. Watch the STEP and CHECKPOINT lines as the job proceeds.

In [ ]:
# Submit or resume the saved CPU job, then display only checked results.
if globals().get('RUN_COMPLETE_EXPERIMENT',True) and globals().get('RUN_COMPLETE_EXPERIMENT',True):
    GUIDE_CPU = automate.execute(GUIDE_ROOT,'cpu',GUIDE_SETTINGS,emit=lambda text: print(text,flush=True))
else:
    GUIDE_CPU = {'device':'cpu','status':'SKIPPED','reason':'Disabled in settings'}
GUIDE_REPORT.append(GUIDE_CPU)
automate.show_result(GUIDE_CPU,display,HTML)

### ParaView post-processing

The simulation has produced numbers on a grid. We now give that grid to ParaView to see the temperature distribution and sample a line through it. The job loads the ParaView software module and uses its own Python interpreter, rather than the notebook's interpreter.

We save the image, field, line profile and viewing state together. The preview appears here after a successful job and again at the end of the notebook, beside the GPU completion status. A missing ParaView installation is reported as SKIPPED; the separate SVG preview is not labelled as a ParaView result.

In [ ]:
# ParaView reads the validated serial field so rendering cannot hide a solver mismatch.
if globals().get('RUN_COMPLETE_EXPERIMENT',True) and globals().get('RUN_PARAVIEW',True) and GUIDE_CPU.get('status')=='PASS':
    para_settings=dict(GUIDE_SETTINGS,heat_input=str(Path(GUIDE_CPU['evidence'])/'reference-r0.bin'))
    GUIDE_PARAVIEW=automate.execute(GUIDE_ROOT,'paraview',para_settings,emit=lambda text:print(text,flush=True))
else:
    GUIDE_PARAVIEW={'device':'paraview','status':'SKIPPED','reason':'Disabled or no validated CPU field'}
GUIDE_REPORT.append(GUIDE_PARAVIEW)
automate.show_result(GUIDE_PARAVIEW,display,HTML)

### CUDA and OpenACC heat solvers

We now run the same heat update on an allocated GPU. CUDA makes memory copies and kernel launches explicit; OpenACC describes the parallel loops with directives. Both keep the arrays on the device between steps and return the final field for checking.

The CPU reference runs on the GPU node, so the comparison uses that node's CPU. Watch for a submitted job ID, RUNNING status, the CUDA/OpenACC result lines and the full-field checks. QUEUED means the GPU calculation has not started. The final summary states separately whether CUDA and OpenACC passed.

In [ ]:
# GPU jobs wait in Slurm until a device is allocated; never execute on the login node.
if globals().get('RUN_COMPLETE_EXPERIMENT',True) and globals().get('RUN_GPU_HEAT',True):
    GUIDE_GPU = automate.execute(GUIDE_ROOT,'gpu',GUIDE_SETTINGS,emit=lambda text: print(text,flush=True))
else:
    GUIDE_GPU = {'device':'gpu','status':'SKIPPED','reason':'Disabled in settings'}
GUIDE_REPORT.append(GUIDE_GPU)
automate.show_result(GUIDE_GPU,display,HTML)

### CPU profile

We have measured elapsed time; now we ask which functions used the CPU. This job builds an instrumented serial binary and collects a sampling profile. Read the largest time entries and relate them to the loops in the source. The profiler's own overhead means its timings are not part of our speedup comparison.

In [ ]:
# Profiling uses a separate instrumented executable, outside the benchmark table.
if globals().get('RUN_COMPLETE_EXPERIMENT',True) and globals().get('RUN_CPU_PROFILE',True):
    GUIDE_PROFILE = automate.execute(GUIDE_ROOT,'profile',GUIDE_SETTINGS,emit=lambda text: print(text,flush=True))
else:
    GUIDE_PROFILE = {'device':'profile','status':'SKIPPED','reason':'Disabled in settings'}
GUIDE_REPORT.append(GUIDE_PROFILE)
automate.show_result(GUIDE_PROFILE,display,HTML)

### Review the experiment

Check the summary and saved job folders. CPU PASS requires every requested solver and library check to complete. The Kelvin example remains an expected mismatch. GPU PASS requires CUDA to match the serial field; OpenACC is listed separately if unavailable.

If a job reaches the waiting limit, it remains in Slurm. Rerunning the notebook resumes monitoring its saved job ID. Keep the kernel connected for an uninterrupted run. The small default grid is useful for checking the workflow; increase the problem size and repetitions before drawing performance conclusions.

In [ ]:
from html import escape
rows=['<table><tr><th>Experiment</th><th>Status</th><th>Job</th><th>Saved folder</th></tr>']
for r in GUIDE_REPORT:
    rows.append('<tr>'+''.join('<td>'+escape(str(v))+'</td>' for v in (r['device'],r['status'],r.get('job_id',''),r.get('run_dir','')))+'</tr>')
display(HTML(''.join(rows)+'</table>'))
(GUIDE_ROOT/'experiment-report.json').write_text(json.dumps(GUIDE_REPORT,indent=2))
print('Saved report:',GUIDE_ROOT/'experiment-report.json')
if any(r['status']=='FAIL' for r in GUIDE_REPORT):print('Experiment needs attention: inspect the failed job logs above.')
elif any(r['status'] in ('WAIT LIMIT','UNKNOWN','NOT RUN') for r in GUIDE_REPORT):print('Experiment incomplete; see the recorded status. No unexecuted stage is marked PASS.')
else:print('Requested experiments finished. Read each PASS and SKIPPED entry separately.')

## Guided examples for the slides

These activities cover the rank/thread mapper, serial/MPI/hybrid heat comparison, full-field checks, temperature plot, timing discussion, CUDA vector addition and the original questions. Use them during the lecture if you want to inspect one concept at a time.

Set `RUN_GUIDED_EXAMPLE=True` in the first settings cell to execute them. With the default setting, Run All skips these extra submissions because the complete heat experiment has already covered the solver workflow. The vector-add GPU example is separate from the CUDA/OpenACC heat solvers.

| Slides in M5S1 | Notebook activity |
|---|---|
| 15-18 | Heat equation, serial update and two time-level arrays |
| 19-25 | MPI ownership, halo exchange and OpenMP work within a rank |
| 26-27 | Slurm submission and rank/thread placement |
| 31-35 | GPU execution and kernel versus whole-operation time |
| 41-47 | Overhead, timing, profiling and correctness |
| 48-49 | Compare complete saved fields and inspect actual results |
| 51 | Explain the computing model using your measurements |

The weather and molecular-dynamics slides are discussion examples. This notebook does not run WRF, LAMMPS, multi-node GPU-aware MPI or a complete MPI/OpenMP/GPU heat application. Its GPU heat exercise uses one GPU, and its hybrid CPU exercise uses MPI with OpenMP on one node.


### Before running

A two-dimensional heat simulation updates each cell using its neighbours. Write down your answers before submitting the job.

1. Can two MPI processes update different grid rows?
2. Why do they need to exchange halo rows at each step?
3. Will two ranks with two threads each print two or four rank/thread pairs?
4. Must MPI/OpenMP be faster than serial on a small grid?

Compare your predictions with the output. Remember that matching numerical results and achieving a speedup are separate questions.

### Settings and environment

The CPU job uses one node, two MPI ranks and up to two threads per rank. It runs three configurations of the same heat problem in sequence.

Start by changing only `GRID_N` or `STEPS`. Ranks remain separate processes even when they run on the same node. The resources visible to the notebook kernel are not necessarily the resources allocated to the batch job.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    from pathlib import Path
    import os, sys, json, re, time, uuid, hashlib, shutil, subprocess, statistics
    from IPython.display import display, Markdown
    BUILD = "M5S1-2026-10-13-v3"
    WORKSPACE = (Path.cwd() / "m5s1_runs").resolve()  # MUST be a student-owned filesystem shared with CPU compute nodes
    GRID_N = 256                # 16..2048; 256 is a quick classroom problem
    STEPS = 350                # 1..10000; diffusion time steps
    NEW_RUN = False            # True = submit a new job even with identical settings
    RUN_GPU = True             # Default: submit one GPU batch; Slurm queues it when GPUs are busy
    GPU_NEW_RUN = False         # Avoid duplicate jobs unless explicitly starting a new GPU experiment
    GPU_STATUS_WAIT_SECONDS = 20  # Brief optional notebook status wait; Slurm continues queueing after it returns
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    assert 16 <= GRID_N <= 2048 and 1 <= STEPS <= 10000
    print("Build:", BUILD, "| Python host:", os.uname().nodename)
    print("Working directory:", WORKSPACE)
    for tool in ("sbatch", "squeue", "sinfo", "srun", "mpicc"):
        print(f"{tool}: {shutil.which(tool) or 'not visible in notebook kernel'}")
    print("Slurm allocation for notebook:", os.environ.get("SLURM_JOB_ID", "not set"))
    print("CPU memory/count seen by notebook is NOT the resource allocation for the later batch job.")
    print("GPU job:", "auto-submit to Slurm; wait in queue if GPUs are busy" if RUN_GPU else "SKIPPED (RUN_GPU=False)")

### Read the C code

The next cell contains the source files and batch scripts. Look for the following operations.

- `MPI_Sendrecv` exchanges boundary rows with neighbouring ranks.
- `#pragma omp parallel for` divides local rows between CPU threads.
- Each update reads `old` and writes `next`, then swaps the arrays.
- `MPI_Gatherv` collects the final temperature field.
- `MPI_Reduce` combines checksums and finds the slowest-rank elapsed time.

The interior starts at 0 °C. The top edge stays at 100 °C, including the corners, and the remaining boundary cells stay at 0 °C.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    SRC_HELLO = "#include <mpi.h>\n#include <omp.h>\n#include <stdio.h>\nint main(int argc,char **argv) {\n  int provided,rank,size;\n  MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&provided);\n  MPI_Comm_rank(MPI_COMM_WORLD,&rank);\n  MPI_Comm_size(MPI_COMM_WORLD,&size);\n  #pragma omp parallel\n  {\n    #pragma omp critical\n    printf(\"MAP rank=%d of=%d thread=%d of_threads=%d\\n\",\n           rank,size,omp_get_thread_num(),omp_get_num_threads());\n  }\n  MPI_Finalize(); return 0;\n}"
    SRC_HEAT = "/* Explicit 5-point Jacobi heat update. Row decomposition and MPI halos.\n   At each step every rank reads old[] and writes next[]; no in-place race. */\n#include <mpi.h>\n#include <omp.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\nstatic int rows_for(int r,int n,int p){return n/p+(r<n%p);}\nstatic int offset_for(int r,int n,int p){return r*(n/p)+(r<n%p?r:n%p);}\nint main(int argc,char **argv) {\n  int rank,p,provided;\n  MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&provided);\n  MPI_Comm_rank(MPI_COMM_WORLD,&rank); MPI_Comm_size(MPI_COMM_WORLD,&p);\n  if(argc!=5){if(rank==0)fprintf(stderr,\"usage: heat N STEPS output.bin label\\n\"); MPI_Abort(MPI_COMM_WORLD,2);}\n  int n=atoi(argv[1]),steps=atoi(argv[2]),rows=rows_for(rank,n,p),first=offset_for(rank,n,p);\n  if(n<16||n>2048||steps<1||steps>10000||p>n){if(rank==0)fprintf(stderr,\"invalid parameters\\n\"); MPI_Abort(MPI_COMM_WORLD,3);}\n  size_t count=(size_t)(rows+2)*n;\n  double *old=(double*)calloc(count,sizeof(double)),*next=(double*)calloc(count,sizeof(double));\n  if(!old||!next)MPI_Abort(MPI_COMM_WORLD,4);\n  for(int y=1;y<=rows;y++)if(first+y-1==0)for(int x=0;x<n;x++)old[(size_t)y*n+x]=100.0;\n  int up=(rank==0?MPI_PROC_NULL:rank-1),down=(rank==p-1?MPI_PROC_NULL:rank+1);\n  MPI_Barrier(MPI_COMM_WORLD); double t0=MPI_Wtime();\n  for(int step=0;step<steps;step++){\n    MPI_Sendrecv(old+n,n,MPI_DOUBLE,up,10,old+(size_t)(rows+1)*n,n,MPI_DOUBLE,down,10,MPI_COMM_WORLD,MPI_STATUS_IGNORE);\n    MPI_Sendrecv(old+(size_t)rows*n,n,MPI_DOUBLE,down,11,old,n,MPI_DOUBLE,up,11,MPI_COMM_WORLD,MPI_STATUS_IGNORE);\n    #pragma omp parallel for schedule(static)\n    for(int y=1;y<=rows;y++){\n      int global_y=first+y-1;\n      if(global_y==0||global_y==n-1){\n        memcpy(next+(size_t)y*n,old+(size_t)y*n,(size_t)n*sizeof(double));\n      }else{\n        next[(size_t)y*n]=0.0; next[(size_t)y*n+n-1]=0.0;\n        for(int x=1;x<n-1;x++)next[(size_t)y*n+x]=0.25*(old[(size_t)(y-1)*n+x]+old[(size_t)(y+1)*n+x]+old[(size_t)y*n+x-1]+old[(size_t)y*n+x+1]);\n      }\n    }\n    double *tmp=old;old=next;next=tmp;\n  }\n  MPI_Barrier(MPI_COMM_WORLD); double elapsed=MPI_Wtime()-t0,max_elapsed=0;\n  MPI_Reduce(&elapsed,&max_elapsed,1,MPI_DOUBLE,MPI_MAX,0,MPI_COMM_WORLD);\n  double partial=0,probe_local=0,checksum=0,probe=0;\n  for(int y=1;y<=rows;y++){\n    int gy=first+y-1;\n    for(int x=0;x<n;x++)partial+=old[(size_t)y*n+x];\n    if(gy==8)probe_local=old[(size_t)y*n+n/2];\n  }\n  MPI_Reduce(&partial,&checksum,1,MPI_DOUBLE,MPI_SUM,0,MPI_COMM_WORLD);\n  MPI_Reduce(&probe_local,&probe,1,MPI_DOUBLE,MPI_SUM,0,MPI_COMM_WORLD);\n  double *global=NULL;int *counts=NULL,*offsets=NULL;\n  if(rank==0){\n    global=(double*)malloc((size_t)n*n*sizeof(double));\n    counts=(int*)malloc((size_t)p*sizeof(int));offsets=(int*)malloc((size_t)p*sizeof(int));\n    if(!global||!counts||!offsets)MPI_Abort(MPI_COMM_WORLD,5);\n    for(int r=0;r<p;r++){counts[r]=rows_for(r,n,p)*n;offsets[r]=offset_for(r,n,p)*n;}\n  }\n  MPI_Gatherv(old+n,rows*n,MPI_DOUBLE,global,counts,offsets,MPI_DOUBLE,0,MPI_COMM_WORLD);\n  if(rank==0){\n    FILE *f=fopen(argv[3],\"wb\");\n    if(!f || fwrite(global,sizeof(double),(size_t)n*n,f)!=(size_t)n*n){fprintf(stderr,\"Unable to write heat grid\\n\");MPI_Abort(MPI_COMM_WORLD,6);}\n    fclose(f);\n    printf(\"RESULT mode=%s ranks=%d threads=%d n=%d steps=%d elapsed_s=%.9f checksum=%.12f probe=%.12f\\n\",\n      argv[4],p,omp_get_max_threads(),n,steps,max_elapsed,checksum,probe);\n    fflush(stdout);\n  }\n  free(old);free(next);free(global);free(counts);free(offsets);\n  MPI_Finalize();return 0;\n}"
    SRC_CUDA = "/* Single-GPU teaching experiment: one vector addition, actual CUDA events. */\n#include <cuda_runtime.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <math.h>\n#include <chrono>\n#define CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){fprintf(stderr,\"CUDA error: %s\\n\",cudaGetErrorString(e));return 2;} }while(0)\n__global__ void add(const float *a,const float *b,float *c,int n){int i=blockIdx.x*blockDim.x+threadIdx.x;if(i<n)c[i]=a[i]+b[i];}\nint main(){\n  const int n=1<<20; size_t bytes=(size_t)n*sizeof(float);\n  float *a=(float*)malloc(bytes),*b=(float*)malloc(bytes),*c=(float*)malloc(bytes);\n  if(!a||!b||!c)return 3;\n  for(int i=0;i<n;i++){a[i]=0.25f*i;b[i]=2.0f;}\n  float *da=NULL,*db=NULL,*dc=NULL;\n  auto start=std::chrono::steady_clock::now();\n  CHECK(cudaMalloc((void**)&da,bytes));CHECK(cudaMalloc((void**)&db,bytes));CHECK(cudaMalloc((void**)&dc,bytes));\n  CHECK(cudaMemcpy(da,a,bytes,cudaMemcpyHostToDevice));CHECK(cudaMemcpy(db,b,bytes,cudaMemcpyHostToDevice));\n  cudaEvent_t e0,e1;CHECK(cudaEventCreate(&e0));CHECK(cudaEventCreate(&e1));\n  CHECK(cudaEventRecord(e0));\n  add<<<(n+255)/256,256>>>(da,db,dc,n);\n  CHECK(cudaGetLastError());CHECK(cudaEventRecord(e1));CHECK(cudaEventSynchronize(e1));\n  float kernel_ms=0;CHECK(cudaEventElapsedTime(&kernel_ms,e0,e1));\n  CHECK(cudaMemcpy(c,dc,bytes,cudaMemcpyDeviceToHost));\n  auto end=std::chrono::steady_clock::now();\n  double total_ms=std::chrono::duration<double,std::milli>(end-start).count();\n  float worst=0;\n  for(int i=0;i<n;i++){float diff=fabsf(c[i]-(a[i]+b[i]));if(diff>worst)worst=diff;}\n  printf(\"GPU_RESULT n=%d kernel_ms=%.5f end_to_end_ms=%.5f max_error=%.7g\\n\",n,kernel_ms,total_ms,worst);\n  CHECK(cudaEventDestroy(e0));CHECK(cudaEventDestroy(e1));\n  CHECK(cudaFree(da));CHECK(cudaFree(db));CHECK(cudaFree(dc));\n  free(a);free(b);free(c);return worst<0.001f?0:4;\n}"
    CPU_SCRIPT = "#!/bin/bash\n#SBATCH --job-name=m5s1-hybrid\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=2\n#SBATCH --cpus-per-task=2\n#SBATCH --mem=2G\n#SBATCH --time=00:06:00\n#SBATCH --output=cpu-%j.out\nset -euo pipefail\ncd \"$SLURM_SUBMIT_DIR\"\nif [ -f /etc/profile.d/modules.sh ]; then source /etc/profile.d/modules.sh; fi\nif command -v module >/dev/null 2>&1; then\n  module load OpenMPI/4.1.5-GCC-12.3.0 2>/dev/null || module load foss/2023b 2>/dev/null || true\n  module list 2>&1 || true\nfi\ncommand -v mpicc || { echo 'FAIL: mpicc unavailable'; exit 3; }\ncommand -v srun || { echo 'FAIL: srun unavailable'; exit 4; }\necho \"JOB allocation=$SLURM_JOB_ID host=$(hostname) nodes=$SLURM_JOB_NUM_NODES\"\ntest -f hybrid_hello.c || { echo \"FAIL: hybrid_hello.c missing in $(pwd)\"; exit 7; }\ntest -f hybrid_heat.c || { echo \"FAIL: hybrid_heat.c missing in $(pwd)\"; exit 7; }\nmpicc -O2 -fopenmp hybrid_hello.c -o hybrid_hello\nmpicc -O2 -fopenmp hybrid_heat.c -o hybrid_heat\nexport OMP_DYNAMIC=FALSE\nexport OMP_NUM_THREADS=2\necho '--- Rank / thread map (real Slurm allocation) ---'\nsrun --ntasks=2 --cpus-per-task=2 ./hybrid_hello\necho '--- Same scientific problem, three configurations ---'\nexport OMP_NUM_THREADS=1\nsrun --ntasks=1 --cpus-per-task=1 ./hybrid_heat __N__ __STEPS__ serial.bin serial\nsrun --ntasks=2 --cpus-per-task=1 ./hybrid_heat __N__ __STEPS__ mpi.bin mpi\nexport OMP_NUM_THREADS=2\nsrun --ntasks=2 --cpus-per-task=2 ./hybrid_heat __N__ __STEPS__ hybrid.bin hybrid\necho 'CPU_LAB_DONE'\n"
    GPU_SCRIPT = "#!/bin/bash\n#SBATCH --job-name=m5s1-gpu\n#SBATCH --partition=gpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=2\n#SBATCH --gpus=1\n#SBATCH --mem=2G\n#SBATCH --time=00:05:00\n#SBATCH --output=gpu-%j.out\nset -euo pipefail\ncd \"$SLURM_SUBMIT_DIR\"\nif [ -f /etc/profile.d/modules.sh ]; then source /etc/profile.d/modules.sh; fi\nif command -v module >/dev/null 2>&1; then\n  module load nvhpc/25.7 2>/dev/null || true\n  module list 2>&1 || true\nfi\ncommand -v nvidia-smi || { echo 'SKIPPED: no NVIDIA runtime'; exit 0; }\nnvidia-smi --query-gpu=name --format=csv,noheader\ncommand -v nvcc || { echo 'SKIPPED: nvcc unavailable; see M2.S4 GPU notebook'; exit 0; }\ntest -f gpu_vector_add.cu || { echo \"FAIL: gpu_vector_add.cu missing in $(pwd)\"; exit 7; }\nnvcc -O2 gpu_vector_add.cu -o gpu_vector_add\n./gpu_vector_add\necho 'GPU_LAB_DONE'\n"
    SOURCES = {"hybrid_hello.c": SRC_HELLO, "hybrid_heat.c": SRC_HEAT, "gpu_vector_add.cu": SRC_CUDA}
    print("Actual embedded source files:", ", ".join(f"{k} ({len(v)} bytes)" for k,v in SOURCES.items()))
    print("The heat loop uses MPI_Sendrecv and #pragma omp parallel for:", "MPI_Sendrecv" in SRC_HEAT, "#pragma omp parallel for" in SRC_HEAT)
    print("\n--- Executed halo exchange ---")
    i = SRC_HEAT.index("MPI_Sendrecv")
    print(SRC_HEAT[i:i+380])
    print("\n--- Executed OpenMP loop ---")
    i = SRC_HEAT.index("#pragma omp parallel for")
    print(SRC_HEAT[i:i+620])

### Save the source and settings

The notebook creates a run folder from the source and scientific settings. With unchanged settings and `NEW_RUN=False`, it reuses that folder and its job ID.

A queued job must keep the source and settings it was submitted with. Each run therefore has its own files and Slurm stdout log. The job uses the SciTech MPI module `OpenMPI/4.1.5-GCC-12.3.0` when available and writes binary temperature fields for the checks and plots below.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    FINGERPRINT = hashlib.sha256(json.dumps({"build":BUILD,"n":GRID_N,"steps":STEPS,"sources":SOURCES,"script":CPU_SCRIPT},sort_keys=True).encode()).hexdigest()
    RUN_DIR = WORKSPACE / ("cpu-"+FINGERPRINT[:18]+("-"+uuid.uuid4().hex[:6] if NEW_RUN else ""))
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    for name, source in SOURCES.items():
        target = RUN_DIR / name
        if not target.exists(): target.write_text(source)
        elif target.read_text() != source: raise RuntimeError("Frozen source mismatch: "+str(target))
    batch = CPU_SCRIPT.replace("__N__",str(GRID_N)).replace("__STEPS__",str(STEPS))
    for name,source in {"cpu.sbatch":batch}.items():
        target=RUN_DIR/name
        if not target.exists():target.write_text(source)
        elif target.read_text()!=source:raise RuntimeError("Frozen batch mismatch")
    print("Frozen source/settings fingerprint:",FINGERPRINT[:18])
    print("Run dir:",RUN_DIR)
    print("Grid:",GRID_N,"x",GRID_N,"| steps:",STEPS)
    print("Batch resources: 1 CPU node, 2 tasks, 2 cores/task, 2 GiB, 6 minutes")
    print("Example rank layouts: 1x1 (serial); 2x1 (MPI); 2x2 (MPI+OpenMP)")

### Submit the CPU job

How many MAP lines do you expect from two ranks with two threads each? Which configuration will exchange the most MPI messages?

The batch job compiles the C source and runs three configurations with the same grid and number of time steps.

1. One rank and one thread.
2. Two ranks and one thread per rank.
3. Two ranks and two threads per rank.

These configurations use different numbers of CPU cores. Interpret their timings with that difference in mind. Small-problem overhead and worker placement also affect the result.

If the job is queued, wait and rerun the collection cell. If Slurm is unavailable, the submission cell reports NOT RUN.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    CPU_JOB_FILE = RUN_DIR / "cpu_job_id.txt"
    CPU_JOB_ID = CPU_JOB_FILE.read_text().strip() if CPU_JOB_FILE.exists() else None
    if CPU_JOB_ID and not NEW_RUN:
        print("Existing CPU job:",CPU_JOB_ID,"(no resubmission)")
    elif not shutil.which("sbatch"):
        print("NOT RUN: sbatch unavailable. Use SciTech JupyterHub / cluster environment.")
    else:
        env = os.environ.copy()
        for key in ("SLURM_MEM_PER_CPU","SLURM_MEM_PER_GPU","SLURM_MEM_PER_NODE"):env.pop(key,None)
        p = subprocess.run(["sbatch","--parsable","cpu.sbatch"],
                           cwd=str(RUN_DIR),capture_output=True,text=True,env=env,timeout=35)
        if p.returncode:
            print("FAIL: CPU job not accepted:",p.stderr.strip() or p.stdout.strip())
        else:
            CPU_JOB_ID=p.stdout.strip().split(";")[0]
            CPU_JOB_FILE.write_text(CPU_JOB_ID)
            print("Submitted CPU job:",CPU_JOB_ID,"| inspect with: squeue -j",CPU_JOB_ID)

### Collect the CPU output

Read the log for your current run. When the batch finishes, check the four MAP lines and the three RESULT lines.

PENDING means that the completion marker has not appeared yet. FAIL means that execution ended without the required results or that a check failed. Keep the log so you can diagnose the problem.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    CPU_RESULTS = {}
    CPU_STATUS = "NOT RUN"
    CPU_LOG = ""
    if not CPU_JOB_ID:
        print("NOT RUN: no CPU batch job submitted.")
    else:
        logfile=RUN_DIR / ("cpu-"+CPU_JOB_ID+".out")
        if not logfile.exists():
            CPU_STATUS="PENDING"
            print("PENDING: waiting for log",logfile.name)
        else:
            CPU_LOG=logfile.read_text(errors="replace")
            for line in CPU_LOG.splitlines():
                if line.startswith(("JOB ","MAP ","RESULT ","FAIL:","srun:","slurmstepd:")):print(line)
            for line in CPU_LOG.splitlines():
                if line.startswith("RESULT "):
                    fields=dict(x.split("=",1) for x in line.split()[1:] if "=" in x)
                    try:
                        mode=fields["mode"]
                        CPU_RESULTS[mode]={"ranks":int(fields["ranks"]),"threads":int(fields["threads"]),
                          "n":int(fields["n"]),"steps":int(fields["steps"]),"elapsed_s":float(fields["elapsed_s"]),
                          "checksum":float(fields["checksum"]),"probe":float(fields["probe"])}
                    except (ValueError,KeyError) as e:print("FAIL: unreadable RESULT:",e)
            if "CPU_LAB_DONE" not in CPU_LOG: CPU_STATUS="PENDING" if shutil.which("squeue") and subprocess.run(["squeue","-h","-j",CPU_JOB_ID],capture_output=True,text=True,timeout=20).stdout.strip() else "FAIL"
            elif set(CPU_RESULTS)=={"serial","mpi","hybrid"} and all((RUN_DIR/(m+".bin")).exists() for m in CPU_RESULTS):
                # Full-field validation uses only Python stdlib, even if NumPy is absent.
                try:
                    from array import array
                    def read_grid(mode):
                        buf=array("d")
                        buf.frombytes((RUN_DIR/(mode+".bin")).read_bytes())
                        if len(buf)!=GRID_N*GRID_N:raise ValueError("Unexpected grid size: "+mode)
                        return buf
                    reference=read_grid("serial")
                    differences={mode:max(abs(a-b) for a,b in zip(reference,read_grid(mode)))
                                 for mode in ("mpi","hybrid")}
                    print("Maximum full-grid differences vs serial:", differences)
                    summary_ok=all(abs(CPU_RESULTS[x]["checksum"]-CPU_RESULTS["serial"]["checksum"])<=1e-6
                                   and abs(CPU_RESULTS[x]["probe"]-CPU_RESULTS["serial"]["probe"])<=1e-8
                                   for x in ("mpi","hybrid"))
                    CPU_STATUS="PASS" if summary_ok and all(err<=1e-9 for err in differences.values()) else "FAIL"
                except Exception as e:
                    CPU_STATUS="FAIL"
                    print("Full-grid check failed:",repr(e))
            else: CPU_STATUS="FAIL"
            if CPU_STATUS=="FAIL":print("\n--- Tail of actual batch log ---\n",CPU_LOG[-3500:])
        print("CPU status:",CPU_STATUS)
        if CPU_STATUS=="PENDING": print("Rerun THIS COLLECTION cell later. Do not resubmit another job.")

### Ranks and threads

The mapper starts two MPI ranks, each with two OpenMP threads. Each thread prints its rank and thread ID inside a critical region.

Look for `rank=0 thread=0` and `rank=1 thread=1`. There should be four MAP lines after a successful run, but their order can vary.

Explain why a rank is a process identity rather than a physical node. Threads within one rank share an address space; separate ranks do not.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    if CPU_STATUS!="PASS":
        print("NOT RUN (this activity): first complete the CPU job. Current status:",CPU_STATUS)
    else:
        m=[line for line in CPU_LOG.splitlines() if line.startswith("MAP ")]
        print("Observed MAP lines:",len(m))
        for line in sorted(m):print(line)
        distinct=set((re.search(r"rank=(\d+)",line).group(1),re.search(r"thread=(\d+)",line).group(1)) for line in m)
        print("MAP correctness:", "PASS" if len(m)==4 and len(distinct)==4 else "FAIL")

### Check the temperature fields

All three configurations use the same stencil and keep the old and next time levels separate. MPI ranks exchange their halo rows at every step.

Compare the checksums and the probe at row 8, centre column. Then compare every cell in the saved fields. Matching a checksum alone is not sufficient.

Increase `STEPS` to 450 and rerun from setup. Keep the grid size unchanged and explain how the temperature field changes.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    if CPU_STATUS!="PASS":
        print("NOT RUN: CPU results are required.")
    else:
        print(f"{'mode':<12} {'ranks x threads':<18} {'checksum':>18} {'T[8,mid]':>14} {'seconds':>12}")
        for mode in ("serial","mpi","hybrid"):
            r=CPU_RESULTS[mode];print(f"{mode:<12} {r['ranks']} x {r['threads']:<14} {r['checksum']:>18.6f} {r['probe']:>14.6f} {r['elapsed_s']:>12.6f}")
        try:
            import numpy as np
            base=np.fromfile(RUN_DIR/"serial.bin",dtype=np.float64).reshape((GRID_N,GRID_N))
            for mode in ("mpi","hybrid"):
                other=np.fromfile(RUN_DIR/(mode+".bin"),dtype=np.float64).reshape((GRID_N,GRID_N))
                error=float(np.max(np.abs(base-other)))
                print(f"Full-grid verification for {mode}: max absolute difference = {error:.3g} ->", "PASS" if error<=1e-9 else "FAIL")
            print("Physical sanity (top boundary 100, bottom and non-top sides 0):",
                  "PASS" if np.all(base[0,:]==100) and np.all(base[-1,:]==0) and np.all(base[1:,0]==0) and np.all(base[1:,-1]==0) else "FAIL")
        except ImportError:
            print("SKIPPED: NumPy not installed in kernel; compute-job checksum/probe cross-check still executed.")
        except Exception as e: print("FAIL: field verification error:",repr(e))

### See the heat flow

The solver writes `serial.bin`, `mpi.bin` and `hybrid.bin`. The next cell reads and plots the saved field.

We expect the warmest region near the top edge. Predict how it changes after more time steps, then compare runs with 350 and 450 steps while keeping `GRID_N` fixed.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    if CPU_STATUS != "PASS":
        print("NOT RUN: heat grid not yet produced")
    else:
        import array
        heat=array.array('d')
        heat.frombytes((RUN_DIR/'hybrid.bin').read_bytes())
        display(HTML(heat_svg(heat,GRID_N,'Saved hybrid temperature field')))
        print('Observed probe:',heat[8*GRID_N+GRID_N//2], '°C')


### Compare execution times

The C solver uses `MPI_Wtime` to measure the update-and-halo loop and reports the slowest-rank duration. Compilation and binary output are outside that interval.

Could MPI/OpenMP be slower than serial for this grid? Use the measured durations to explain your answer. Repeat the experiment with `NEW_RUN=True` to see how much the timings vary.

You can also double `GRID_N` within the class resource limits. That changes the problem size, so compare the configurations at the new size rather than treating it as the same-workload speedup.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    if CPU_STATUS!="PASS":
        print("NOT RUN: no measured CPU results.")
    else:
        base=CPU_RESULTS["serial"]["elapsed_s"]
        for mode in ("serial","mpi","hybrid"):
            r=CPU_RESULTS[mode]
            print(f"{mode:>7}: actual elapsed {r['elapsed_s']:.6f} s; "
                  f"ratio serial/mode = {base/r['elapsed_s']:.3f}x; "
                  f"layout {r['ranks']} rank(s) × {r['threads']} thread(s)")
        print("Interpretation: these are individual measured runs. Ratios include MPI/OpenMP overhead and different core allocations.")


### Optional CUDA experiment

With `RUN_GPU=True`, the next cell submits one GPU batch job. If the GPU is busy, Slurm queues it and starts it later. Rerunning the cell reuses the saved job ID. Set `RUN_GPU=False` to skip it, or use `scancel JOB_ID` to cancel a submitted job.

The program adds two vectors on one GPU and checks every output element. CUDA events measure kernel time. A host clock measures the broader operation, including allocations and transfers.

Which duration do you expect to be greater, and why?

The job requests one GPU, two CPU cores, 2 GiB and five minutes. It loads `nvhpc/25.7` when available. The collection cell checks progress briefly and returns; the Slurm job continues independently of Jupyter.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    # Submit the optional GPU job by default, even if all GPUs are occupied.
    # Slurm queues it and starts it automatically when a GPU becomes available.
    # This cell does not run GPU work on the Jupyter/login node.
    GPU_DIR = WORKSPACE / ("gpu-" + hashlib.sha256(
        (SRC_CUDA + GPU_SCRIPT + BUILD).encode()).hexdigest()[:16]
        + ("-" + uuid.uuid4().hex[:6] if GPU_NEW_RUN else ""))
    GPU_JOB_FILE = GPU_DIR / "gpu_job_id.txt"
    GPU_JOB_ID = None
    GPU_SUBMISSION_STATUS = "NOT RUN"
    GPU_SUBMISSION_REASON = ""

    if not RUN_GPU:
        GPU_SUBMISSION_STATUS = "SKIPPED"
        GPU_SUBMISSION_REASON = "RUN_GPU=False: optional GPU job disabled."
    else:
        GPU_DIR.mkdir(parents=True, exist_ok=True)
        for name, source in {"gpu_vector_add.cu": SRC_CUDA, "gpu.sbatch": GPU_SCRIPT}.items():
            target = GPU_DIR / name
            if target.exists() and target.read_text() != source:
                raise RuntimeError("Frozen GPU source mismatch: " + str(target))
            if not target.exists():
                target.write_text(source)

        if GPU_JOB_FILE.exists() and not GPU_NEW_RUN:
            GPU_JOB_ID = GPU_JOB_FILE.read_text().strip()
            if GPU_JOB_ID.isdigit():
                GPU_SUBMISSION_STATUS = "SUBMITTED"
                GPU_SUBMISSION_REASON = f"Reusing GPU job {GPU_JOB_ID} (no duplicate submission)."
            else:
                GPU_JOB_ID = None
                GPU_SUBMISSION_STATUS = "FAIL"
                GPU_SUBMISSION_REASON = "Saved GPU job ID is invalid; inspect " + str(GPU_JOB_FILE)
        elif not shutil.which("sbatch"):
            GPU_SUBMISSION_STATUS = "NOT RUN"
            GPU_SUBMISSION_REASON = "Slurm sbatch is unavailable; no GPU job submitted."
        else:
            env = os.environ.copy()
            for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
                env.pop(key, None)
            try:
                # Deliberately NO --immediate: let Slurm queue the job if GPUs are busy.
                # SciTech's sbatch version does not support the earlier --immediate=15 option.
                p = subprocess.run(
                    ["sbatch", "--parsable", "gpu.sbatch"],
                    cwd=str(GPU_DIR),capture_output=True, text=True, env=env, timeout=35)
                if p.returncode:
                    GPU_SUBMISSION_STATUS = "FAIL"
                    GPU_SUBMISSION_REASON = "Slurm rejected the GPU job: " + (
                        p.stderr.strip() or p.stdout.strip() or f"return code {p.returncode}")
                else:
                    candidate = p.stdout.strip().split(";")[0]
                    if not candidate.isdigit():
                        GPU_SUBMISSION_STATUS = "FAIL"
                        GPU_SUBMISSION_REASON = "Unexpected sbatch response: " + p.stdout.strip()
                    else:
                        GPU_JOB_ID = candidate
                        GPU_JOB_FILE.write_text(GPU_JOB_ID)
                        GPU_SUBMISSION_STATUS = "SUBMITTED"
                        GPU_SUBMISSION_REASON = f"GPU job {GPU_JOB_ID} submitted to Slurm; it may be QUEUED."
            except (OSError, subprocess.TimeoutExpired) as exc:
                GPU_SUBMISSION_STATUS = "FAIL"
                GPU_SUBMISSION_REASON = "Unable to submit GPU job: " + repr(exc)

    print("GPU submission:", GPU_SUBMISSION_STATUS, "-", GPU_SUBMISSION_REASON)
    if GPU_JOB_ID:
        print("Check queue: squeue -j", GPU_JOB_ID, "| Cancel if needed: scancel", GPU_JOB_ID)


### Collect the CUDA output

Look for `GPU_RESULT`, with kernel time, end-to-end time and maximum numerical error. PASS requires the completion marker and an error no greater than 0.001.

If the compiler or runtime is unavailable, the exercise reports SKIPPED. QUEUED means Slurm is waiting for resources; RUNNING means the job has started. Rerun this collection cell later if necessary.

Explain the difference between kernel time and the complete operation. A GPU speedup claim also needs a CPU comparison with equivalent work.

In [ ]:
if RUN_GUIDED_EXAMPLE:
    # Observe the real Slurm state. Never fake kernel results or treat a queued job as FAIL.
    def _gpu_slurm_state(job_id):
        if shutil.which("squeue"):
            try:
                q = subprocess.run(
                    ["squeue", "-h", "-j", str(job_id), "-o", "%T|%R"],
                    capture_output=True, text=True, timeout=15)
                if q.returncode == 0 and q.stdout.strip():
                    state, _, reason = q.stdout.strip().splitlines()[0].partition("|")
                    return state.strip().upper(), reason.strip()
            except (OSError, subprocess.TimeoutExpired):
                pass
        if shutil.which("sacct"):
            try:
                a = subprocess.run(["sacct", "-n", "-P", "-X", "-j", str(job_id),
                                    "-o", "State"], capture_output=True, text=True, timeout=15)
                if a.returncode == 0 and a.stdout.strip():
                    return a.stdout.strip().splitlines()[0].split("|")[0].strip().upper(), "Slurm accounting"
            except (OSError, subprocess.TimeoutExpired):
                pass
        return "UNKNOWN", "Not found in the visible Slurm queue/accounting"

    def _collect_gpu_once():
        if not RUN_GPU:
            return "SKIPPED", "GPU experiment was disabled."
        if not GPU_JOB_ID:
            return GPU_SUBMISSION_STATUS, GPU_SUBMISSION_REASON
        logfile = GPU_DIR / ("gpu-" + GPU_JOB_ID + ".out")
        log = logfile.read_text(errors="replace") if logfile.exists() else ""
        if "GPU_LAB_DONE" in log and "GPU_RESULT" in log:
            match = re.search(r"max_error=([0-9.eE+-]+)", log)
            return ("PASS" if match and float(match.group(1)) <= 0.001 else "FAIL",
                    "GPU result verified." if match else "Missing numerical correctness result.")
        if "SKIPPED:" in log:
            return "SKIPPED", log[log.index("SKIPPED:"):].splitlines()[0]
        state, reason = _gpu_slurm_state(GPU_JOB_ID)
        if state in ("PENDING", "CONFIGURING", "SUSPENDED", "REQUEUED", "RESV_DEL_HOLD"):
            return "QUEUED", reason
        if state in ("RUNNING", "COMPLETING", "STAGE_OUT"):
            return "RUNNING", reason
        if state.startswith(("FAILED", "CANCELLED", "TIMEOUT", "NODE_FAIL", "OUT_OF_MEMORY", "BOOT_FAIL")):
            return "FAIL", f"Slurm {state}: {reason}; log tail: {log[-1200:]}"
        if state == "COMPLETED":
            return "FAIL", "Slurm reports COMPLETED but the required GPU_RESULT/GPU_LAB_DONE evidence is missing: " + log[-1200:]
        if log:
            return "UNKNOWN", "Incomplete log; inspect: " + log[-1200:]
        return "UNKNOWN", f"Job {GPU_JOB_ID} is not visible yet; check squeue/sacct and retry."

    GPU_STATUS, GPU_STATUS_REASON = _collect_gpu_once()
    if GPU_JOB_ID and GPU_STATUS in ("QUEUED", "RUNNING"):
        deadline = time.monotonic() + max(0, min(int(GPU_STATUS_WAIT_SECONDS), 120))
        while time.monotonic() < deadline and GPU_STATUS in ("QUEUED", "RUNNING"):
            time.sleep(min(5, max(0, deadline-time.monotonic())))
            GPU_STATUS, GPU_STATUS_REASON = _collect_gpu_once()
    print("Optional GPU status:", GPU_STATUS)
    print("Details:", GPU_STATUS_REASON)
    if GPU_JOB_ID:
        logfile = GPU_DIR / ("gpu-" + GPU_JOB_ID + ".out")
        if logfile.exists():
            print("--- Actual GPU batch output (last 3000 chars) ---")
            print(logfile.read_text(errors="replace")[-3000:])
        if GPU_STATUS in ("QUEUED", "RUNNING", "UNKNOWN"):
            print("The Slurm job can continue without Jupyter being open. Re-run this status cell later.")
            print("Queue command: squeue -j", GPU_JOB_ID)


## 12 [CHECKPOINT] Four short questions

Q1. Two CPU nodes each have eight cores. Is `2 MPI ranks × 8 threads` always faster than `4 ranks × 4 threads`?
- A. Yes, because it has fewer ranks.
- B. Yes, because OpenMP is always cheaper.
- C. No. Placement, workload and communication determine performance.
- D. No, because MPI cannot work within one node.

Q2. In the heat solver, what is the purpose of an MPI halo exchange?
- A. Copy the entire simulation into every rank.
- B. Supply neighbouring temperatures needed by a local stencil.
- C. Make all ranks use shared memory.
- D. Replace synchronization between time steps.

Q3. The GPU kernel takes 0.2 ms, but the complete GPU operation takes 4 ms. Which is justified?
- A. The entire GPU solution takes 0.2 ms.
- B. Data movement and setup contribute to total time.
- C. The GPU result must be incorrect.
- D. MPI automatically fixes the difference.

Q4. Which check best supports numerical correctness of hybrid heat computation?
- A. The hybrid version is the fastest.
- B. MPI prints two rank IDs.
- C. Compare the complete final grids with the same parameters.
- D. GPU utilization is 100%.

<details><summary>Answers (reveal after discussion)</summary>

- Q1 C: neither layout is universally best; A/B ignore resource placement and D is false.
- Q2 B: halo cells store copies of nearby rank-owned data; A/C/D misstate their function.
- Q3 B: end-to-end time includes costs outside the kernel; the smaller number is not whole-application latency.
- Q4 C: numerical agreement is independent of speed and resource utilization.

</details>

### Change one setting

Choose one experiment and record your prediction before running it.

1. Set `STEPS=450` and keep the grid fixed. What temperature change do you expect near the top?
2. Set `GRID_N=512` and keep the steps fixed. Does overhead become less important?
3. Set `NEW_RUN=True` without changing the calculation. How much do repeated timings vary?
4. Collect the optional GPU job and explain the difference between kernel and end-to-end time.

Save the changed setting, job ID, check status, measured output and your explanation.

MPI divides data between processes. OpenMP shares local work between threads. GPUs accelerate suitable kernels, with a cost for moving data. We need to check correctness and measure the complete execution before deciding which layout helps.

In the next session we will examine how communication, threading and data placement affect scalability.

### The final image and GPU result

The image below is the PNG saved by the ParaView job for this experiment. Use the colour scale to explain where the plate is warmest and compare it with the centre-line profile. The same cell states whether the GPU calculation completed and passed its numerical checks. If rendering or GPU execution did not complete, it shows that status rather than an example result.

In [ ]:
automate.show_final(GUIDE_REPORT, display, HTML)
